In [ ]:
#!/usr/bin/env python3
"""
DREAM Cloud Analysis Pipeline
==============================
Night: configurable (default 2026-01-25)
Three pointing strategies:
  Green  - Absolute minimum cloud coverage
  Blue   - Predicted minimum via cloud motion tracking
  Red    - Baseline scheduler observations

Pipeline steps:
  1. Load night frames from CSV
  2. Patchiness check: sample ~10 frames, compute spatial variance.
     - All-clear OR all-cloudy  → skip (no science in tracking)
     - Patchy (high variation)  → proceed with full analysis
  3. Full analysis:
     a. Cloud motion tracking for all frames
     b. Metrics (photons, slew, efficiency) with correct slew-gating
     c. Metrics summary plot (left panel only, no live video subplot)
     d. MP4 cloud tracking video (map panel only)
     e. ZP cross-validation: DREAM .zps vs ConsDB
     f. ZP comparison plot

Modifications (v2):
  (a) Moon avoidance: never point within MOON_EXCLUSION_DEG of the Moon
      for absolute-min and motion-tracking strategies (moon position computed
      per-frame from Astropy ephemeris).
  (b) Zenith-proximity weighting in motion-tracking candidate selection:
      scored as  score = w_cloud * cloud_val + w_slew * slew_cost + w_zenith * zenith_cost
      where zenith_cost = zenith_angle / 90.
  (c) Post-CSV atmospheric + seeing quality corrections applied to summary
      stats using ConsDB quicklook columns (eff_time_*, psf_sigma_median,
      zero_point_median, seeing_zenith_500nm_median, airmass).
  (d) Spatial pointing tensors saved per-night (and across all nights) as
      compressed NumPy .npz files, enabling sky-map visualisations later.

Coordinate / math notes (bugs fixed):
  - Scheduler field RA/Dec converted to alt/az at the ACTUAL frame timestamp,
    then rejected if alt < 15 deg (below horizon guard).
  - xy_to_altaz and radec_to_altaz_xy use consistent projection:
      x = -cos(alt)*sin(az),  y = cos(alt)*cos(az),  scale = R/sin(alt)
  - Slew gating: expose_t = max(0, SLOT_TIME - slew_t);
      photons  = photons_full * (expose_t / EXPOSURE_TIME)

HDF5 schema (confirmed):
  clouds        sys extinction in mag  (float32)
  sigma         1-sigma uncertainty    (float32)
  flags         quality flags 0-3      (int16,  0 = good)
  mask_visible  bool, True = observable
  nobs          number of ref-star obs (int32)
"""

import io
import os
import re
import sqlite3
import warnings
import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.dates as mdates
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import gaussian_filter
from scipy import ndimage, stats
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath
from lsst.summit.utils import ConsDbClient

warnings.filterwarnings("ignore")

os.environ.setdefault("no_proxy", "")
if ".consdb" not in os.environ["no_proxy"]:
    os.environ["no_proxy"] += ",.consdb"

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
NSIDE_EXPECTED = 32
NEST           = True
UNSEEN         = hp.UNSEEN if hasattr(hp, "UNSEEN") else np.nan

RUBIN_LAT      = -30.244639
RUBIN_LON      = -70.749417
RUBIN_HEIGHT_M = 2663.0

BIN_SIZE_KM  = 1000
R_PROJECTION = 10000.0   # km, height of projection plane above observer

CBAR_VMIN = -0.2
CBAR_VMAX  = 2.0

# Photometry
RUBIN_EFFECTIVE_AREA     = np.pi * (6.4 / 2) ** 2   # m²
RUBIN_QUANTUM_EFFICIENCY = 0.8
EXPOSURE_TIME            = 30.0                      # s
READOUT_TIME             = 2.0                       # s
SLOT_TIME                = EXPOSURE_TIME + READOUT_TIME  # 32 s
PHOTON_FLUX_MAG20_ZENITH = 100.0                     # photons s⁻¹ m⁻² QE-weighted

# Slew
MAX_SLEW_SPEED_ALT = 3.5   # deg/s
MAX_SLEW_SPEED_AZ  = 7.0   # deg/s
SLEW_SETTLE_TIME   = 2.0   # s

# Pixel quality thresholds
MAX_SIGMA_MAG  = 0.3
MAX_FLAG_VALUE = 0

# Horizon guard for scheduler
MIN_ALT_DEG = 15.0

# ── (a) Moon avoidance ──────────────────────────────────────────────────────
MOON_EXCLUSION_DEG = 30.0   # reject pointings within this angular radius of Moon

# ── (b) Scoring weights for motion-tracking candidate selection ─────────────
# score = w_cloud * normalised_extinction
#       + w_slew  * normalised_slew_cost
#       + w_zenith* normalised_zenith_angle
# Lower score = better.
W_CLOUD  = 0.50   # cloud extinction weight
W_SLEW   = 0.25   # slew-time weight (matches original slew-preference)
W_ZENITH = 0.25   # zenith-proximity weight (new)

# Normalisation references
MAX_EXTINCTION_NORM = 2.0   # mag — used to normalise extinction to [0,1]
MAX_SLEW_NORM       = 60.0  # s   — slew times beyond this are clipped to 1

# Patchiness thresholds
PATCHINESS_SAMPLE_N   = 10   # frames to sample for patchiness check
PATCHINESS_LOW_FRAC   = 0.10  # below this → mostly clear, skip
PATCHINESS_HIGH_FRAC  = 0.90  # above this → mostly cloudy, skip
PATCHINESS_VAR_THRESH = 0.05  # mag²; spatial variance threshold for "patchy"

# ConsDB / ZP cross-val
CONSDB_URL  = "http://consdb-pq.consdb:8080/consdb"
INSTRUMENT  = "lsstcam"
N_ZP_SAMPLES = 20
ZP_SEED      = 42
ALPHA_SEEING = 0.6
BETA_SEEING  = 0.2
FWHM_CONSTANT = 2 * np.sqrt(2 * np.log(2))
EFFECTIVE_WAVELENGTHS = {
    'u': 366.3, 'g': 481.0, 'r': 622.2,
    'i': 754.5, 'z': 869.1, 'y': 971.0
}
FILENAME_BAND_MAP = {
    'B': None, 'u': 'u', 'g': 'g',
    'r': 'r',  'i': 'i', 'z': 'z', 'y': 'y',
}

URL_COL  = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL = "time"

# ── (d) Spatial tensor dtype ─────────────────────────────────────────────────
# Each frame row: [frame_idx, unix_time, alt, az, x_km, y_km, ext, strategy_id]
# strategy_id: 0=absolute, 1=motion, 2=scheduler
TENSOR_DTYPE = np.float32


# ─────────────────────────────────────────────────────────────────────────────
# URL helpers
# ─────────────────────────────────────────────────────────────────────────────

def transform_url(url: str) -> str:
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url


def band_from_url(url: str):
    m = re.search(r'_([A-Za-z])_cloud_zps', url)
    return m.group(1) if m else None


# ─────────────────────────────────────────────────────────────────────────────
# HDF5 loaders
# ─────────────────────────────────────────────────────────────────────────────

def fetch_sys_map(url: str):
    """Load cloud_sys HDF5. Returns (clouds, sigma) with bad pixels → NaN."""
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    with h5py.File(io.BytesIO(data), "r") as f:
        clouds       = np.array(f["clouds"],       dtype=float).ravel()
        sigma        = np.array(f["sigma"],        dtype=float).ravel()
        flags        = np.array(f["flags"],        dtype=int  ).ravel()
        mask_visible = np.array(f["mask_visible"], dtype=bool ).ravel()
        nobs         = np.array(f["nobs"],         dtype=int  ).ravel()
    bad = (
        ~mask_visible
        | (nobs == 0)
        | (flags > MAX_FLAG_VALUE)
        | (sigma > MAX_SIGMA_MAG)
        | ~np.isfinite(clouds)
    )
    clouds[bad] = np.nan
    sigma [bad] = np.nan
    return clouds, sigma


def fetch_zps_map(url: str):
    """Load cloud_zps HDF5. Returns (clouds, sigma) with bad pixels → NaN."""
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    with h5py.File(io.BytesIO(data), "r") as f:
        clouds       = np.array(f["clouds"],       dtype=float).ravel()
        sigma        = np.array(f["sigma"],        dtype=float).ravel()
        flags        = np.array(f["flags"],        dtype=int  ).ravel()
        mask_visible = np.array(f["mask_visible"], dtype=bool ).ravel()
        nobs         = np.array(f["nobs"],         dtype=int  ).ravel()
    bad = (
        ~mask_visible
        | (nobs == 0)
        | (flags > MAX_FLAG_VALUE)
        | (sigma > MAX_SIGMA_MAG)
        | ~np.isfinite(clouds)
    )
    clouds[bad] = np.nan
    sigma [bad] = np.nan
    return clouds, sigma


# ─────────────────────────────────────────────────────────────────────────────
# Coordinate utilities
# ─────────────────────────────────────────────────────────────────────────────

def _make_location():
    return EarthLocation(lat=RUBIN_LAT*u.deg,
                         lon=RUBIN_LON*u.deg,
                         height=RUBIN_HEIGHT_M*u.m)


def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc


def radec_to_altaz(ra_deg, dec_deg, obstime):
    """Convert RA/Dec → (alt, az) in degrees."""
    t   = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    return float(aa.alt.deg), float(aa.az.deg % 360.0)


def altaz_to_xy(alt_deg, az_deg):
    """
    Project (alt, az) to flat-sky km coordinates.
    Returns (x_km, y_km) or (None, None) if below horizon.
    """
    if alt_deg < MIN_ALT_DEG:
        return None, None
    alt_r = np.radians(alt_deg)
    az_r  = np.radians(az_deg)
    scale = R_PROJECTION / np.sin(alt_r)
    x_km  = -np.cos(alt_r) * np.sin(az_r) * scale
    y_km  =  np.cos(alt_r) * np.cos(az_r) * scale
    return x_km, y_km


def xy_to_altaz(x_km, y_km):
    """Inverse projection: km → (alt, az) in degrees."""
    r       = np.sqrt(x_km**2 + y_km**2)
    alt_deg = float(np.degrees(np.arctan2(R_PROJECTION, r)))
    az_deg  = float(np.degrees(np.arctan2(-x_km, y_km)) % 360.0)
    return alt_deg, az_deg


def xy_to_zenith_angle(x_km, y_km):
    alt_deg, _ = xy_to_altaz(x_km, y_km)
    return 90.0 - alt_deg


def radec_to_xy(ra_deg, dec_deg, obstime):
    """
    Convert RA/Dec to flat-sky km, rejecting below-horizon pointings.
    Returns (x_km, y_km, alt_deg, az_deg); x/y are None if alt < MIN_ALT_DEG.
    """
    alt, az = radec_to_altaz(ra_deg, dec_deg, obstime)
    x, y    = altaz_to_xy(alt, az)
    return x, y, alt, az


def healpix_to_altaz_vals(mp, nside, obstime):
    """Map all HEALPix pixels to (az, alt, vals) arrays."""
    npix = hp.nside2npix(nside)
    pix  = np.arange(npix)
    theta, phi = hp.pix2ang(nside, pix, nest=NEST)
    ra  = np.degrees(phi)
    dec = 90.0 - np.degrees(theta)
    t   = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    vals = np.asarray(mp, dtype=float)
    vals = np.where(vals == UNSEEN, np.nan, vals)
    return aa.az.deg % 360.0, aa.alt.deg, vals


def _healpix_to_grid(mp, obstime, bin_size=BIN_SIZE_KM):
    """Grid a HEALPix map onto flat-sky km bins."""
    az, alt, vals = healpix_to_altaz_vals(mp, NSIDE_EXPECTED, obstime)
    alt_r = np.radians(alt)
    az_r  = np.radians(az)
    above = alt > MIN_ALT_DEG
    scale = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf    = -np.cos(alt_r) * np.sin(az_r) * scale
    yf    =  np.cos(alt_r) * np.cos(az_r) * scale
    vf    = np.where(above, vals, np.nan)

    r  = np.sqrt(xf**2 + yf**2)
    c  = r <= 15000.0
    ok = ~np.isnan(vf[c])

    x_edges = np.arange(-15000, 15001, bin_size)
    y_edges = np.arange(-15000, 15001, bin_size)
    Hs, _, _ = np.histogram2d(xf[c][ok], yf[c][ok],
                               bins=[x_edges, y_edges], weights=vf[c][ok])
    Hc, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[x_edges, y_edges])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (x_edges[:-1] + x_edges[1:]) / 2
    yc = (y_edges[:-1] + y_edges[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15000] = np.nan
    return H, Xg, Yg, x_edges, y_edges


def process_to_grid(clouds, sigma, obstime, bin_size=BIN_SIZE_KM):
    ext_grid, Xg, Yg, xe, ye = _healpix_to_grid(clouds, obstime, bin_size)
    sig_grid, *_              = _healpix_to_grid(sigma,  obstime, bin_size)
    return ext_grid, sig_grid, Xg, Yg, xe, ye


# ─────────────────────────────────────────────────────────────────────────────
# (a) Moon position helpers
# ─────────────────────────────────────────────────────────────────────────────

def get_moon_altaz(obstime):
    """
    Return (alt_deg, az_deg) of the Moon at obstime as seen from Rubin.
    Uses Astropy's get_body for the ephemeris.
    """
    t   = _ensure_time(obstime)
    loc = _make_location()
    moon = get_body("moon", t, loc)
    aa   = moon.transform_to(AltAz(obstime=t, location=loc))
    return float(aa.alt.deg), float(aa.az.deg % 360.0)


def angular_separation_altaz(alt1, az1, alt2, az2):
    """Great-circle angular separation between two alt/az pairs (degrees)."""
    a1, a2 = np.radians(alt1), np.radians(alt2)
    z1, z2 = np.radians(az1),  np.radians(az2)
    cos_sep = (np.sin(a1)*np.sin(a2)
               + np.cos(a1)*np.cos(a2)*np.cos(z1 - z2))
    cos_sep = np.clip(cos_sep, -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_sep)))


def is_moon_safe(alt_deg, az_deg, moon_alt, moon_az,
                 exclusion_deg=MOON_EXCLUSION_DEG):
    """True if the pointing is outside the moon exclusion zone."""
    if alt_deg < MIN_ALT_DEG:
        return False
    sep = angular_separation_altaz(alt_deg, az_deg, moon_alt, moon_az)
    return sep >= exclusion_deg


def xy_moon_safe(x_km, y_km, moon_alt, moon_az,
                 exclusion_deg=MOON_EXCLUSION_DEG):
    """True if grid position (x_km, y_km) is moon-safe."""
    alt, az = xy_to_altaz(x_km, y_km)
    return is_moon_safe(alt, az, moon_alt, moon_az, exclusion_deg)


# ─────────────────────────────────────────────────────────────────────────────
# Patchiness detection
# ─────────────────────────────────────────────────────────────────────────────

def compute_patchiness(grids):
    idx = np.linspace(0, len(grids) - 1, min(PATCHINESS_SAMPLE_N, len(grids)),
                      dtype=int)
    fracs, vars_ = [], []
    for i in idx:
        g = grids[i]
        valid = ~np.isnan(g)
        frac  = np.sum(valid) / g.size
        var   = float(np.nanvar(g)) if np.sum(valid) > 10 else 0.0
        fracs.append(frac)
        vars_.append(var)

    mean_frac = float(np.mean(fracs))
    mean_var  = float(np.mean(vars_))

    if mean_frac < PATCHINESS_LOW_FRAC:
        return False, mean_frac, mean_var, \
            f"Sky mostly clear (valid_frac={mean_frac:.2f} < {PATCHINESS_LOW_FRAC})"
    if mean_frac > PATCHINESS_HIGH_FRAC:
        return False, mean_frac, mean_var, \
            f"Sky mostly clouded out (valid_frac={mean_frac:.2f} > {PATCHINESS_HIGH_FRAC})"
    if mean_var < PATCHINESS_VAR_THRESH:
        return False, mean_frac, mean_var, \
            f"Low spatial variation (var={mean_var:.4f} < {PATCHINESS_VAR_THRESH}) — uniform cover"
    return True, mean_frac, mean_var, \
        f"Patchy clouds detected (valid_frac={mean_frac:.2f}, var={mean_var:.4f})"


# ─────────────────────────────────────────────────────────────────────────────
# Slew / photometry
# ─────────────────────────────────────────────────────────────────────────────

def calculate_slew_time(alt1, az1, alt2, az2):
    da  = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180:
        daz = 360 - daz
    return max(da / MAX_SLEW_SPEED_ALT, daz / MAX_SLEW_SPEED_AZ) + SLEW_SETTLE_TIME


def compute_photon_collection(ext_mag, _zenith_angle_deg=None):
    """Full-exposure photons (no slew penalty applied here)."""
    if np.isnan(ext_mag):
        return np.nan
    flux = 10 ** (-0.4 * ext_mag)
    rate = (PHOTON_FLUX_MAG20_ZENITH
            * RUBIN_EFFECTIVE_AREA
            * RUBIN_QUANTUM_EFFICIENCY
            * flux)
    return rate * EXPOSURE_TIME


def photons_to_magnitude(photons):
    if photons is None or np.isnan(photons) or photons <= 0:
        return np.nan
    snr = photons / (5 * np.sqrt(photons))
    return 20.0 - 2.5 * np.log10(snr) if snr > 0 else np.nan


# ─────────────────────────────────────────────────────────────────────────────
# Grid helpers
# ─────────────────────────────────────────────────────────────────────────────

def get_value_at_position(grid, x_km, y_km):
    xi = int(round((x_km / BIN_SIZE_KM) + 15))
    yi = int(round((y_km / BIN_SIZE_KM) + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan


def find_absolute_minimum(grid, moon_alt=None, moon_az=None):
    """
    Find the grid cell with minimum valid extinction.
    If moon_alt/moon_az are provided, cells within MOON_EXCLUSION_DEG of the
    Moon are masked out first.
    """
    work = grid.copy()

    # (a) Moon masking
    if moon_alt is not None and moon_az is not None:
        ny, nx = work.shape
        for yi in range(ny):
            for xi in range(nx):
                if np.isnan(work[yi, xi]):
                    continue
                x_km = (xi - 15) * BIN_SIZE_KM
                y_km = (yi - 15) * BIN_SIZE_KM
                if not xy_moon_safe(x_km, y_km, moon_alt, moon_az):
                    work[yi, xi] = np.nan

    if not np.any(~np.isnan(work)):
        # Fallback: ignore moon masking if it wipes everything
        work = grid.copy()
    if not np.any(~np.isnan(work)):
        return 0, 0, np.nan

    idx = np.nanargmin(work)
    yi, xi = np.unravel_index(idx, work.shape)
    return (xi - 15) * BIN_SIZE_KM, (yi - 15) * BIN_SIZE_KM, grid[yi, xi]


# ─────────────────────────────────────────────────────────────────────────────
# (b) Scored motion-tracking candidate selection
#     Combines cloud extinction, slew cost, and zenith proximity.
# ─────────────────────────────────────────────────────────────────────────────

def _score_candidate(x_km, y_km, ext_val, prev_alt, prev_az,
                     moon_alt=None, moon_az=None):
    """
    Return a scalar score (lower = better) for a candidate pointing.
    Returns np.inf if the candidate is invalid (NaN extinction, below horizon,
    inside moon exclusion zone).
    """
    if np.isnan(ext_val):
        return np.inf
    alt, az = xy_to_altaz(x_km, y_km)
    if alt < MIN_ALT_DEG:
        return np.inf
    if moon_alt is not None and moon_az is not None:
        if not is_moon_safe(alt, az, moon_alt, moon_az):
            return np.inf

    # Normalised extinction  [0, 1]
    cloud_norm = np.clip(ext_val / MAX_EXTINCTION_NORM, 0.0, 1.0)

    # Normalised slew cost  [0, 1]
    if prev_alt is not None:
        slew_s = calculate_slew_time(prev_alt, prev_az, alt, az)
    else:
        slew_s = 0.0
    slew_norm = np.clip(slew_s / MAX_SLEW_NORM, 0.0, 1.0)

    # Normalised zenith angle  [0, 1]
    zenith_norm = (90.0 - alt) / 90.0

    return W_CLOUD * cloud_norm + W_SLEW * slew_norm + W_ZENITH * zenith_norm


def predict_future_position(cx, cy, dx_pix, dy_pix, current_grid,
                             prev_alt=None, prev_az=None,
                             moon_alt=None, moon_az=None,
                             fallback_threshold=0.5,
                             n_candidates=9):
    """
    Predict next best pointing given cloud motion (dx_pix, dy_pix).

    Strategy:
      1. Primary: the motion-extrapolated position.
      2. Fallback grid: sample n_candidates positions around the primary,
         score each with _score_candidate, pick the best.
      3. If all candidates fail, fall back to find_absolute_minimum
         (with moon masking).
    """
    nx = cx - dx_pix * BIN_SIZE_KM
    ny = cy - dy_pix * BIN_SIZE_KM
    # Clamp to observable disk
    r = np.sqrt(nx**2 + ny**2)
    if r > 14000:
        nx *= 14000 / r
        ny *= 14000 / r

    # Build candidate grid around the primary prediction
    offsets = [0] + list(range(-2, 3))  # -2,-1,0,1,2 in each axis
    candidates = []
    for dyi in offsets:
        for dxi in offsets:
            cx_ = nx + dxi * BIN_SIZE_KM
            cy_ = ny + dyi * BIN_SIZE_KM
            ext = get_value_at_position(current_grid, cx_, cy_)
            candidates.append((cx_, cy_, ext))

    best_score = np.inf
    best_x, best_y = nx, ny
    for (cx_, cy_, ext) in candidates:
        s = _score_candidate(cx_, cy_, ext, prev_alt, prev_az,
                             moon_alt, moon_az)
        if s < best_score:
            best_score = s
            best_x, best_y = cx_, cy_

    if best_score == np.inf:
        # All candidates failed — fall back to global minimum with moon masking
        best_x, best_y, _ = find_absolute_minimum(current_grid,
                                                   moon_alt, moon_az)
    return best_x, best_y


# ─────────────────────────────────────────────────────────────────────────────
# Cloud motion tracking
# ─────────────────────────────────────────────────────────────────────────────

def compute_motion_with_correlation(grid1, grid2, sigma=5.0, search_range=10):
    g1 = np.nan_to_num(grid1, nan=0)
    g2 = np.nan_to_num(grid2, nan=0)
    g1s = gaussian_filter(g1, sigma=sigma)
    g2s = gaussian_filter(g2, sigma=sigma)
    m1  = ~np.isnan(grid1) & (grid1 != 0)
    m2  = ~np.isnan(grid2) & (grid2 != 0)
    best_corr, best_dx, best_dy = -np.inf, 0, 0
    for dy in range(-search_range, search_range + 1):
        for dx in range(-search_range, search_range + 1):
            sh  = ndimage.shift(g2s,       (dy, dx), order=1, mode="constant", cval=0)
            sm  = ndimage.shift(m2.astype(float), (dy, dx), order=0, mode="constant", cval=0) > 0.5
            val = m1 & sm
            if np.sum(val) < 100:
                continue
            v1, v2 = g1s[val], sh[val]
            if np.std(v1) > 0 and np.std(v2) > 0:
                corr = np.corrcoef(v1, v2)[0, 1]
                if corr > best_corr:
                    best_corr, best_dx, best_dy = corr, dx, dy
    return best_dx, best_dy, best_corr


def compute_all_positions(all_grids, all_metas):
    """
    Compute absolute and predicted positions for all frames.
    Moon position computed once per frame from the frame timestamp.
    all_metas must contain 'time' keys.
    """
    t0      = all_metas[0]["time"]
    m_alt0, m_az0 = get_moon_altaz(t0)

    x0, y0, _ = find_absolute_minimum(all_grids[0], m_alt0, m_az0)
    abs_pos  = [(x0, y0)]
    pred_pos = [(x0, y0)]
    motion_v = [(0, 0)]
    moon_pos = [(m_alt0, m_az0)]
    xp, yp   = x0, y0
    prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)
    HL = 3

    for i in range(1, len(all_grids)):
        if i % 50 == 0:
            print(f"  Tracking frame {i}/{len(all_grids)}")

        m_alt, m_az = get_moon_altaz(all_metas[i]["time"])
        moon_pos.append((m_alt, m_az))

        xa, ya, _ = find_absolute_minimum(all_grids[i], m_alt, m_az)
        abs_pos.append((xa, ya))

        if i >= HL:
            dxs, dys = [], []
            for j in range(1, HL + 1):
                dx, dy, conf = compute_motion_with_correlation(
                    all_grids[i-j], all_grids[i-j+1])
                if conf > 0.5:
                    dxs.append(dx); dys.append(dy)
            dx_avg = float(np.mean(dxs)) if dxs else 0.0
            dy_avg = float(np.mean(dys)) if dys else 0.0
        else:
            dx_avg, dy_avg, _ = compute_motion_with_correlation(
                all_grids[i-1], all_grids[i])

        motion_v.append((dx_avg, dy_avg))

        # (a)+(b) scored prediction with moon avoidance + zenith weighting
        xp, yp = predict_future_position(
            xp, yp, dx_avg, dy_avg, all_grids[i],
            prev_alt=prev_alt_p, prev_az=prev_az_p,
            moon_alt=m_alt, moon_az=m_az)
        pred_pos.append((xp, yp))
        prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)

    return abs_pos, pred_pos, motion_v, moon_pos


# ─────────────────────────────────────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────────────────────────────────────

def load_all_sys_frames(csv_file):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains(".hdf5",     case=False, na=False)].copy()
    df = df[df[URL_COL].str.contains("cloud_sys", case=False, na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    shifted = df[TIME_COL] - pd.Timedelta(hours=12)
    df["night_key"] = shifted.dt.date
    return df


def load_all_zps_frames(csv_file):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains(".hdf5",     case=False, na=False)].copy()
    df = df[df[URL_COL].str.contains("cloud_zps", case=False, na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    shifted = df[TIME_COL] - pd.Timedelta(hours=12)
    df["night_key"] = shifted.dt.date
    df["dream_band"] = df[URL_COL].apply(band_from_url)
    return df


def get_night_df(all_df, night_key):
    return all_df[all_df["night_key"] == night_key].copy().reset_index(drop=True)


def quick_patchiness_check(df_night, n_probe=PATCHINESS_SAMPLE_N):
    if len(df_night) == 0:
        return False, 0.0, 0.0, "No frames", []
    idx    = np.linspace(0, len(df_night) - 1, min(n_probe, len(df_night)), dtype=int)
    grids  = []
    metas  = []
    for i in idx:
        row = df_night.iloc[i]
        url = transform_url(row[URL_COL])
        try:
            clouds, sigma = fetch_sys_map(url)
            meta = {"url": url, "time": row[TIME_COL].to_pydatetime()}
            ext_g, sig_g, _, _, _, _ = process_to_grid(clouds, sigma, meta["time"])
            grids.append(ext_g)
            metas.append(meta)
        except Exception as e:
            print(f"    probe load failed: {e}")
    if len(grids) < 2:
        return False, 0.0, 0.0, "Too few probe frames loaded", []
    patchy, mean_frac, mean_var, reason = compute_patchiness(grids)
    return patchy, mean_frac, mean_var, reason, grids


def load_scheduler_observations(db_file):
    print("\n" + "=" * 70)
    print("LOADING SCHEDULER OBSERVATIONS")
    print("=" * 70)
    conn   = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]
    table_name = next(
        (n for n in ["observations", "SummaryAllProps", "Summary", "obs"]
         if n in tables),
        tables[0] if tables else None)
    print(f"Using table: {table_name}")
    obs_df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()

    night_col = next((c for c in obs_df.columns if "night" in c.lower()), None)
    if night_col is None:
        raise RuntimeError("Could not find night column in scheduler DB")
    obs_df["night"] = obs_df[night_col].astype(int)
    for req in ["observationStartMJD", "fieldRA", "fieldDec"]:
        for col in obs_df.columns:
            if req.lower() in col.lower() and col != req:
                obs_df[req] = obs_df[col]

    night_scores = {}
    for night in obs_df["night"].unique()[:500]:
        no  = obs_df[obs_df["night"] == night]
        if len(no) < 200:
            continue
        dur = (no["observationStartMJD"].max() - no["observationStartMJD"].min()) * 24
        if dur < 5:
            continue
        night_scores[night] = (len(no) * 10 + dur, len(no), dur)
    if not night_scores:
        selected = obs_df.groupby("night").size().idxmax()
    else:
        selected = max(night_scores, key=lambda k: night_scores[k][0])
        _, n, dur = night_scores[selected]
        print(f"  Night {selected}: {n} obs over {dur:.1f} hours")

    night_obs = obs_df[obs_df["night"] == selected].copy()
    night_obs = night_obs.sort_values("observationStartMJD").reset_index(drop=True)
    print(f"Selected night {selected}: {len(night_obs)} observations")
    return night_obs, selected


def match_scheduler_to_frames(scheduler_obs, frame_times):
    n         = len(frame_times)
    fracs     = np.linspace(0, 1, n)
    mjd       = scheduler_obs["observationStartMJD"].values
    mjd_range = mjd.max() - mjd.min()
    s_fracs   = (mjd - mjd.min()) / mjd_range if mjd_range > 0 else np.zeros(len(mjd))
    matched   = [scheduler_obs.iloc[np.argmin(np.abs(s_fracs - f))] for f in fracs]
    print(f"  Matched {len(matched)} frames to scheduler observations")
    return matched


# ─────────────────────────────────────────────────────────────────────────────
# Metrics computation
# ─────────────────────────────────────────────────────────────────────────────

def _record_obs(metrics, key, photons_full, sigma_val, za, ext,
                slew_t, frame_i, alt, az, prev_alt, prev_az,
                x_km, y_km):
    expose_t  = max(0.0, SLOT_TIME - slew_t)
    collected = photons_full * (expose_t / EXPOSURE_TIME)
    metrics[key]["photons"].append(collected)
    metrics[key]["sigma"].append(sigma_val)
    metrics[key]["zenith_angles"].append(za)
    metrics[key]["extinctions"].append(ext)
    metrics[key]["slew_times"].append(slew_t)
    metrics[key]["expose_times"].append(expose_t)
    metrics[key]["frame_indices"].append(frame_i)
    metrics[key]["alt_az"].append((alt, az))
    metrics[key]["xy_km"].append((x_km, y_km))   # (d) store pointing xy
    prev_alt[key] = alt
    prev_az[key]  = az


def compute_metrics(all_grids, all_sigma_grids, all_metas,
                    abs_positions, pred_positions, moon_positions,
                    matched_scheduler):
    print("\n" + "=" * 70)
    print("COMPUTING METRICS (photon collection + slew gating)")
    print("=" * 70)
    print(f"  Slot = {EXPOSURE_TIME}s exp + {READOUT_TIME}s readout = {SLOT_TIME}s")
    print(f"  Horizon guard: alt ≥ {MIN_ALT_DEG}°")
    print(f"  Moon exclusion: {MOON_EXCLUSION_DEG}°  "
          f"| Zenith weight: {W_ZENITH}  Slew weight: {W_SLEW}")

    empty = {"photons": [], "sigma": [], "zenith_angles": [], "extinctions": [],
             "slew_times": [], "expose_times": [], "frame_indices": [],
             "alt_az": [], "xy_km": []}
    metrics  = {s: {k: list(v) for k, v in empty.items()}
                for s in ("absolute", "motion", "scheduler")}
    prev_alt = {s: None for s in metrics}
    prev_az  = {s: None for s in metrics}
    sched_rejected = 0

    for i in range(len(all_grids)):
        grid  = all_grids[i]
        sgrid = all_sigma_grids[i]
        m_alt, m_az = moon_positions[i]

        # ── Absolute minimum  (a: moon-masked) ───────────────────────────────
        xa, ya = abs_positions[i]
        alt_a, az_a = xy_to_altaz(xa, ya)
        if alt_a >= MIN_ALT_DEG and is_moon_safe(alt_a, az_a, m_alt, m_az):
            ext = get_value_at_position(grid,  xa, ya)
            sig = get_value_at_position(sgrid, xa, ya)
            za  = xy_to_zenith_angle(xa, ya)
            if not np.isnan(ext):
                slew = (calculate_slew_time(prev_alt["absolute"], prev_az["absolute"],
                                            alt_a, az_a)
                        if prev_alt["absolute"] is not None else 0.0)
                _record_obs(metrics, "absolute",
                            compute_photon_collection(ext),
                            sig, za, ext, slew, i, alt_a, az_a, prev_alt, prev_az,
                            xa, ya)

        # ── Motion tracking  (a+b: moon-masked + zenith-weighted) ────────────
        xp, yp = pred_positions[i]
        alt_p, az_p = xy_to_altaz(xp, yp)
        if alt_p >= MIN_ALT_DEG and is_moon_safe(alt_p, az_p, m_alt, m_az):
            ext = get_value_at_position(grid,  xp, yp)
            sig = get_value_at_position(sgrid, xp, yp)
            za  = xy_to_zenith_angle(xp, yp)
            if not np.isnan(ext):
                slew = (calculate_slew_time(prev_alt["motion"], prev_az["motion"],
                                            alt_p, az_p)
                        if prev_alt["motion"] is not None else 0.0)
                _record_obs(metrics, "motion",
                            compute_photon_collection(ext),
                            sig, za, ext, slew, i, alt_p, az_p, prev_alt, prev_az,
                            xp, yp)

        # ── Scheduler (unchanged — scheduler owns its own targeting logic) ───
        obs = matched_scheduler[i]
        xs, ys, alt_s, az_s = radec_to_xy(
            float(obs["fieldRA"]), float(obs["fieldDec"]),
            all_metas[i]["time"])
        if xs is None:
            sched_rejected += 1
            continue
        ext = get_value_at_position(grid,  xs, ys)
        sig = get_value_at_position(sgrid, xs, ys)
        za  = xy_to_zenith_angle(xs, ys)
        if not np.isnan(ext):
            slew = (calculate_slew_time(prev_alt["scheduler"], prev_az["scheduler"],
                                        alt_s, az_s)
                    if prev_alt["scheduler"] is not None else 0.0)
            _record_obs(metrics, "scheduler",
                        compute_photon_collection(ext),
                        sig, za, ext, slew, i, alt_s, az_s, prev_alt, prev_az,
                        xs, ys)

    if sched_rejected > 0:
        print(f"  ⚠ Scheduler: {sched_rejected} frames rejected (field below "
              f"alt={MIN_ALT_DEG}°)")
    return metrics


def print_statistics(metrics):
    print("\n" + "=" * 70)
    print("NIGHT PERFORMANCE SUMMARY")
    print("=" * 70)
    for strategy in ("absolute", "motion", "scheduler"):
        photons      = np.array(metrics[strategy]["photons"])
        expose_times = np.array(metrics[strategy]["expose_times"])
        slew_times   = np.array(metrics[strategy]["slew_times"])
        sigmas       = np.array(metrics[strategy]["sigma"])
        if len(photons) == 0:
            print(f"\n{strategy.upper()}: no valid data")
            continue

        total_ph   = np.sum(photons)
        total_exp  = np.sum(expose_times)
        total_slew = np.sum(slew_times)
        total_slot = len(photons) * SLOT_TIME
        eff        = total_exp / total_slot * 100 if total_slot > 0 else 0
        pct_lost   = np.mean(expose_times == 0) * 100
        mags       = [photons_to_magnitude(p) for p in photons]
        mags       = [m for m in mags if m is not None and not np.isnan(m)]
        med_depth  = np.median(mags) if mags else np.nan

        print(f"\n{strategy.upper()} STRATEGY:")
        print(f"  Total photons collected:     {total_ph:.3e}")
        print(f"  Number of slots:             {len(photons)}")
        print(f"  On-sky exposure time:        {total_exp:.0f} s  ({total_exp/3600:.2f} hr)")
        print(f"  Total slew time:             {total_slew:.0f} s  ({total_slew/3600:.2f} hr)")
        print(f"  Shutter-open efficiency:     {eff:.1f}%")
        print(f"  Slots lost to slew:          {pct_lost:.1f}%")
        print(f"  Median 5σ depth:             {med_depth:.2f} mag")
        print(f"  Mean pixel σ:                {np.nanmean(sigmas):.4f} mag")
        print(f"  Mean zenith angle:           {np.mean(metrics[strategy]['zenith_angles']):.1f}°")
        print(f"  Mean extinction:             {np.mean(metrics[strategy]['extinctions']):.3f} mag")


# ─────────────────────────────────────────────────────────────────────────────
# (c) Atmospheric + seeing quality correction using ConsDB quicklook columns
# ─────────────────────────────────────────────────────────────────────────────

def _safe_median(arr):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    return float(np.median(a)) if len(a) else np.nan


def _find_nearest_visits(consdb_df, x_km, y_km, obstime, window_s=300.0,
                         max_sep_deg=2.0):
    """
    Return rows from consdb_df whose pointing is within max_sep_deg of
    (x_km, y_km) and whose obs_midpoint is within ±window_s of obstime.
    Returns empty DataFrame if no match.
    """
    if consdb_df.empty:
        return pd.DataFrame()
    if isinstance(obstime, pd.Timestamp):
        t_center = obstime if obstime.tzinfo is not None else obstime.tz_localize("UTC")
    else:
        ts = pd.Timestamp(obstime)
        t_center = ts if ts.tzinfo is not None else ts.tz_localize("UTC")
    t_lo = t_center - pd.Timedelta(seconds=window_s)
    t_hi = t_center + pd.Timedelta(seconds=window_s)

    nearby = consdb_df[
        (consdb_df["obs_midpoint"] >= t_lo) &
        (consdb_df["obs_midpoint"] <= t_hi)
    ].copy()
    if nearby.empty:
        return nearby

    # Angular separation filter
    alt_p, az_p = xy_to_altaz(x_km, y_km)
    seps = []
    for _, row in nearby.iterrows():
        ra_v  = float(row.get("ra",  row.get("s_ra",  np.nan)))
        dec_v = float(row.get("dec", row.get("s_dec", np.nan)))
        if np.isnan(ra_v) or np.isnan(dec_v):
            seps.append(np.inf)
            continue
        alt_v, az_v = radec_to_altaz(ra_v, dec_v, t_center)
        seps.append(angular_separation_altaz(alt_p, az_p, alt_v, az_v))
    nearby = nearby[np.array(seps) <= max_sep_deg].copy()
    return nearby


def apply_atm_seeing_correction(metrics, all_metas, consdb_df):
    """
    (c) Post-process metrics to add seeing- and atmosphere-corrected photon
    estimates for each recorded slot.

    For every slot in every strategy, we look up the nearest ConsDB visit(s)
    and extract:
      - airmass                 → atmospheric transmission correction
      - psf_sigma_median        → seeing FWHM proxy
      - eff_time_median / eff_time_zero_point_scale_median → effective-time scale
      - zero_point_median       → zeropoint offset
      - seeing_zenith_500nm_median → zenith-normalised seeing

    The corrected metric is stored alongside the raw one.
    New keys added to each strategy sub-dict:
      photons_atm_corrected : list[float]   raw photons × eff_time_scale
      eff_time_scale        : list[float]   eff_time_zero_point_scale_median
      seeing_fwhm           : list[float]   arcsec FWHM at pointing
      airmass_consdb        : list[float]
      zero_point_consdb     : list[float]
    """
    print("\n" + "=" * 70)
    print("APPLYING ATMOSPHERIC + SEEING CORRECTIONS  (ConsDB quicklook)")
    print("=" * 70)

    if consdb_df is None or consdb_df.empty:
        print("  No ConsDB data — skipping corrections.")
        for s in metrics:
            n = len(metrics[s]["photons"])
            metrics[s]["photons_atm_corrected"] = [np.nan] * n
            metrics[s]["eff_time_scale"]        = [np.nan] * n
            metrics[s]["seeing_fwhm"]           = [np.nan] * n
            metrics[s]["airmass_consdb"]        = [np.nan] * n
            metrics[s]["zero_point_consdb"]     = [np.nan] * n
        return metrics

    for strategy in ("absolute", "motion", "scheduler"):
        n = len(metrics[strategy]["photons"])
        ph_corr   = []
        eft_scale = []
        see_fwhm  = []
        am_list   = []
        zp_list   = []

        for slot_i in range(n):
            frame_i = metrics[strategy]["frame_indices"][slot_i]
            x_km, y_km = metrics[strategy]["xy_km"][slot_i]
            obstime = all_metas[frame_i]["time"]
            raw_ph  = metrics[strategy]["photons"][slot_i]

            nearby = _find_nearest_visits(consdb_df, x_km, y_km, obstime)

            if nearby.empty:
                ph_corr.append(raw_ph)
                eft_scale.append(np.nan)
                see_fwhm.append(np.nan)
                am_list.append(np.nan)
                zp_list.append(np.nan)
                continue

            # Aggregate multiple close visits
            am  = _safe_median(nearby.get("airmass",                         [np.nan]))
            eft = _safe_median(nearby.get("eff_time_zero_point_scale_median", [np.nan]))
            if np.isnan(eft):
                eft = _safe_median(nearby.get("eff_time_median", [np.nan]))
            zp  = _safe_median(nearby.get("zero_point_median",               [np.nan]))

            # Seeing: prefer seeing_zenith_500nm_median, fall back to
            # psf_sigma_median → FWHM via FWHM_CONSTANT
            sz  = _safe_median(nearby.get("seeing_zenith_500nm_median", [np.nan]))
            if np.isnan(sz):
                ps = _safe_median(nearby.get("psf_sigma_median", [np.nan]))
                sz = float(FWHM_CONSTANT * ps) if np.isfinite(ps) else np.nan

            # Scale photons by effective-time scale factor (captures
            # atmosphere + sky background + PSF quality together)
            scale = eft if np.isfinite(eft) else 1.0
            ph_corr.append(raw_ph * scale)
            eft_scale.append(eft)
            see_fwhm.append(sz)
            am_list.append(am)
            zp_list.append(zp)

        metrics[strategy]["photons_atm_corrected"] = ph_corr
        metrics[strategy]["eff_time_scale"]        = eft_scale
        metrics[strategy]["seeing_fwhm"]           = see_fwhm
        metrics[strategy]["airmass_consdb"]        = am_list
        metrics[strategy]["zero_point_consdb"]     = zp_list

        n_matched = sum(1 for v in eft_scale if np.isfinite(v))
        print(f"  {strategy:12s}: {n_matched}/{n} slots matched to ConsDB visits")
        if n_matched > 0:
            print(f"    mean eff_time_scale = {np.nanmean(eft_scale):.4f}")
            print(f"    mean seeing FWHM    = {np.nanmean(see_fwhm):.3f} arcsec")

    return metrics


def print_corrected_statistics(metrics):
    """Print the atm/seeing-corrected summary."""
    print("\n" + "=" * 70)
    print("CORRECTED PERFORMANCE SUMMARY  (eff_time_scale applied)")
    print("=" * 70)
    for strategy in ("absolute", "motion", "scheduler"):
        ph_c = np.array(metrics[strategy].get("photons_atm_corrected", []))
        if len(ph_c) == 0 or not np.any(np.isfinite(ph_c)):
            print(f"\n{strategy.upper()}: no corrected data")
            continue
        print(f"\n{strategy.upper()} (corrected):")
        print(f"  Total corrected photons:  {np.nansum(ph_c):.3e}")
        print(f"  Mean eff_time_scale:      "
              f"{np.nanmean(metrics[strategy]['eff_time_scale']):.4f}")
        print(f"  Mean seeing FWHM:         "
              f"{np.nanmean(metrics[strategy]['seeing_fwhm']):.3f} arcsec")
        print(f"  Mean airmass (ConsDB):    "
              f"{np.nanmean(metrics[strategy]['airmass_consdb']):.3f}")


# ─────────────────────────────────────────────────────────────────────────────
# (d) Spatial pointing tensor I/O
# ─────────────────────────────────────────────────────────────────────────────
# Schema per row: [frame_idx, unix_time, alt_deg, az_deg, x_km, y_km,
#                  extinction, strategy_id(0/1/2)]
# Saved as float32 array shape (N_slots, 8).

STRATEGY_IDS = {"absolute": 0, "motion": 1, "scheduler": 2}


def build_pointing_tensor(metrics, all_metas):
    """
    Assemble an (N, 8) float32 array of all pointing records across all
    strategies for a single night.
    Columns: frame_idx, unix_time, alt, az, x_km, y_km, extinction, strategy_id
    """
    rows = []
    for strategy, sid in STRATEGY_IDS.items():
        fi_list   = metrics[strategy]["frame_indices"]
        xy_list   = metrics[strategy]["xy_km"]
        ext_list  = metrics[strategy]["extinctions"]
        for k in range(len(fi_list)):
            fi       = fi_list[k]
            x_km, y_km = xy_list[k]
            alt, az  = xy_to_altaz(x_km, y_km)
            t_unix   = float(all_metas[fi]["time"].timestamp()) \
                if hasattr(all_metas[fi]["time"], "timestamp") \
                else float(pd.Timestamp(all_metas[fi]["time"]).timestamp())
            ext      = ext_list[k]
            rows.append([float(fi), t_unix, alt, az, x_km, y_km,
                         float(ext), float(sid)])
    if not rows:
        return np.zeros((0, 8), dtype=TENSOR_DTYPE)
    return np.array(rows, dtype=TENSOR_DTYPE)


def save_pointing_tensor(tensor, output_path, night_key,
                         all_tensors_accumulator=None):
    """
    Save per-night pointing tensor as .npz.
    Also appends to all_tensors_accumulator list for cross-night aggregation.
    """
    np.savez_compressed(
        output_path,
        pointing_tensor=tensor,
        night_key=np.array([str(night_key)]),
        columns=np.array(["frame_idx", "unix_time", "alt_deg", "az_deg",
                          "x_km", "y_km", "extinction", "strategy_id"]),
        strategy_ids=np.array([0, 1, 2]),
        strategy_names=np.array(["absolute", "motion", "scheduler"]),
    )
    print(f"  Pointing tensor saved: {output_path}  "
          f"({tensor.shape[0]} rows × {tensor.shape[1]} cols)")
    if all_tensors_accumulator is not None:
        all_tensors_accumulator.append(tensor)


def save_all_nights_tensor(all_tensors, output_path):
    """Concatenate per-night tensors and save as one .npz."""
    if not all_tensors:
        print("  No tensors to aggregate.")
        return
    combined = np.concatenate(all_tensors, axis=0)
    np.savez_compressed(
        output_path,
        pointing_tensor=combined,
        columns=np.array(["frame_idx", "unix_time", "alt_deg", "az_deg",
                          "x_km", "y_km", "extinction", "strategy_id"]),
        strategy_ids=np.array([0, 1, 2]),
        strategy_names=np.array(["absolute", "motion", "scheduler"]),
    )
    print(f"\nAll-nights pointing tensor saved: {output_path}  "
          f"({combined.shape[0]} total rows)")


# ─────────────────────────────────────────────────────────────────────────────
# Metrics plot  (LEFT side only — no live-video subplot)
# ─────────────────────────────────────────────────────────────────────────────

def create_metrics_plot(metrics, output_file="metrics_jan25_2026.png"):
    print("\n" + "=" * 70)
    print("CREATING METRICS PLOT")
    print("=" * 70)

    strategies = ("absolute", "motion", "scheduler")
    colors     = ("green", "blue", "red")
    labels     = ("Absolute Min", "Motion Tracking", "Scheduler")

    fig = plt.figure(figsize=(18, 14))
    gs  = fig.add_gridspec(4, 3, hspace=0.40, wspace=0.32)

    # ── Row 0: Zenith | Slew | σ distributions ───────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    for s, c, l in zip(strategies, colors, labels):
        if metrics[s]["zenith_angles"]:
            ax.hist(metrics[s]["zenith_angles"], bins=20, alpha=0.6,
                    color=c, label=l, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Zenith Angle (°)"); ax.set_ylabel("Count")
    ax.set_title("Zenith Angle Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 1])
    for s, c, l in zip(strategies, colors, labels):
        slew = metrics[s]["slew_times"][1:]
        if slew:
            ax.hist(slew, bins=30, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Slew Time (s)"); ax.set_ylabel("Count")
    ax.set_title("Slew Time Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 2])
    for s, c, l in zip(strategies, colors, labels):
        sigs = [x for x in metrics[s]["sigma"] if not np.isnan(x)]
        if sigs:
            ax.hist(sigs, bins=30, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.axvline(MAX_SIGMA_MAG, color="k", linestyle="--", lw=1.5,
               label=f"Mask threshold ({MAX_SIGMA_MAG})")
    ax.set_xlabel("Pixel σ uncertainty (mag)"); ax.set_ylabel("Count")
    ax.set_title("Extinction Uncertainty Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # ── Row 1: Photon timeline (full width) ──────────────────────────────────
    ax = fig.add_subplot(gs[1, :])
    for s, c, l in zip(strategies, colors, labels):
        fi = metrics[s]["frame_indices"]
        ph = metrics[s]["photons"]
        ph_c = metrics[s].get("photons_atm_corrected", [])
        if fi:
            ax.plot(fi, ph, color=c, alpha=0.45, linewidth=1.2,
                    label=f"{l} (raw)")
            if len(ph_c) == len(fi) and np.any(np.isfinite(ph_c)):
                ax.plot(fi, ph_c, color=c, alpha=0.9, linewidth=1.8,
                        linestyle="--", label=f"{l} (atm-corrected)")
    ax.set_xlabel("Frame Number (Time →)")
    ax.set_ylabel("Slew-gated photons per slot")
    ax.set_title("Photon Collection Over the Night  "
                 "(solid=raw, dashed=atm+seeing corrected)", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

    # ── Row 2: Total photons (raw + corrected) | Median depth | Efficiency ───
    ax = fig.add_subplot(gs[2, 0])
    x_pos  = np.arange(len(strategies))
    width  = 0.35
    totals_raw = [np.sum(metrics[s]["photons"]) if metrics[s]["photons"] else 0
                  for s in strategies]
    totals_cor = [np.nansum(metrics[s].get("photons_atm_corrected", [np.nan]))
                  for s in strategies]
    bars1 = ax.bar(x_pos - width/2, totals_raw, width, color=colors,
                   alpha=0.55, edgecolor="black", linewidth=1.2, label="raw")
    bars2 = ax.bar(x_pos + width/2, totals_cor, width, color=colors,
                   alpha=0.90, edgecolor="black", linewidth=1.2,
                   hatch="//", label="atm-corrected")
    baseline = totals_raw[0] if totals_raw[0] > 0 else 1
    for i, t in enumerate(totals_raw):
        ax.text(i - width/2, t, f"{((t-baseline)/baseline)*100:+.1f}%",
                ha="center", va="bottom", fontsize=9, weight="bold")
    ax.set_xticks(x_pos); ax.set_xticklabels(labels)
    ax.set_ylabel("Total Photons")
    ax.set_title("Total Night Photon Collection", weight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
    ax.ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

    ax = fig.add_subplot(gs[2, 1])
    depths = []
    for s in strategies:
        mags = [photons_to_magnitude(p) for p in metrics[s]["photons"]]
        mags = [m for m in mags if m is not None and not np.isnan(m)]
        depths.append(np.median(mags) if mags else 0)
    bars = ax.bar(labels, depths, color=colors, alpha=0.7,
                  edgecolor="black", linewidth=1.5)
    for bar, val in zip(bars, depths):
        ax.text(bar.get_x() + bar.get_width()/2, val, f"{val:.2f}",
                ha="center", va="bottom", fontsize=10, weight="bold")
    ax.set_ylabel("Median 5σ Depth (mag)")
    ax.set_title("Survey Depth (slew-scaled)", weight="bold")
    ax.grid(alpha=0.3, axis="y"); ax.invert_yaxis()

    ax = fig.add_subplot(gs[2, 2])
    effs = []
    for s in strategies:
        et    = np.array(metrics[s]["expose_times"])
        slots = len(et) * SLOT_TIME
        effs.append(np.sum(et) / slots * 100 if slots > 0 else 0)
    bars = ax.bar(labels, effs, color=colors, alpha=0.7,
                  edgecolor="black", linewidth=1.5)
    for bar, val in zip(bars, effs):
        ax.text(bar.get_x() + bar.get_width()/2, val, f"{val:.1f}%",
                ha="center", va="bottom", fontsize=10, weight="bold")
    ax.set_ylabel("Shutter-open efficiency (%)")
    ax.set_title("Observation Efficiency", weight="bold")
    ax.grid(alpha=0.3, axis="y"); ax.set_ylim(0, 105)

    # ── Row 3: Seeing FWHM and eff_time_scale from ConsDB ────────────────────
    ax = fig.add_subplot(gs[3, 0])
    for s, c, l in zip(strategies, colors, labels):
        sf = [v for v in metrics[s].get("seeing_fwhm", []) if np.isfinite(v)]
        if sf:
            ax.hist(sf, bins=20, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Seeing FWHM (arcsec)"); ax.set_ylabel("Count")
    ax.set_title("Seeing Distribution (ConsDB)", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[3, 1])
    for s, c, l in zip(strategies, colors, labels):
        fi  = metrics[s]["frame_indices"]
        ets = metrics[s].get("eff_time_scale", [])
        if fi and len(ets) == len(fi):
            ax.plot(fi, ets, color=c, alpha=0.8, linewidth=1.5, label=l)
    ax.axhline(1.0, color="k", ls="--", lw=1.2, label="ideal (1.0)")
    ax.set_xlabel("Frame Number"); ax.set_ylabel("eff_time_scale")
    ax.set_title("Effective-Time Scale Factor (ConsDB)", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[3, 2])
    for s, c, l in zip(strategies, colors, labels):
        fi  = metrics[s]["frame_indices"]
        am  = metrics[s].get("airmass_consdb", [])
        if fi and len(am) == len(fi):
            am_f = [v if np.isfinite(v) else np.nan for v in am]
            ax.plot(fi, am_f, color=c, alpha=0.8, linewidth=1.5, label=l)
    ax.set_xlabel("Frame Number"); ax.set_ylabel("Airmass")
    ax.set_title("Airmass Over Night (ConsDB)", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    plt.suptitle(
        "Rubin Observatory Pointing Strategy Comparison\n"
        "(sys map  |  σ-masked  |  slew-gated  |  moon-avoided  |"
        "  zenith-weighted  |  atm+seeing corrected)",
        fontsize=12, weight="bold", y=0.999)
    plt.savefig(output_file, dpi=150, bbox_inches="tight")
    print(f"  Metrics plot saved: {output_file}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# Video  (map panel only)
# ─────────────────────────────────────────────────────────────────────────────

def create_video(all_grids, all_sigma_grids, all_metas,
                 abs_positions, pred_positions, motion_vectors, moon_positions,
                 matched_scheduler, x_edges, y_edges,
                 output_file="cloud_tracking.mp4", fps=10):
    print("\n" + "=" * 70)
    print("CREATING VIDEO (map panel only)")
    print("=" * 70)

    sched_xy = []
    for i, obs in enumerate(matched_scheduler):
        x, y, alt, az = radec_to_xy(
            float(obs["fieldRA"]), float(obs["fieldDec"]),
            all_metas[i]["time"])
        sched_xy.append((x, y, alt, az))

    fig, ax_map = plt.subplots(1, 1, figsize=(12, 10))

    def update(fi):
        ax_map.clear()
        grid  = all_grids[fi]
        sgrid = all_sigma_grids[fi]
        ts    = max(0, fi - 12)
        m_alt, m_az = moon_positions[fi]

        ax_map.pcolormesh(x_edges, y_edges, grid,
                          cmap="viridis", vmin=CBAR_VMIN, vmax=CBAR_VMAX,
                          shading="flat", alpha=0.85)
        th = np.linspace(0, 2*np.pi, 300)
        ax_map.plot(15000*np.cos(th), 15000*np.sin(th),
                    "k-", lw=2, alpha=0.5)
        ax_map.plot(0, 0, "r+", ms=15, mew=3, label="Zenith", zorder=5)

        # Moon exclusion circle (if moon is above horizon)
        if m_alt > 0:
            moon_x, moon_y = altaz_to_xy(m_alt, m_az)
            if moon_x is not None:
                # Draw exclusion circle in km (approximate)
                excl_r = MOON_EXCLUSION_DEG / 90.0 * R_PROJECTION
                ax_map.plot(moon_x + excl_r*np.cos(th),
                            moon_y + excl_r*np.sin(th),
                            "y--", lw=1.5, alpha=0.7, label=f"Moon excl. {MOON_EXCLUSION_DEG}°")
                ax_map.plot(moon_x, moon_y, "y*", ms=14, mew=1.5, mec="orange",
                            zorder=6, label=f"Moon alt={m_alt:.1f}°")

        # Green — absolute minimum
        xa, ya = abs_positions[fi]
        va = get_value_at_position(grid,  xa, ya)
        sa = get_value_at_position(sgrid, xa, ya)
        gx = [abs_positions[j][0] for j in range(ts, fi)]
        gy = [abs_positions[j][1] for j in range(ts, fi)]
        if len(gx) > 1:
            ax_map.plot(gx, gy, "g--", alpha=0.35, lw=1.5)
        ax_map.plot(xa, ya, "go", ms=20, mew=2.5, mec="white",
                    label=f"Abs Min  ext={va:.3f} σ={sa:.3f}", zorder=10)

        # Blue — motion tracking
        xp, yp = pred_positions[fi]
        vp = get_value_at_position(grid,  xp, yp)
        sp = get_value_at_position(sgrid, xp, yp)
        bx = [pred_positions[j][0] for j in range(ts, fi)]
        by = [pred_positions[j][1] for j in range(ts, fi)]
        if len(bx) > 1:
            ax_map.plot(bx, by, "b--", alpha=0.35, lw=1.5)
        ax_map.plot(xp, yp, "bo", ms=20, mew=2.5, mec="white",
                    label=f"Motion   ext={vp:.3f} σ={sp:.3f}", zorder=10)

        # Motion arrow
        if fi > 0:
            dx_pix, dy_pix = motion_vectors[fi]
            dxk = dx_pix * BIN_SIZE_KM
            dyk = dy_pix * BIN_SIZE_KM
            mag = np.sqrt(dxk**2 + dyk**2)
            if mag > 300:
                ax_map.add_patch(FancyArrowPatch(
                    (xp, yp), (xp+dxk, yp+dyk),
                    arrowstyle="->", mutation_scale=28, lw=2.5,
                    color="cyan", alpha=0.9, zorder=15))
                ax_map.text(xp+dxk+400, yp+dyk+400, f"{mag:.0f} km",
                            fontsize=9, color="cyan", weight="bold",
                            bbox=dict(boxstyle="round", fc="black", alpha=0.6))

        # Red — scheduler
        xs, ys, alt_s, _ = sched_xy[fi]
        if xs is not None:
            vs = get_value_at_position(grid,  xs, ys)
            ss = get_value_at_position(sgrid, xs, ys)
            ax_map.plot(xs, ys, "ro", ms=20, mew=2.5, mec="white",
                        label=f"Scheduler ext={vs:.3f} σ={ss:.3f}", zorder=10)
        else:
            ax_map.plot([], [], "ro", ms=20, mew=2.5, mec="white", alpha=0.3,
                        label=f"Scheduler (below horizon, alt={alt_s:.1f}°)")

        def _fmt(v, ref):
            return f"{v:.3f} ({v-ref:+.3f})" if not np.isnan(v) else "N/A"

        txt = (f"EXTINCTION (sys, σ-masked)\n"
               f" Abs Min:  {va:.3f} mag  σ={sa:.3f}  [baseline]\n"
               f" Motion:   {_fmt(vp, va)}  σ={sp:.3f}\n"
               f" Sched:    {_fmt(vs if xs is not None else np.nan, va)}"
               f"  σ={ss:.3f}" if xs is not None else
               f"  Sched: below horizon")
        ax_map.text(0.02, 0.98, txt, transform=ax_map.transAxes,
                    fontsize=9.5, va="top", family="monospace", zorder=20,
                    bbox=dict(boxstyle="round", fc="white",
                              alpha=0.92, ec="black", lw=1.5))

        ax_map.set_xlabel("X (km)"); ax_map.set_ylabel("Y (km)")
        ax_map.set_aspect("equal"); ax_map.grid(alpha=0.2)
        ax_map.set_xlim(-16000, 16000); ax_map.set_ylim(-16000, 16000)
        ts_str = all_metas[fi]["time"].strftime("%Y-%m-%d %H:%M:%S UTC")
        ax_map.set_title(f"Cloud Coverage — {ts_str}\nFrame {fi+1}/{len(all_grids)}",
                         fontsize=12, weight="bold")
        ax_map.legend(loc="upper right", fontsize=8.5, framealpha=0.9)
        return []

    anim = animation.FuncAnimation(
        fig, update, frames=len(all_grids),
        interval=1000/fps, blit=False, repeat=False)
    print(f"Saving {output_file}  ({len(all_grids)} frames @ {fps} fps) ...")
    Writer = animation.writers["ffmpeg"]
    writer = Writer(fps=fps, bitrate=5000, codec="libx264",
                    extra_args=["-pix_fmt", "yuv420p"])
    anim.save(output_file, writer=writer)
    print(f"  Video saved: {output_file}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# ZP cross-validation
# ─────────────────────────────────────────────────────────────────────────────

def _safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def zps_value_at_radec(clouds, sigma_map, ra_deg, dec_deg, obstime):
    t   = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    if aa.alt.deg < MIN_ALT_DEG:
        return np.nan, np.nan

    npix  = hp.nside2npix(NSIDE_EXPECTED)
    pix   = np.arange(npix)
    th, ph = hp.pix2ang(NSIDE_EXPECTED, pix, nest=NEST)
    all_sky = SkyCoord(ra=np.degrees(ph)*u.deg,
                       dec=(90-np.degrees(th))*u.deg, frame="icrs")
    all_aa  = all_sky.transform_to(AltAz(obstime=t, location=loc))

    v_alt = np.radians(aa.alt.deg)
    v_az  = np.radians(aa.az.deg % 360)
    ar    = np.radians(all_aa.alt.deg)
    azr   = np.radians(all_aa.az.deg % 360)
    cos_sep = (np.sin(v_alt)*np.sin(ar)
               + np.cos(v_alt)*np.cos(ar)*np.cos(v_az - azr))

    valid = ~np.isnan(clouds) & (all_aa.alt.deg >= MIN_ALT_DEG)
    if not np.any(valid):
        return np.nan, np.nan
    best = np.nanargmax(np.where(valid, cos_sep, -np.inf))
    return float(clouds[best]), float(sigma_map[best])


def build_consdb_df(start_night, instrument=INSTRUMENT, consdb_url=CONSDB_URL,
                    h5_file=None):
    """
    Build ConsDB DataFrame from live API or from a pre-saved HDF5 file.
    Pass h5_file="visits_20261024_20260219.h5" to load from disk instead.
    """
    print("\n" + "=" * 70)
    print("BUILDING ConsDB DATAFRAME")
    print("=" * 70)

    if h5_file and os.path.exists(h5_file):
        print(f"  Loading ConsDB visits from {h5_file} …")
        df = pd.read_hdf(h5_file, key="visits")
        # Normalise column names (may vary between dumps)
        df.columns = df.columns.str.strip()
        print(f"  Loaded {len(df)} rows, {len(df.columns)} columns")
        # Ensure required derived columns exist
        if "obs_midpoint" not in df.columns:
            for sc, ec in [("obs_start", "obs_end"),
                           ("obs_start_mjd", "obs_end_mjd")]:
                if sc in df.columns and ec in df.columns:
                    t0 = pd.to_datetime(df[sc], utc=True, errors="coerce")
                    t1 = pd.to_datetime(df[ec], utc=True, errors="coerce")
                    df["obs_midpoint"] = t0 + (t1 - t0) / 2
                    break
        # Map column aliases
        for want, cands in {
            "ra":  ["s_ra",  "ra_v1",  "ra_ql",  "ra"],
            "dec": ["s_dec", "dec_v1", "dec_ql", "dec"],
        }.items():
            if want not in df.columns:
                for c in cands:
                    if c in df.columns:
                        df[want] = df[c]; break
        if "lambda_eff" not in df.columns and "band" in df.columns:
            df["lambda_eff"] = df["band"].map(EFFECTIVE_WAVELENGTHS)
        if "fwhm_observed" not in df.columns and "psf_sigma_median" in df.columns:
            df["fwhm_observed"] = FWHM_CONSTANT * pd.to_numeric(
                df["psf_sigma_median"], errors="coerce")
        if "fwhm_zenith" not in df.columns and "fwhm_observed" in df.columns \
                and "airmass" in df.columns:
            df["fwhm_zenith"] = (pd.to_numeric(df["fwhm_observed"], errors="coerce")
                                 / pd.to_numeric(df["airmass"], errors="coerce") ** ALPHA_SEEING)
        if "seeing" not in df.columns and "fwhm_zenith" in df.columns \
                and "lambda_eff" in df.columns:
            df["seeing"] = (pd.to_numeric(df["fwhm_zenith"], errors="coerce")
                            * (pd.to_numeric(df["lambda_eff"], errors="coerce") / 500.0)
                            ** BETA_SEEING)
        df["zp_consdb"] = pd.to_numeric(
            df.get("zero_point_median", pd.Series(np.nan, index=df.index)),
            errors="coerce")
        df = df[df["ra"].notna() & df["dec"].notna()].reset_index(drop=True)
        print(f"  Final: {len(df)} rows, zp_consdb non-null: {df['zp_consdb'].notna().sum()}")
        return df

    # ── Live API path ─────────────────────────────────────────────────────────
    client = ConsDbClient(consdb_url)
    t0     = pd.to_datetime(start_night)
    nights = np.arange(int(t0.strftime("%Y%m%d")),
                       int(pd.Timestamp.now().strftime("%Y%m%d")))
    dfs    = []
    zp_col = None

    for night in nights:
        try:
            v1 = client.query(
                f"SELECT * FROM cdb_{instrument}.visit1 "
                f"WHERE day_obs = {night}"
            ).to_pandas()
            ql = client.query(
                f"SELECT * FROM cdb_{instrument}.visit1_quicklook "
                f"WHERE day_obs = {night}"
            ).to_pandas()
            if ql.empty or v1.empty:
                continue
            merged = pd.merge(ql, v1, on="visit_id", how="inner",
                              suffixes=("_ql", "_v1"))
            merged = merged[merged["airmass"].notnull()
                            & (merged["airmass"] != 0.0)].copy()
            if merged.empty:
                continue
            if zp_col is None:
                candidates = [c for c in merged.columns
                              if any(k in c.lower() for k in
                                     ["zeropoint","zero_point","zp","fluxmag0",
                                      "flux_mag0","photometric","calib"])]
                for name in ["zeroPoint","zero_point","zp_mean","zeropoint",
                             "flux_mag0","fluxMag0"]:
                    if name in merged.columns:
                        zp_col = name; break
                if zp_col is None and candidates:
                    zp_col = candidates[0]
                print(f"  ZP column: '{zp_col}'")

            actual_zp = None
            if zp_col:
                for sfx in ("", "_ql", "_v1"):
                    if zp_col + sfx in merged.columns:
                        actual_zp = zp_col + sfx; break
            merged["zp_consdb"] = (pd.to_numeric(merged[actual_zp], errors="coerce")
                                   if actual_zp else np.nan)
            merged["lambda_eff"]    = merged["band"].map(EFFECTIVE_WAVELENGTHS)
            merged["fwhm_observed"] = FWHM_CONSTANT * merged["psf_sigma_median"]
            merged["fwhm_zenith"]   = merged["fwhm_observed"] / (merged["airmass"] ** ALPHA_SEEING)
            merged["seeing"]        = (merged["fwhm_zenith"]
                                       * (merged["lambda_eff"] / 500.0) ** BETA_SEEING)
            merged["obs_start"]    = pd.to_datetime(merged["obs_start"],
                                                     format="ISO8601", utc=True,
                                                     errors="coerce")
            merged["obs_end"]      = pd.to_datetime(merged["obs_end"],
                                                     format="ISO8601", utc=True,
                                                     errors="coerce")
            merged["obs_midpoint"] = (merged["obs_start"]
                                      + (merged["obs_end"] - merged["obs_start"]) / 2)
            for want, cands in {
                "ra":  ["s_ra",  "ra_v1",  "ra_ql",  "ra"],
                "dec": ["s_dec", "dec_v1", "dec_ql", "dec"],
            }.items():
                if want not in merged.columns:
                    for c in cands:
                        if c in merged.columns:
                            merged[want] = merged[c]; break
            dfs.append(merged)
            print(f"  night {night}: {len(merged)} visits")
        except Exception as e:
            print(f"  night {night}: skipped ({e})")

    if not dfs:
        print("No ConsDB data found.")
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df = df[df["ra"].notna() & df["dec"].notna()].reset_index(drop=True)
    df["zp_consdb"] = pd.to_numeric(df["zp_consdb"], errors="coerce")
    print(f"ConsDB total: {len(df)} visits, "
          f"zp_consdb non-null: {df['zp_consdb'].notna().sum()}")
    return df


def cross_validate_zp(dream_zps_df, consdb_df,
                       n_samples=N_ZP_SAMPLES, seed=ZP_SEED):
    print("\n" + "=" * 70)
    print("ZP CROSS-VALIDATION (DREAM .zps vs ConsDB)")
    print("=" * 70)
    if consdb_df.empty:
        print("No ConsDB data — skipping cross-validation.")
        return pd.DataFrame()

    d_min, d_max = dream_zps_df[TIME_COL].min(), dream_zps_df[TIME_COL].max()
    c_min, c_max = consdb_df["obs_midpoint"].min(), consdb_df["obs_midpoint"].max()
    ov_start = max(d_min, c_min)
    ov_end   = min(d_max, c_max)
    if ov_start >= ov_end:
        print("No temporal overlap between .zps frames and ConsDB.")
        return pd.DataFrame()

    dream_ov  = dream_zps_df[(dream_zps_df[TIME_COL] >= ov_start)
                              & (dream_zps_df[TIME_COL] <= ov_end)].reset_index(drop=True)
    consdb_ov = consdb_df[(consdb_df["obs_midpoint"] >= ov_start)
                           & (consdb_df["obs_midpoint"] <= ov_end)].reset_index(drop=True)
    print(f"DREAM .zps in overlap: {len(dream_ov)},  ConsDB: {len(consdb_ov)}")
    if dream_ov.empty or consdb_ov.empty:
        return pd.DataFrame()

    rng    = np.random.default_rng(seed)
    idx    = rng.choice(len(dream_ov), size=min(n_samples, len(dream_ov)), replace=False)
    sample = dream_ov.iloc[sorted(idx)].reset_index(drop=True)

    cadence = max(
        (dream_ov[TIME_COL].max() - dream_ov[TIME_COL].min()).total_seconds()
        / max(len(dream_ov), 1), 300.0)
    window = pd.Timedelta(seconds=cadence / 2)
    print(f"Match window: ±{int(window.total_seconds())}s\n")

    rows = []
    for _, frame in sample.iterrows():
        t_frame   = frame[TIME_COL]
        url       = transform_url(frame[URL_COL])
        dbnd      = frame.get("dream_band")
        try:
            clouds, sigma_map = fetch_zps_map(url)
        except Exception as e:
            print(f"   load failed: {e}")
            continue

        nearby = consdb_ov[
            (consdb_ov["obs_midpoint"] >= t_frame - window)
            & (consdb_ov["obs_midpoint"] <= t_frame + window)
        ].copy()
        lsst_band = FILENAME_BAND_MAP.get(dbnd)
        if lsst_band:
            nearby = nearby[nearby["band"] == lsst_band]
        print(f"  {t_frame.strftime('%H:%M:%S')} dream_band={dbnd}  "
              f"consdb matches={len(nearby)}")

        for _, visit in nearby.iterrows():
            ra  = _safe_float(visit.get("ra"))
            dec = _safe_float(visit.get("dec"))
            if np.isnan(ra) or np.isnan(dec):
                continue
            zp_d, sig_d = zps_value_at_radec(clouds, sigma_map, ra, dec, t_frame)
            zp_c        = _safe_float(visit.get("zp_consdb", np.nan))
            airmass     = _safe_float(visit.get("airmass",   np.nan))
            seeing      = _safe_float(visit.get("seeing",    np.nan))
            band        = str(visit.get("band", "?"))
            # (c) attach quicklook quality columns to the ZP row
            eff_time_med = _safe_float(visit.get("eff_time_median",                    np.nan))
            eft_zp_scale = _safe_float(visit.get("eff_time_zero_point_scale_median",   np.nan))
            psf_sig_med  = _safe_float(visit.get("psf_sigma_median",                   np.nan))
            sz500        = _safe_float(visit.get("seeing_zenith_500nm_median",          np.nan))
            zp_med       = _safe_float(visit.get("zero_point_median",                  np.nan))
            sky_bg       = _safe_float(visit.get("sky_bg_median",                      np.nan))
            delta = (zp_d - zp_c
                     if not (np.isnan(zp_d) or np.isnan(zp_c)) else np.nan)
            rows.append(dict(
                frame_time=t_frame, visit_id=visit.get("visit_id"),
                band=band, dream_band=dbnd, ra=ra, dec=dec,
                airmass=airmass, seeing=seeing,
                zp_offset_dream=zp_d, sig_dream=sig_d,
                zp_consdb=zp_c, delta=delta,
                eff_time_median=eff_time_med,
                eff_time_zero_point_scale_median=eft_zp_scale,
                psf_sigma_median=psf_sig_med,
                seeing_zenith_500nm_median=sz500,
                zero_point_median=zp_med,
                sky_bg_median=sky_bg,
            ))
    return pd.DataFrame(rows)


def plot_zp_comparison(df, output="dream_zps_vs_consdb.png"):
    print("\n" + "=" * 70)
    print("CREATING ZP COMPARISON PLOT")
    print("=" * 70)
    if df.empty:
        print("No data to plot.")
        return

    for col in ["zp_offset_dream", "sig_dream", "zp_consdb", "delta",
                "airmass", "seeing", "eff_time_zero_point_scale_median",
                "seeing_zenith_500nm_median"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    valid = df.dropna(subset=["zp_offset_dream"])
    if valid.empty:
        print("No valid DREAM ZP offsets.")
        return

    bands  = sorted(valid["band"].dropna().unique())
    cmap   = plt.cm.get_cmap("tab10", max(len(bands), 1))
    bc     = {b: cmap(i) for i, b in enumerate(bands)}
    has_cdb = valid["zp_consdb"].notna().any()
    ncols   = 4 if has_cdb else 3

    fig, axes = plt.subplots(1, ncols, figsize=(7*ncols, 5))
    fig.suptitle("DREAM .zps ZP Offset vs ConsDB", fontsize=13, weight="bold")

    # Panel 1: ZP offset vs airmass
    ax = axes[0]
    for b in bands:
        s = valid[valid["band"] == b].dropna(subset=["airmass"])
        ax.errorbar(s["airmass"], s["zp_offset_dream"],
                    yerr=s["sig_dream"], fmt="o", ms=6,
                    color=bc[b], label=b, alpha=0.8, capsize=3)
    ax.axhline(0, color="k", ls="--", lw=1.5, label="zero offset")
    ax.set_xlabel("Airmass"); ax.set_ylabel("DREAM ZP offset (mag)")
    ax.set_title("ZP offset vs Airmass\n(0 = photometric)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 2: ZP offset timeline
    ax = axes[1]
    for b in bands:
        s = valid[valid["band"] == b]
        ax.scatter(s["frame_time"], s["zp_offset_dream"],
                   color=bc[b], s=40, alpha=0.9, label=b)
    ax.axhline(0, color="k", ls="--", lw=1.5)
    ax.set_ylabel("DREAM ZP offset (mag)")
    ax.set_title("ZP offset over night")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 3: ZP offset vs seeing_zenith_500nm
    ax = axes[2]
    see_col = "seeing_zenith_500nm_median" if "seeing_zenith_500nm_median" in valid.columns \
        else "seeing"
    for b in bands:
        s = valid[valid["band"] == b].dropna(subset=[see_col])
        if s.empty:
            continue
        ax.scatter(s[see_col], s["zp_offset_dream"],
                   color=bc[b], s=40, alpha=0.9, label=b)
    ax.axhline(0, color="k", ls="--", lw=1.5)
    ax.set_xlabel("Seeing zenith 500nm (arcsec)"); ax.set_ylabel("DREAM ZP offset (mag)")
    ax.set_title("ZP offset vs Seeing")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 4: DREAM vs ConsDB scatter (if data available)
    if has_cdb:
        valid2 = valid.dropna(subset=["zp_consdb"])
        ax = axes[3]
        for b in bands:
            s = valid2[valid2["band"] == b]
            if s.empty:
                continue
            ax.errorbar(s["zp_consdb"], s["zp_offset_dream"],
                        yerr=s["sig_dream"], fmt="o", ms=6,
                        color=bc[b], label=b, alpha=0.8, capsize=3)
        if len(valid2) >= 2:
            sl, ic, r, *_ = stats.linregress(
                valid2["zp_consdb"].astype(float),
                valid2["zp_offset_dream"].astype(float))
            xs = np.linspace(valid2["zp_consdb"].min(),
                             valid2["zp_consdb"].max(), 100)
            ax.plot(xs, sl*xs+ic, "r-", lw=1.5,
                    label=f"slope={sl:.2f}  r={r:.2f}")
        ax.set_xlabel("ConsDB ZP"); ax.set_ylabel("DREAM ZP offset (mag)")
        ax.set_title("DREAM offset vs ConsDB ZP")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(output, dpi=150, bbox_inches="tight")
    print(f"  ZP comparison plot saved: {output}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# MAIN per-night runner
# ─────────────────────────────────────────────────────────────────────────────

def run_night(night_key, df_sys_night, df_zps_night,
              scheduler_obs, consdb_df,
              output_dir=".", fps=10, max_frames=None,
              all_tensors_accumulator=None):
    night_str  = str(night_key)
    tag        = night_str.replace("-", "")
    plot_file   = os.path.join(output_dir, f"metrics_{tag}.png")
    video_file  = os.path.join(output_dir, f"cloud_tracking_{tag}.mp4")
    zp_file     = os.path.join(output_dir, f"dream_zps_vs_consdb_{tag}.png")
    tensor_file = os.path.join(output_dir, f"pointing_tensor_{tag}.npz")  # (d)

    print("\n" + "█" * 70)
    print(f"  NIGHT: {night_str}  ({len(df_sys_night)} sys frames)")
    print("█" * 70)

    if len(df_sys_night) < 5:
        print("  Too few frames — skipping.")
        return None

    print(f"\n  [Patchiness probe — {PATCHINESS_SAMPLE_N} frames]")
    patchy, mean_frac, mean_var, reason, _ = quick_patchiness_check(df_sys_night)
    print(f"    valid_frac={mean_frac:.3f}  spatial_var={mean_var:.4f} mag²")
    print(f"    → {reason}")
    if not patchy:
        print("  Skipping night (not patchy).")
        return None

    print("  Patchy clouds confirmed — loading all frames …")

    all_grids  = []
    all_sigma  = []
    all_metas  = []
    x_edges = y_edges = None

    rows = (df_sys_night.iloc[:max_frames] if max_frames else df_sys_night).iterrows()
    for _, row in rows:
        url = transform_url(row[URL_COL])
        try:
            clouds, sigma = fetch_sys_map(url)
            meta = {"url": url, "time": row[TIME_COL].to_pydatetime()}
            ext_g, sig_g, _, _, xe, ye = process_to_grid(clouds, sigma, meta["time"])
            all_grids.append(ext_g)
            all_sigma.append(sig_g)
            all_metas.append(meta)
            if x_edges is None:
                x_edges, y_edges = xe, ye
            if len(all_grids) % 100 == 0:
                print(f"    Loaded {len(all_grids)} frames …")
        except Exception as e:
            print(f"    Frame load failed: {e}")

    print(f"  Loaded {len(all_grids)} frames total")
    if len(all_grids) < 2:
        print("  Too few frames loaded — skipping.")
        return None

    patchy2, mf2, mv2, reason2 = compute_patchiness(all_grids)
    print(f"  Full-set patchiness: valid_frac={mf2:.3f}  var={mv2:.4f}  → {reason2}")
    if not patchy2:
        print("  Full-set check failed — skipping.")
        return None

    matched_sched = match_scheduler_to_frames(
        scheduler_obs, [m["time"] for m in all_metas])

    print(f"\n  [Cloud motion tracking — {len(all_grids)} frames]")
    # (a)+(b) moon positions computed inside compute_all_positions
    abs_pos, pred_pos, motion_v, moon_pos = compute_all_positions(
        all_grids, all_metas)

    metrics = compute_metrics(
        all_grids, all_sigma, all_metas,
        abs_pos, pred_pos, moon_pos, matched_sched)
    print_statistics(metrics)

    # (c) Atmospheric + seeing correction
    metrics = apply_atm_seeing_correction(metrics, all_metas, consdb_df)
    print_corrected_statistics(metrics)

    create_metrics_plot(metrics, output_file=plot_file)

    create_video(all_grids, all_sigma, all_metas,
                 abs_pos, pred_pos, motion_v, moon_pos,
                 matched_sched, x_edges, y_edges,
                 output_file=video_file, fps=fps)

    # ── (d) Save pointing tensor ─────────────────────────────────────────────
    tensor = build_pointing_tensor(metrics, all_metas)
    save_pointing_tensor(tensor, tensor_file, night_key,
                         all_tensors_accumulator=all_tensors_accumulator)

    # ── ZP cross-validation ──────────────────────────────────────────────────
    if not df_zps_night.empty and not consdb_df.empty:
        print(f"\n  [ZP cross-validation — {len(df_zps_night)} zps frames]")
        comparison = cross_validate_zp(df_zps_night, consdb_df)
        if not comparison.empty:
            for col in ["zp_offset_dream", "sig_dream", "zp_consdb", "delta", "airmass"]:
                if col in comparison.columns:
                    comparison[col] = pd.to_numeric(comparison[col], errors="coerce")
            valid_zp = comparison.dropna(subset=["zp_offset_dream"])
            print(f"  Mean DREAM ZP offset : {valid_zp['zp_offset_dream'].mean():.4f} mag")
            print(f"  Std  DREAM ZP offset : {valid_zp['zp_offset_dream'].std():.4f} mag")
            valid2 = comparison.dropna(subset=["zp_offset_dream", "zp_consdb"])
            if len(valid2) >= 2:
                r, p = stats.pearsonr(
                    valid2["zp_offset_dream"].astype(float),
                    valid2["zp_consdb"].astype(float))
                print(f"  Pearson r : {r:.3f}  (p={p:.4f})")
            plot_zp_comparison(comparison, output=zp_file)
        else:
            print("  No matched ZP pairs found.")
    else:
        print("  Skipping ZP validation (no zps frames or no ConsDB data).")

    print(f"\n  ✓ Night {night_str} outputs:")
    print(f"      {plot_file}")
    print(f"      {video_file}")
    print(f"      {zp_file}")
    print(f"      {tensor_file}")
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main(
    csv_file     = "feb5_data.csv",
    db_file      = "baseline_v5.1.0_10yrs.db",
    consdb_start = "2025-05-01",
    consdb_h5    = None,          # e.g. "visits_20261024_20260219.h5"
    output_dir   = ".",
    fps          = 10,
    max_frames   = None,
):
    """
    Iterate over every distinct night present in the DREAM CSV.
    For each night:
      - Quick patchiness probe (only ~10 frames fetched)
      - If not patchy → print reason and move on
      - If patchy     → full analysis (tracking, metrics, video, ZP, tensor)
    Scheduler observations and ConsDB data are loaded once and reused.

    New arguments:
      consdb_h5 : path to pre-saved HDF5 visits file; if provided, loaded
                  instead of querying the live ConsDB API.
    """
    print("\n" + "=" * 70)
    print("DREAM CLOUD ANALYSIS PIPELINE  —  ALL NIGHTS")
    print("=" * 70)
    print(f"  CSV          : {csv_file}")
    print(f"  Scheduler DB : {db_file}")
    print(f"  Output dir   : {output_dir}")
    print(f"  Moon excl.   : {MOON_EXCLUSION_DEG}°")
    print(f"  Zenith wt.   : {W_ZENITH}  Slew wt.: {W_SLEW}  Cloud wt.: {W_CLOUD}")
    os.makedirs(output_dir, exist_ok=True)

    print("\nIndexing CSV …")
    all_sys = load_all_sys_frames(csv_file)
    all_zps = load_all_zps_frames(csv_file)

    all_nights = sorted(all_sys["night_key"].unique())
    print(f"Found {len(all_nights)} distinct nights in CSV:")
    for nk in all_nights:
        n = (all_sys["night_key"] == nk).sum()
        print(f"  {nk}  ({n} sys frames)")

    scheduler_obs, _ = load_scheduler_observations(db_file)

    print("\nLoading ConsDB …")
    consdb_df = build_consdb_df(consdb_start, h5_file=consdb_h5)

    summary              = []
    all_tensors          = []   # (d) accumulator for cross-night tensor
    nights_checked       = 0    # total nights with enough frames to probe
    nights_patchy        = 0    # nights that passed patchiness and ran fully

    for night_key in all_nights:
        df_sys_night = get_night_df(all_sys, night_key)
        df_zps_night = get_night_df(all_zps, night_key)

        if len(df_sys_night) >= 5:
            nights_checked += 1

        metrics = run_night(
            night_key, df_sys_night, df_zps_night,
            scheduler_obs, consdb_df,
            output_dir=output_dir, fps=fps, max_frames=max_frames,
            all_tensors_accumulator=all_tensors)

        if metrics is not None:
            nights_patchy += 1
            row = {"night": str(night_key)}
            for s in ("absolute", "motion", "scheduler"):
                ph   = np.array(metrics[s]["photons"])
                ph_c = np.array(metrics[s].get("photons_atm_corrected", [np.nan]))
                row[f"total_photons_{s}"]          = float(np.sum(ph))   if len(ph)   else 0.0
                row[f"total_photons_corr_{s}"]     = float(np.nansum(ph_c)) if len(ph_c) else 0.0
                row[f"n_slots_{s}"]                = len(ph)
                row[f"mean_seeing_{s}"]            = float(np.nanmean(
                    metrics[s].get("seeing_fwhm", [np.nan])))
                row[f"mean_eff_time_scale_{s}"]    = float(np.nanmean(
                    metrics[s].get("eff_time_scale", [np.nan])))
            summary.append(row)

    # (d) All-nights combined tensor
    all_tensor_file = os.path.join(output_dir, "pointing_tensor_all_nights.npz")
    save_all_nights_tensor(all_tensors, all_tensor_file)

    print("\n" + "=" * 70)
    print("ALL-NIGHTS SUMMARY")
    print("=" * 70)

    # Patchiness overview — always printed regardless of whether any ran
    n_total_csv = len(all_nights)
    n_skipped_frames = n_total_csv - nights_checked
    n_skipped_uniform = nights_checked - nights_patchy
    print(f"\n{'PATCHINESS OVERVIEW':}")
    print(f"  Total nights in CSV          : {n_total_csv}")
    print(f"  Skipped (< 5 frames)         : {n_skipped_frames}")
    print(f"  Probed for patchiness        : {nights_checked}")
    print(f"  Skipped (not patchy / clear) : {n_skipped_uniform}")
    print(f"  Patchy — full analysis run   : {nights_patchy}  "
          f"({100*nights_patchy/max(nights_checked,1):.1f}% of probed nights)")

    if summary:
        df_sum = pd.DataFrame(summary)
        print(df_sum.to_string(index=False))
        sum_file = os.path.join(output_dir, "all_nights_summary.csv")
        df_sum.to_csv(sum_file, index=False)
        print(f"\nSummary saved → {sum_file}")

        # Patchiness log — one row per night in CSV, patchy flag included
        patch_rows = []
        for nk in all_nights:
            n_frames = int((all_sys["night_key"] == nk).sum())
            was_patchy = str(nk) in df_sum["night"].values
            patch_rows.append({"night": str(nk),
                                "n_sys_frames": n_frames,
                                "enough_frames": n_frames >= 5,
                                "patchy": was_patchy})
        patch_df = pd.DataFrame(patch_rows)
        patch_file = os.path.join(output_dir, "patchiness_log.csv")
        patch_df.to_csv(patch_file, index=False)
        print(f"Patchiness log saved → {patch_file}")
        print(f"\n{patch_df[['night','n_sys_frames','patchy']].to_string(index=False)}")
    else:
        print("No patchy nights found in this CSV.")

    print("\nDone.")
    return summary


if __name__ == "__main__":
    main()


DREAM CLOUD ANALYSIS PIPELINE  —  ALL NIGHTS
  CSV          : feb5_data.csv
  Scheduler DB : baseline_v5.1.0_10yrs.db
  Output dir   : .
  Moon excl.   : 30.0°
  Zenith wt.   : 0.25  Slew wt.: 0.25  Cloud wt.: 0.5

Indexing CSV …
Found 189 distinct nights in CSV:
  2025-06-25  (432 sys frames)
  2025-06-26  (1536 sys frames)
  2025-06-27  (1248 sys frames)
  2025-06-28  (1559 sys frames)
  2025-06-29  (1583 sys frames)
  2025-06-30  (1396 sys frames)
  2025-07-01  (1377 sys frames)
  2025-07-02  (1376 sys frames)
  2025-07-03  (1375 sys frames)
  2025-07-04  (1375 sys frames)
  2025-07-05  (495 sys frames)
  2025-07-06  (1371 sys frames)
  2025-07-07  (1371 sys frames)
  2025-07-08  (942 sys frames)
  2025-07-09  (1152 sys frames)
  2025-07-10  (1369 sys frames)
  2025-07-11  (1367 sys frames)
  2025-07-12  (1375 sys frames)
  2025-07-13  (1060 sys frames)
  2025-07-14  (1304 sys frames)
  2025-07-15  (1361 sys frames)
  2025-07-16  (1360 sys frames)
  2025-07-17  (1359 sys frames)
  

In [7]:
#!/usr/bin/env python3
"""
plot_sky_maps.py
================
Load one or more pointing_tensor_*.npz files produced by the DREAM pipeline
and render sky-map visualisations.

Tensor column layout (float32, 8 cols):
  0  frame_idx
  1  unix_time
  2  alt_deg
  3  az_deg
  4  x_km
  5  y_km
  6  extinction  (mag)
  7  strategy_id  (0=absolute, 1=motion, 2=scheduler)

Usage
-----
  # Single night
  python plot_sky_maps.py --npz pointing_tensor_20250627.npz

  # Every per-night file in a directory (one plot set each)
  python plot_sky_maps.py --npz_dir ./outputs

  # Merge all nights into one combined set
  python plot_sky_maps.py --npz_dir ./outputs --combined

  # Cross-night comparison grid
  python plot_sky_maps.py --npz_dir ./outputs --cross_night

Output files (written next to each .npz, or to --out_dir):
  sky_map_<tag>_density_xy.png   pointing density (alt/az + flat-sky km)
  sky_map_<tag>_extinction.png   median extinction heat-map in alt/az
  sky_map_<tag>_time_seq.png     pointing track coloured by time
  sky_map_<tag>_rose.png         polar azimuth / zenith-angle rose
  sky_map_<tag>_ext_hist.png     extinction distribution histogram
  sky_map_cross_night.png        cross-night stacked comparison (optional)
"""

import argparse
import glob
import os
import sys

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from scipy.stats import binned_statistic_2d

# ─────────────────────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────────────────────
STRATEGY_NAMES  = {0: "Absolute Min", 1: "Motion Tracking", 2: "Scheduler"}
STRATEGY_COLORS = {0: "#2ca02c", 1: "#1f77b4", 2: "#d62728"}

MIN_ALT_DEG = 15.0
MAX_ALT_DEG = 90.0

ALT_BINS = 30        # 15–90° altitude
AZ_BINS  = 72        # 0–360° azimuth  (5° each)
XY_BINS  = 40        # –15 000 to +15 000 km flat-sky


# ─────────────────────────────────────────────────────────────────────────────
# I/O
# ─────────────────────────────────────────────────────────────────────────────

def load_tensor(npz_path: str) -> np.ndarray:
    d = np.load(npz_path, allow_pickle=True)
    t = d["pointing_tensor"].astype(float)
    if t.ndim != 2 or t.shape[1] != 8:
        raise ValueError(f"{npz_path}: expected (N,8) tensor, got {t.shape}")
    print(f"  {os.path.basename(npz_path)}: {t.shape[0]:,} rows")
    return t


def load_all_tensors(npz_paths: list) -> np.ndarray:
    parts = []
    for p in sorted(npz_paths):
        try:
            parts.append(load_tensor(p))
        except Exception as e:
            print(f"  WARN: skipping {p}: {e}")
    if not parts:
        raise RuntimeError("No valid tensor files loaded.")
    return np.concatenate(parts, axis=0)


def tag_from_path(p: str) -> str:
    return os.path.splitext(os.path.basename(p))[0]


# ─────────────────────────────────────────────────────────────────────────────
# Binning helpers
# ─────────────────────────────────────────────────────────────────────────────

def _az_alt_edges():
    return (np.linspace(0, 360, AZ_BINS + 1),
            np.linspace(MIN_ALT_DEG, MAX_ALT_DEG, ALT_BINS + 1))


def _xy_edges():
    e = np.linspace(-15_000, 15_000, XY_BINS + 1)
    return e, e


def _count2d(x, y, xe, ye):
    H, _, _ = np.histogram2d(x, y, bins=[xe, ye])
    H[H == 0] = np.nan
    return H.T


def _median2d(x, y, vals, xe, ye):
    res = binned_statistic_2d(x, y, vals, statistic="median",
                              bins=[xe, ye])
    return res.statistic.T


def _zenith_rings(ax, alt_lo, alt_hi):
    """Overlay dashed ZA reference lines on an alt/az pcolormesh."""
    for za in [30, 45, 60, 75]:
        alt = 90 - za
        if alt_lo <= alt <= alt_hi:
            ax.axhline(alt, color="white", lw=0.7, ls="--", alpha=0.45)
            ax.text(363, alt, f"ZA {za}°", fontsize=7, color="white",
                    va="center", alpha=0.65, clip_on=False)


# ─────────────────────────────────────────────────────────────────────────────
# Panel renderers
# Each accepts an array of 3 matplotlib Axes (one per strategy).
# ─────────────────────────────────────────────────────────────────────────────

def _render_density(tensor, axes3):
    aze, alte = _az_alt_edges()
    Hs, gmax = {}, 1.0
    for sid in STRATEGY_NAMES:
        sub = tensor[tensor[:, 7] == sid]
        if len(sub):
            H = _count2d(sub[:, 3], sub[:, 2], aze, alte)
            Hs[sid] = H
            v = np.nanpercentile(H[~np.isnan(H)], 98)
            gmax = max(gmax, v)

    for j, (sid, sname) in enumerate(STRATEGY_NAMES.items()):
        ax = axes3[j]
        if sid not in Hs:
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes)
            ax.set_title(sname); continue
        im = ax.pcolormesh(aze, alte, Hs[sid],
                           cmap="plasma", vmin=0, vmax=gmax)
        plt.colorbar(im, ax=ax, label="# pointings", shrink=0.82)
        _zenith_rings(ax, MIN_ALT_DEG, MAX_ALT_DEG)
        ax.set_xlim(0, 360); ax.set_ylim(MIN_ALT_DEG, MAX_ALT_DEG)
        ax.set_xlabel("Azimuth (°)"); ax.set_ylabel("Altitude (°)")
        ax.set_title(sname, weight="bold", color=STRATEGY_COLORS[sid])


def _render_extinction(tensor, axes3):
    aze, alte = _az_alt_edges()
    for j, (sid, sname) in enumerate(STRATEGY_NAMES.items()):
        ax  = axes3[j]
        sub = tensor[tensor[:, 7] == sid]
        if len(sub) == 0:
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes); continue
        H  = _median2d(sub[:, 3], sub[:, 2], sub[:, 6], aze, alte)
        im = ax.pcolormesh(aze, alte, H, cmap="RdYlGn_r", vmin=-0.1, vmax=1.5)
        plt.colorbar(im, ax=ax, label="Extinction (mag)", shrink=0.82)
        _zenith_rings(ax, MIN_ALT_DEG, MAX_ALT_DEG)
        ax.set_xlim(0, 360); ax.set_ylim(MIN_ALT_DEG, MAX_ALT_DEG)
        ax.set_xlabel("Azimuth (°)"); ax.set_ylabel("Altitude (°)")
        ax.set_title(sname, weight="bold", color=STRATEGY_COLORS[sid])


def _render_xy(tensor, axes3):
    xe, ye = _xy_edges()
    th = np.linspace(0, 2*np.pi, 300)
    for j, (sid, sname) in enumerate(STRATEGY_NAMES.items()):
        ax  = axes3[j]
        sub = tensor[tensor[:, 7] == sid]
        if len(sub) == 0:
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes); continue
        H  = _count2d(sub[:, 4], sub[:, 5], xe, ye)
        vm = np.nanpercentile(H[~np.isnan(H)], 98) if np.any(~np.isnan(H)) else 1
        im = ax.pcolormesh(xe, ye, H, cmap="magma", vmin=0, vmax=vm)
        plt.colorbar(im, ax=ax, label="# pointings", shrink=0.82)
        ax.plot(15000*np.cos(th), 15000*np.sin(th), "w--", lw=0.8, alpha=0.4)
        ax.plot(0, 0, "r+", ms=10, mew=2, label="Zenith")
        ax.set_aspect("equal")
        ax.set_xlabel("X (km)"); ax.set_ylabel("Y (km)")
        ax.set_title(sname, weight="bold", color=STRATEGY_COLORS[sid])
        ax.legend(fontsize=8, loc="upper right")


def _render_time_seq(tensor, axes3, max_pts=6000):
    for j, (sid, sname) in enumerate(STRATEGY_NAMES.items()):
        ax  = axes3[j]
        sub = tensor[tensor[:, 7] == sid]
        if len(sub) == 0:
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes); continue
        sub = sub[np.argsort(sub[:, 1])]
        if len(sub) > max_pts:
            idx = np.linspace(0, len(sub)-1, max_pts, dtype=int)
            sub = sub[idx]
        t_n = (sub[:, 1] - sub[:, 1].min()) / max(sub[:, 1].ptp(), 1)
        sc  = ax.scatter(sub[:, 3], sub[:, 2],
                         c=t_n, cmap="cool", s=7, alpha=0.65, vmin=0, vmax=1)
        plt.colorbar(sc, ax=ax, label="Relative time (0=start)", shrink=0.82)
        _zenith_rings(ax, MIN_ALT_DEG, MAX_ALT_DEG)
        ax.set_xlim(0, 360); ax.set_ylim(MIN_ALT_DEG, MAX_ALT_DEG)
        ax.set_xlabel("Azimuth (°)"); ax.set_ylabel("Altitude (°)")
        ax.set_title(sname, weight="bold", color=STRATEGY_COLORS[sid])


def _render_rose(fig, axes3, tensor):
    aze_rad = np.linspace(0, 2*np.pi, AZ_BINS + 1)
    az_mid  = (aze_rad[:-1] + aze_rad[1:]) / 2
    width   = aze_rad[1] - aze_rad[0]
    norm    = mcolors.Normalize(vmin=0, vmax=75)

    for j, (sid, sname) in enumerate(STRATEGY_NAMES.items()):
        ax  = axes3[j]
        sub = tensor[tensor[:, 7] == sid]
        if len(sub) == 0:
            ax.set_title(sname); continue
        az_r      = np.radians(sub[:, 3])
        za        = 90.0 - sub[:, 2]
        counts, _ = np.histogram(az_r, bins=aze_rad)
        za_med    = binned_statistic_2d(
            az_r, np.zeros_like(az_r), za,
            statistic="median", bins=[aze_rad, [0, 1]]).statistic[:, 0]
        ax.bar(az_mid, counts, width=width,
               color=cm.viridis_r(norm(za_med)), alpha=0.85, align="center")
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)
        ax.set_title(sname, weight="bold",
                     color=STRATEGY_COLORS[sid], pad=14)

    sm = plt.cm.ScalarMappable(cmap="viridis_r", norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=list(axes3), label="Median ZA (°)",
                 shrink=0.55, pad=0.06)


def _render_ext_hist(tensor, ax):
    bins = np.linspace(-0.2, 2.5, 60)
    for sid, sname in STRATEGY_NAMES.items():
        ext = tensor[tensor[:, 7] == sid, 6]
        ext = ext[np.isfinite(ext)]
        if len(ext) == 0:
            continue
        ax.hist(ext, bins=bins, alpha=0.55, color=STRATEGY_COLORS[sid],
                label=f"{sname}  n={len(ext):,}", edgecolor="none")
    ax.axvline(0, color="k", ls="--", lw=1.2, label="Photometric (0 mag)")
    ax.set_xlabel("Cloud Extinction (mag)"); ax.set_ylabel("Count")
    ax.set_title("Extinction Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)


# ─────────────────────────────────────────────────────────────────────────────
# Figure builders — each returns a closed-ready plt.Figure
# ─────────────────────────────────────────────────────────────────────────────

def make_density_xy_figure(tensor, title) -> plt.Figure:
    """2 rows × 3 cols: alt/az density + flat-sky km coverage."""
    fig, axes = plt.subplots(2, 3, figsize=(21, 10),
                             gridspec_kw={"hspace": 0.42, "wspace": 0.30})
    fig.suptitle(f"Pointing Density  {title}", fontsize=13, weight="bold")
    _render_density(tensor, axes[0])
    _render_xy(     tensor, axes[1])
    for ax in axes.flat:
        ax.tick_params(labelsize=8)
    fig.text(0.5, 0.54, "— Alt / Az Hit Map —", ha="center",
             fontsize=9, style="italic", color="grey")
    fig.text(0.5, 0.01, "— Flat-Sky (km) Coverage —", ha="center",
             fontsize=9, style="italic", color="grey")
    return fig


def make_extinction_figure(tensor, title) -> plt.Figure:
    """1 row × 3 cols: median extinction alt/az."""
    fig, axes = plt.subplots(1, 3, figsize=(21, 5), sharey=True,
                             gridspec_kw={"wspace": 0.30})
    fig.suptitle(f"Median Cloud Extinction (mag)  {title}",
                 fontsize=13, weight="bold")
    _render_extinction(tensor, axes)
    return fig


def make_time_seq_figure(tensor, title) -> plt.Figure:
    """1 row × 3 cols: pointing track coloured by time."""
    fig, axes = plt.subplots(1, 3, figsize=(21, 5), sharey=True,
                             gridspec_kw={"wspace": 0.30})
    fig.suptitle(f"Pointing Track  {title}", fontsize=13, weight="bold")
    _render_time_seq(tensor, axes)
    return fig


def make_rose_figure(tensor, title) -> plt.Figure:
    """1 row × 3 polar plots: azimuth / ZA rose."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                             subplot_kw={"projection": "polar"})
    fig.suptitle(f"Azimuth / ZA Rose  {title}", fontsize=13, weight="bold")
    _render_rose(fig, axes, tensor)
    return fig


def make_ext_hist_figure(tensor, title) -> plt.Figure:
    """Single panel extinction histogram."""
    fig, ax = plt.subplots(figsize=(9, 5))
    fig.suptitle(f"Extinction Distribution  {title}",
                 fontsize=13, weight="bold")
    _render_ext_hist(tensor, ax)
    plt.tight_layout()
    return fig


def make_cross_night_figure(tensors_by_night: dict) -> plt.Figure:
    """
    Grid (rows=nights, cols=strategies) of pointing-density alt/az maps,
    all sharing a single global colour scale.
    """
    nights = sorted(tensors_by_night)
    aze, alte = _az_alt_edges()

    # Pre-compute histograms and global max
    Hs, gmax = {}, 1.0
    for night, t in tensors_by_night.items():
        Hs[night] = {}
        for sid in STRATEGY_NAMES:
            sub = t[t[:, 7] == sid]
            if len(sub) == 0:
                continue
            H = _count2d(sub[:, 3], sub[:, 2], aze, alte)
            Hs[night][sid] = H
            v = np.nanpercentile(H[~np.isnan(H)], 98)
            gmax = max(gmax, v)

    fig, axes = plt.subplots(
        len(nights), 3,
        figsize=(7*3, 3.5*len(nights)),
        sharex=True, sharey=True, squeeze=False)
    fig.suptitle("Cross-Night Pointing Density\n"
                 "(rows = nights  ·  cols = strategies)",
                 fontsize=13, weight="bold")

    for row, night in enumerate(nights):
        for col, (sid, sname) in enumerate(STRATEGY_NAMES.items()):
            ax = axes[row, col]
            if sid not in Hs.get(night, {}):
                ax.set_visible(False); continue
            ax.pcolormesh(aze, alte, Hs[night][sid],
                          cmap="plasma", vmin=0, vmax=gmax)
            if row == 0:
                ax.set_title(sname, weight="bold",
                             color=STRATEGY_COLORS[sid], fontsize=10)
            if col == 0:
                ax.set_ylabel(f"{night}\nAlt (°)", fontsize=8)
            ax.set_xlim(0, 360); ax.set_ylim(MIN_ALT_DEG, MAX_ALT_DEG)
            ax.tick_params(labelsize=7)

    sm = plt.cm.ScalarMappable(cmap="plasma",
                                norm=mcolors.Normalize(0, gmax))
    sm.set_array([])
    fig.colorbar(sm, ax=axes.ravel().tolist(),
                 label="# pointings", shrink=0.55, pad=0.02)
    plt.tight_layout(rect=[0, 0, 0.97, 0.96])
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Save helper + batch renderer
# ─────────────────────────────────────────────────────────────────────────────

def _save(fig, path):
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    → {path}")


def render_all(tensor, out_dir, tag):
    os.makedirs(out_dir, exist_ok=True)
    base  = os.path.join(out_dir, f"sky_map_{tag}")
    title = f"[{tag.replace('pointing_tensor_', '')}]"

    n = {sid: int(np.sum(tensor[:, 7] == sid)) for sid in STRATEGY_NAMES}
    print(f"  absolute={n[0]:,}  motion={n[1]:,}  scheduler={n[2]:,}")

    _save(make_density_xy_figure(tensor, title), f"{base}_density_xy.png")
    _save(make_extinction_figure(tensor, title), f"{base}_extinction.png")
    _save(make_time_seq_figure(tensor,   title), f"{base}_time_seq.png")
    _save(make_rose_figure(tensor,       title), f"{base}_rose.png")
    _save(make_ext_hist_figure(tensor,   title), f"{base}_ext_hist.png")


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

def main(
    npz=None,
    npz_dir=None,
    combined=False,
    cross_night=False,
    out_dir=None,
):
    """
    Can be called three ways:

    1. From the command line:
           python plot_sky_maps.py --npz_dir ./outputs --cross_night

    2. From a Jupyter cell (keyword args):
           from plot_sky_maps import main
           main(npz_dir="./outputs", cross_night=True)

    3. As __main__ in a Jupyter notebook (top-of-cell):
           # just run the cell — argparse is Jupyter-safe

    Parameters
    ----------
    npz         : list[str] | None   explicit .npz paths
    npz_dir     : str | None         directory to glob for pointing_tensor_*.npz
    combined    : bool               merge all inputs into one plot set
    cross_night : bool               produce cross-night stacked comparison grid
    out_dir     : str | None         output directory (default: next to each .npz)
    """
    # ── Resolve arguments: keyword call wins; fall back to CLI when running
    #    as a script.  parse_known_args() silently ignores Jupyter's -f flag.
    _from_cli = (npz is None and npz_dir is None)
    if _from_cli:
        ap = argparse.ArgumentParser(
            description="Plot DREAM pointing-tensor sky-maps",
            formatter_class=argparse.RawDescriptionHelpFormatter,
            epilog=__doc__)
        ap.add_argument("--npz", nargs="+", default=[],
                        help="Explicit .npz file(s)")
        ap.add_argument("--npz_dir", default=None,
                        help="Directory containing pointing_tensor_*.npz files")
        ap.add_argument("--combined", action="store_true",
                        help="Merge all inputs into one combined plot set")
        ap.add_argument("--cross_night", action="store_true",
                        help="Produce cross-night stacked comparison grid")
        ap.add_argument("--out_dir", default=None,
                        help="Output directory (default: next to each .npz)")
        # parse_known_args ignores unknown args like Jupyter's -f kernel.json
        args, _unknown = ap.parse_known_args()
        npz         = args.npz or []
        npz_dir     = args.npz_dir
        combined    = args.combined
        cross_night = args.cross_night
        out_dir     = args.out_dir

    npz = list(npz) if npz else []

    # Collect files
    if npz_dir:
        npz += sorted(glob.glob(
            os.path.join(npz_dir, "pointing_tensor_*.npz")))
    files = list(dict.fromkeys(npz))   # deduplicate, preserve order

    if not files:
        print(
            "\n⚠  No .npz files found.\n"
            "   Edit the config block at the bottom of this file:\n"
            "     NPZ_DIR  = '/path/to/your/outputs'   ← directory with pointing_tensor_*.npz\n"
            "     NPZ_FILES = ['file1.npz', ...]        ← or list files explicitly\n"
            "   Then re-run the cell."
        )
        return
    print(f"\nFound {len(files)} file(s)")

    if combined:
        tensor = load_all_tensors(files)
        out    = out_dir or os.path.dirname(os.path.abspath(files[0]))
        print(f"\nCombined: {tensor.shape[0]:,} total rows")
        render_all(tensor, out, "all_nights")

    else:
        tensors_by_night = {}
        for path in files:
            tag = tag_from_path(path)
            try:
                t = load_tensor(path)
            except Exception as e:
                print(f"  WARN: {path}: {e}"); continue

            night = tag.replace("pointing_tensor_", "")
            tensors_by_night[night] = t

            out = out_dir or os.path.dirname(os.path.abspath(path))
            print(f"\n[{night}]")
            render_all(t, out, tag)

        if cross_night and len(tensors_by_night) > 1:
            out  = out_dir or os.path.dirname(os.path.abspath(files[0]))
            path = os.path.join(out, "sky_map_cross_night.png")
            print(f"\nCross-night grid ({len(tensors_by_night)} nights):")
            _save(make_cross_night_figure(tensors_by_night), path)

    print("\nDone.")


# ─────────────────────────────────────────────────────────────────────────────
# ✏️  CONFIGURE HERE when running as a Jupyter cell
# ─────────────────────────────────────────────────────────────────────────────
# Edit the values below, then run this cell.
# Leave NPZ_DIR pointing at wherever your pipeline wrote its .npz files.

NPZ_FILES   = []          # e.g. ["./outputs/pointing_tensor_20250627.npz"]
NPZ_DIR     = "."         # directory to scan for pointing_tensor_*.npz
COMBINED    = False       # True  → merge all nights into one plot set
CROSS_NIGHT = True        # True  → also save cross-night comparison grid
OUT_DIR     = None        # None  → write plots next to each .npz file
# ─────────────────────────────────────────────────────────────────────────────


if __name__ == "__main__":
    import sys as _sys
    _is_jupyter = "ipykernel" in _sys.modules or \
                  any("ipykernel" in str(a) or "jupyter" in str(a)
                      for a in _sys.argv)

    if _is_jupyter:
        # Running as a notebook cell — use the config block above
        main(
            npz         = NPZ_FILES   or None,
            npz_dir     = NPZ_DIR,
            combined    = COMBINED,
            cross_night = CROSS_NIGHT,
            out_dir     = OUT_DIR,
        )
    else:
        # Running as a script — parse CLI flags (parse_known_args ignores
        # any extra args that Jupyter might inject)
        main()

FileNotFoundError: No .npz files found.  Pass npz=[...] or npz_dir='...'

In [8]:
#!/usr/bin/env python3
"""
DREAM Cloud Analysis Pipeline
==============================
Night: configurable (default 2026-01-25)
Three pointing strategies:
  Green  - Absolute minimum cloud coverage
  Blue   - Predicted minimum via cloud motion tracking
  Red    - Baseline scheduler observations

Pipeline steps:
  1. Load night frames from CSV
  2. Patchiness check: sample ~10 frames, compute spatial variance.
     - All-clear OR all-cloudy  → skip (no science in tracking)
     - Patchy (high variation)  → proceed with full analysis
  3. Full analysis:
     a. Cloud motion tracking for all frames
     b. Metrics (photons, slew, efficiency) with correct slew-gating
     c. Metrics summary plot (left panel only, no live video subplot)
     d. MP4 cloud tracking video (map panel only)
     e. ZP cross-validation: DREAM .zps vs ConsDB
     f. ZP comparison plot

Modifications (v2):
  (a) Moon avoidance: never point within MOON_EXCLUSION_DEG of the Moon
      for absolute-min and motion-tracking strategies (moon position computed
      per-frame from Astropy ephemeris).
  (b) Zenith-proximity weighting in motion-tracking candidate selection:
      scored as  score = w_cloud * cloud_val + w_slew * slew_cost + w_zenith * zenith_cost
      where zenith_cost = zenith_angle / 90.
  (c) Post-CSV atmospheric + seeing quality corrections applied to summary
      stats using ConsDB quicklook columns (eff_time_*, psf_sigma_median,
      zero_point_median, seeing_zenith_500nm_median, airmass).
  (d) Spatial pointing tensors saved per-night (and across all nights) as
      compressed NumPy .npz files, enabling sky-map visualisations later.

Coordinate / math notes (bugs fixed):
  - Scheduler field RA/Dec converted to alt/az at the ACTUAL frame timestamp,
    then rejected if alt < 15 deg (below horizon guard).
  - xy_to_altaz and radec_to_altaz_xy use consistent projection:
      x = -cos(alt)*sin(az),  y = cos(alt)*cos(az),  scale = R/sin(alt)
  - Slew gating: expose_t = max(0, SLOT_TIME - slew_t);
      photons  = photons_full * (expose_t / EXPOSURE_TIME)

HDF5 schema (confirmed):
  clouds        sys extinction in mag  (float32)
  sigma         1-sigma uncertainty    (float32)
  flags         quality flags 0-3      (int16,  0 = good)
  mask_visible  bool, True = observable
  nobs          number of ref-star obs (int32)
"""

import io
import os
import re
import sqlite3
import warnings
import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.dates as mdates
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import gaussian_filter
from scipy import ndimage, stats
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath
from lsst.summit.utils import ConsDbClient

warnings.filterwarnings("ignore")

os.environ.setdefault("no_proxy", "")
if ".consdb" not in os.environ["no_proxy"]:
    os.environ["no_proxy"] += ",.consdb"

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
NSIDE_EXPECTED = 32
NEST           = True
UNSEEN         = hp.UNSEEN if hasattr(hp, "UNSEEN") else np.nan

RUBIN_LAT      = -30.244639
RUBIN_LON      = -70.749417
RUBIN_HEIGHT_M = 2663.0

BIN_SIZE_KM  = 1000
R_PROJECTION = 10000.0   # km, height of projection plane above observer

CBAR_VMIN = -0.2
CBAR_VMAX  = 2.0

# Photometry
RUBIN_EFFECTIVE_AREA     = np.pi * (6.4 / 2) ** 2   # m²
RUBIN_QUANTUM_EFFICIENCY = 0.8
EXPOSURE_TIME            = 30.0                      # s
READOUT_TIME             = 2.0                       # s
SLOT_TIME                = EXPOSURE_TIME + READOUT_TIME  # 32 s
PHOTON_FLUX_MAG20_ZENITH = 100.0                     # photons s⁻¹ m⁻² QE-weighted

# Slew
MAX_SLEW_SPEED_ALT = 3.5   # deg/s
MAX_SLEW_SPEED_AZ  = 7.0   # deg/s
SLEW_SETTLE_TIME   = 2.0   # s

# Pixel quality thresholds
MAX_SIGMA_MAG  = 0.3
MAX_FLAG_VALUE = 0

# Horizon guard for scheduler
MIN_ALT_DEG = 15.0

# ── (a) Moon avoidance ──────────────────────────────────────────────────────
MOON_EXCLUSION_DEG = 30.0   # reject pointings within this angular radius of Moon

# ── (b) Scoring weights for motion-tracking candidate selection ─────────────
# score = w_cloud * normalised_extinction
#       + w_slew  * normalised_slew_cost
#       + w_zenith* normalised_zenith_angle
# Lower score = better.
W_CLOUD  = 0.50   # cloud extinction weight
W_SLEW   = 0.25   # slew-time weight (matches original slew-preference)
W_ZENITH = 0.25   # zenith-proximity weight (new)

# Normalisation references
MAX_EXTINCTION_NORM = 2.0   # mag — used to normalise extinction to [0,1]
MAX_SLEW_NORM       = 60.0  # s   — slew times beyond this are clipped to 1

# Patchiness thresholds
PATCHINESS_SAMPLE_N   = 10   # frames to sample for patchiness check
PATCHINESS_LOW_FRAC   = 0.10  # below this → mostly clear, skip
PATCHINESS_HIGH_FRAC  = 0.90  # above this → mostly cloudy, skip
PATCHINESS_VAR_THRESH = 0.05  # mag²; spatial variance threshold for "patchy"

# ConsDB / ZP cross-val
CONSDB_URL  = "http://consdb-pq.consdb:8080/consdb"
INSTRUMENT  = "lsstcam"
N_ZP_SAMPLES = 20
ZP_SEED      = 42
ALPHA_SEEING = 0.6
BETA_SEEING  = 0.2
FWHM_CONSTANT = 2 * np.sqrt(2 * np.log(2))
EFFECTIVE_WAVELENGTHS = {
    'u': 366.3, 'g': 481.0, 'r': 622.2,
    'i': 754.5, 'z': 869.1, 'y': 971.0
}
FILENAME_BAND_MAP = {
    'B': None, 'u': 'u', 'g': 'g',
    'r': 'r',  'i': 'i', 'z': 'z', 'y': 'y',
}

URL_COL  = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL = "time"

# ── (d) Spatial tensor dtype ─────────────────────────────────────────────────
# Each frame row: [frame_idx, unix_time, alt, az, x_km, y_km, ext, strategy_id]
# strategy_id: 0=absolute, 1=motion, 2=scheduler
TENSOR_DTYPE = np.float32


# ─────────────────────────────────────────────────────────────────────────────
# URL helpers
# ─────────────────────────────────────────────────────────────────────────────

def transform_url(url: str) -> str:
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url


def band_from_url(url: str):
    m = re.search(r'_([A-Za-z])_cloud_zps', url)
    return m.group(1) if m else None


# ─────────────────────────────────────────────────────────────────────────────
# HDF5 loaders
# ─────────────────────────────────────────────────────────────────────────────

def fetch_sys_map(url: str):
    """Load cloud_sys HDF5. Returns (clouds, sigma) with bad pixels → NaN."""
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    with h5py.File(io.BytesIO(data), "r") as f:
        clouds       = np.array(f["clouds"],       dtype=float).ravel()
        sigma        = np.array(f["sigma"],        dtype=float).ravel()
        flags        = np.array(f["flags"],        dtype=int  ).ravel()
        mask_visible = np.array(f["mask_visible"], dtype=bool ).ravel()
        nobs         = np.array(f["nobs"],         dtype=int  ).ravel()
    bad = (
        ~mask_visible
        | (nobs == 0)
        | (flags > MAX_FLAG_VALUE)
        | (sigma > MAX_SIGMA_MAG)
        | ~np.isfinite(clouds)
    )
    clouds[bad] = np.nan
    sigma [bad] = np.nan
    return clouds, sigma


def fetch_zps_map(url: str):
    """Load cloud_zps HDF5. Returns (clouds, sigma) with bad pixels → NaN."""
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    with h5py.File(io.BytesIO(data), "r") as f:
        clouds       = np.array(f["clouds"],       dtype=float).ravel()
        sigma        = np.array(f["sigma"],        dtype=float).ravel()
        flags        = np.array(f["flags"],        dtype=int  ).ravel()
        mask_visible = np.array(f["mask_visible"], dtype=bool ).ravel()
        nobs         = np.array(f["nobs"],         dtype=int  ).ravel()
    bad = (
        ~mask_visible
        | (nobs == 0)
        | (flags > MAX_FLAG_VALUE)
        | (sigma > MAX_SIGMA_MAG)
        | ~np.isfinite(clouds)
    )
    clouds[bad] = np.nan
    sigma [bad] = np.nan
    return clouds, sigma


# ─────────────────────────────────────────────────────────────────────────────
# Coordinate utilities
# ─────────────────────────────────────────────────────────────────────────────

def _make_location():
    return EarthLocation(lat=RUBIN_LAT*u.deg,
                         lon=RUBIN_LON*u.deg,
                         height=RUBIN_HEIGHT_M*u.m)


def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc


def radec_to_altaz(ra_deg, dec_deg, obstime):
    """Convert RA/Dec → (alt, az) in degrees."""
    t   = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    return float(aa.alt.deg), float(aa.az.deg % 360.0)


def altaz_to_xy(alt_deg, az_deg):
    """
    Project (alt, az) to flat-sky km coordinates.
    Returns (x_km, y_km) or (None, None) if below horizon.
    """
    if alt_deg < MIN_ALT_DEG:
        return None, None
    alt_r = np.radians(alt_deg)
    az_r  = np.radians(az_deg)
    scale = R_PROJECTION / np.sin(alt_r)
    x_km  = -np.cos(alt_r) * np.sin(az_r) * scale
    y_km  =  np.cos(alt_r) * np.cos(az_r) * scale
    return x_km, y_km


def xy_to_altaz(x_km, y_km):
    """Inverse projection: km → (alt, az) in degrees."""
    r       = np.sqrt(x_km**2 + y_km**2)
    alt_deg = float(np.degrees(np.arctan2(R_PROJECTION, r)))
    az_deg  = float(np.degrees(np.arctan2(-x_km, y_km)) % 360.0)
    return alt_deg, az_deg


def xy_to_zenith_angle(x_km, y_km):
    alt_deg, _ = xy_to_altaz(x_km, y_km)
    return 90.0 - alt_deg


def radec_to_xy(ra_deg, dec_deg, obstime):
    """
    Convert RA/Dec to flat-sky km, rejecting below-horizon pointings.
    Returns (x_km, y_km, alt_deg, az_deg); x/y are None if alt < MIN_ALT_DEG.
    """
    alt, az = radec_to_altaz(ra_deg, dec_deg, obstime)
    x, y    = altaz_to_xy(alt, az)
    return x, y, alt, az


def healpix_to_altaz_vals(mp, nside, obstime):
    """Map all HEALPix pixels to (az, alt, vals) arrays."""
    npix = hp.nside2npix(nside)
    pix  = np.arange(npix)
    theta, phi = hp.pix2ang(nside, pix, nest=NEST)
    ra  = np.degrees(phi)
    dec = 90.0 - np.degrees(theta)
    t   = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    vals = np.asarray(mp, dtype=float)
    vals = np.where(vals == UNSEEN, np.nan, vals)
    return aa.az.deg % 360.0, aa.alt.deg, vals


def _healpix_to_grid(mp, obstime, bin_size=BIN_SIZE_KM):
    """Grid a HEALPix map onto flat-sky km bins."""
    az, alt, vals = healpix_to_altaz_vals(mp, NSIDE_EXPECTED, obstime)
    alt_r = np.radians(alt)
    az_r  = np.radians(az)
    above = alt > MIN_ALT_DEG
    scale = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf    = -np.cos(alt_r) * np.sin(az_r) * scale
    yf    =  np.cos(alt_r) * np.cos(az_r) * scale
    vf    = np.where(above, vals, np.nan)

    r  = np.sqrt(xf**2 + yf**2)
    c  = r <= 15000.0
    ok = ~np.isnan(vf[c])

    x_edges = np.arange(-15000, 15001, bin_size)
    y_edges = np.arange(-15000, 15001, bin_size)
    Hs, _, _ = np.histogram2d(xf[c][ok], yf[c][ok],
                               bins=[x_edges, y_edges], weights=vf[c][ok])
    Hc, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[x_edges, y_edges])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (x_edges[:-1] + x_edges[1:]) / 2
    yc = (y_edges[:-1] + y_edges[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15000] = np.nan
    return H, Xg, Yg, x_edges, y_edges


def process_to_grid(clouds, sigma, obstime, bin_size=BIN_SIZE_KM):
    ext_grid, Xg, Yg, xe, ye = _healpix_to_grid(clouds, obstime, bin_size)
    sig_grid, *_              = _healpix_to_grid(sigma,  obstime, bin_size)
    return ext_grid, sig_grid, Xg, Yg, xe, ye


# ─────────────────────────────────────────────────────────────────────────────
# (a) Moon position helpers
# ─────────────────────────────────────────────────────────────────────────────

def get_moon_altaz(obstime):
    """
    Return (alt_deg, az_deg) of the Moon at obstime as seen from Rubin.
    Uses Astropy's get_body for the ephemeris.
    """
    t   = _ensure_time(obstime)
    loc = _make_location()
    moon = get_body("moon", t, loc)
    aa   = moon.transform_to(AltAz(obstime=t, location=loc))
    return float(aa.alt.deg), float(aa.az.deg % 360.0)


def angular_separation_altaz(alt1, az1, alt2, az2):
    """Great-circle angular separation between two alt/az pairs (degrees)."""
    a1, a2 = np.radians(alt1), np.radians(alt2)
    z1, z2 = np.radians(az1),  np.radians(az2)
    cos_sep = (np.sin(a1)*np.sin(a2)
               + np.cos(a1)*np.cos(a2)*np.cos(z1 - z2))
    cos_sep = np.clip(cos_sep, -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_sep)))


def is_moon_safe(alt_deg, az_deg, moon_alt, moon_az,
                 exclusion_deg=MOON_EXCLUSION_DEG):
    """True if the pointing is outside the moon exclusion zone."""
    if alt_deg < MIN_ALT_DEG:
        return False
    sep = angular_separation_altaz(alt_deg, az_deg, moon_alt, moon_az)
    return sep >= exclusion_deg


def xy_moon_safe(x_km, y_km, moon_alt, moon_az,
                 exclusion_deg=MOON_EXCLUSION_DEG):
    """True if grid position (x_km, y_km) is moon-safe."""
    alt, az = xy_to_altaz(x_km, y_km)
    return is_moon_safe(alt, az, moon_alt, moon_az, exclusion_deg)


# ─────────────────────────────────────────────────────────────────────────────
# Patchiness detection
# ─────────────────────────────────────────────────────────────────────────────

def compute_patchiness(grids):
    idx = np.linspace(0, len(grids) - 1, min(PATCHINESS_SAMPLE_N, len(grids)),
                      dtype=int)
    fracs, vars_ = [], []
    for i in idx:
        g = grids[i]
        valid = ~np.isnan(g)
        frac  = np.sum(valid) / g.size
        var   = float(np.nanvar(g)) if np.sum(valid) > 10 else 0.0
        fracs.append(frac)
        vars_.append(var)

    mean_frac = float(np.mean(fracs))
    mean_var  = float(np.mean(vars_))

    if mean_frac < PATCHINESS_LOW_FRAC:
        return False, mean_frac, mean_var, \
            f"Sky mostly clear (valid_frac={mean_frac:.2f} < {PATCHINESS_LOW_FRAC})"
    if mean_frac > PATCHINESS_HIGH_FRAC:
        return False, mean_frac, mean_var, \
            f"Sky mostly clouded out (valid_frac={mean_frac:.2f} > {PATCHINESS_HIGH_FRAC})"
    if mean_var < PATCHINESS_VAR_THRESH:
        return False, mean_frac, mean_var, \
            f"Low spatial variation (var={mean_var:.4f} < {PATCHINESS_VAR_THRESH}) — uniform cover"
    return True, mean_frac, mean_var, \
        f"Patchy clouds detected (valid_frac={mean_frac:.2f}, var={mean_var:.4f})"


# ─────────────────────────────────────────────────────────────────────────────
# Slew / photometry
# ─────────────────────────────────────────────────────────────────────────────

def calculate_slew_time(alt1, az1, alt2, az2):
    da  = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180:
        daz = 360 - daz
    return max(da / MAX_SLEW_SPEED_ALT, daz / MAX_SLEW_SPEED_AZ) + SLEW_SETTLE_TIME


def compute_photon_collection(ext_mag, _zenith_angle_deg=None):
    """Full-exposure photons (no slew penalty applied here)."""
    if np.isnan(ext_mag):
        return np.nan
    flux = 10 ** (-0.4 * ext_mag)
    rate = (PHOTON_FLUX_MAG20_ZENITH
            * RUBIN_EFFECTIVE_AREA
            * RUBIN_QUANTUM_EFFICIENCY
            * flux)
    return rate * EXPOSURE_TIME


def photons_to_magnitude(photons):
    if photons is None or np.isnan(photons) or photons <= 0:
        return np.nan
    snr = photons / (5 * np.sqrt(photons))
    return 20.0 - 2.5 * np.log10(snr) if snr > 0 else np.nan


# ─────────────────────────────────────────────────────────────────────────────
# Grid helpers
# ─────────────────────────────────────────────────────────────────────────────

def get_value_at_position(grid, x_km, y_km):
    xi = int(round((x_km / BIN_SIZE_KM) + 15))
    yi = int(round((y_km / BIN_SIZE_KM) + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan


def find_absolute_minimum(grid, moon_alt=None, moon_az=None):
    """
    Find the grid cell with minimum valid extinction.
    If moon_alt/moon_az are provided, cells within MOON_EXCLUSION_DEG of the
    Moon are masked out first.
    """
    work = grid.copy()

    # (a) Moon masking
    if moon_alt is not None and moon_az is not None:
        ny, nx = work.shape
        for yi in range(ny):
            for xi in range(nx):
                if np.isnan(work[yi, xi]):
                    continue
                x_km = (xi - 15) * BIN_SIZE_KM
                y_km = (yi - 15) * BIN_SIZE_KM
                if not xy_moon_safe(x_km, y_km, moon_alt, moon_az):
                    work[yi, xi] = np.nan

    if not np.any(~np.isnan(work)):
        # Fallback: ignore moon masking if it wipes everything
        work = grid.copy()
    if not np.any(~np.isnan(work)):
        return 0, 0, np.nan

    idx = np.nanargmin(work)
    yi, xi = np.unravel_index(idx, work.shape)
    return (xi - 15) * BIN_SIZE_KM, (yi - 15) * BIN_SIZE_KM, grid[yi, xi]


# ─────────────────────────────────────────────────────────────────────────────
# (b) Scored motion-tracking candidate selection
#     Combines cloud extinction, slew cost, and zenith proximity.
# ─────────────────────────────────────────────────────────────────────────────

def _score_candidate(x_km, y_km, ext_val, prev_alt, prev_az,
                     moon_alt=None, moon_az=None):
    """
    Return a scalar score (lower = better) for a candidate pointing.
    Returns np.inf if the candidate is invalid (NaN extinction, below horizon,
    inside moon exclusion zone).
    """
    if np.isnan(ext_val):
        return np.inf
    alt, az = xy_to_altaz(x_km, y_km)
    if alt < MIN_ALT_DEG:
        return np.inf
    if moon_alt is not None and moon_az is not None:
        if not is_moon_safe(alt, az, moon_alt, moon_az):
            return np.inf

    # Normalised extinction  [0, 1]
    cloud_norm = np.clip(ext_val / MAX_EXTINCTION_NORM, 0.0, 1.0)

    # Normalised slew cost  [0, 1]
    if prev_alt is not None:
        slew_s = calculate_slew_time(prev_alt, prev_az, alt, az)
    else:
        slew_s = 0.0
    slew_norm = np.clip(slew_s / MAX_SLEW_NORM, 0.0, 1.0)

    # Normalised zenith angle  [0, 1]
    zenith_norm = (90.0 - alt) / 90.0

    return W_CLOUD * cloud_norm + W_SLEW * slew_norm + W_ZENITH * zenith_norm


def predict_future_position(cx, cy, dx_pix, dy_pix, current_grid,
                             prev_alt=None, prev_az=None,
                             moon_alt=None, moon_az=None,
                             fallback_threshold=0.5,
                             n_candidates=9):
    """
    Predict next best pointing given cloud motion (dx_pix, dy_pix).

    Strategy:
      1. Primary: the motion-extrapolated position.
      2. Fallback grid: sample n_candidates positions around the primary,
         score each with _score_candidate, pick the best.
      3. If all candidates fail, fall back to find_absolute_minimum
         (with moon masking).
    """
    nx = cx - dx_pix * BIN_SIZE_KM
    ny = cy - dy_pix * BIN_SIZE_KM
    # Clamp to observable disk
    r = np.sqrt(nx**2 + ny**2)
    if r > 14000:
        nx *= 14000 / r
        ny *= 14000 / r

    # Build candidate grid around the primary prediction
    offsets = [0] + list(range(-2, 3))  # -2,-1,0,1,2 in each axis
    candidates = []
    for dyi in offsets:
        for dxi in offsets:
            cx_ = nx + dxi * BIN_SIZE_KM
            cy_ = ny + dyi * BIN_SIZE_KM
            ext = get_value_at_position(current_grid, cx_, cy_)
            candidates.append((cx_, cy_, ext))

    best_score = np.inf
    best_x, best_y = nx, ny
    for (cx_, cy_, ext) in candidates:
        s = _score_candidate(cx_, cy_, ext, prev_alt, prev_az,
                             moon_alt, moon_az)
        if s < best_score:
            best_score = s
            best_x, best_y = cx_, cy_

    if best_score == np.inf:
        # All candidates failed — fall back to global minimum with moon masking
        best_x, best_y, _ = find_absolute_minimum(current_grid,
                                                   moon_alt, moon_az)
    return best_x, best_y


# ─────────────────────────────────────────────────────────────────────────────
# Cloud motion tracking
# ─────────────────────────────────────────────────────────────────────────────

def compute_motion_with_correlation(grid1, grid2, sigma=5.0, search_range=10):
    g1 = np.nan_to_num(grid1, nan=0)
    g2 = np.nan_to_num(grid2, nan=0)
    g1s = gaussian_filter(g1, sigma=sigma)
    g2s = gaussian_filter(g2, sigma=sigma)
    m1  = ~np.isnan(grid1) & (grid1 != 0)
    m2  = ~np.isnan(grid2) & (grid2 != 0)
    best_corr, best_dx, best_dy = -np.inf, 0, 0
    for dy in range(-search_range, search_range + 1):
        for dx in range(-search_range, search_range + 1):
            sh  = ndimage.shift(g2s,       (dy, dx), order=1, mode="constant", cval=0)
            sm  = ndimage.shift(m2.astype(float), (dy, dx), order=0, mode="constant", cval=0) > 0.5
            val = m1 & sm
            if np.sum(val) < 100:
                continue
            v1, v2 = g1s[val], sh[val]
            if np.std(v1) > 0 and np.std(v2) > 0:
                corr = np.corrcoef(v1, v2)[0, 1]
                if corr > best_corr:
                    best_corr, best_dx, best_dy = corr, dx, dy
    return best_dx, best_dy, best_corr


def compute_all_positions(all_grids, all_metas):
    """
    Compute absolute and predicted positions for all frames.
    Moon position computed once per frame from the frame timestamp.
    all_metas must contain 'time' keys.
    """
    t0      = all_metas[0]["time"]
    m_alt0, m_az0 = get_moon_altaz(t0)

    x0, y0, _ = find_absolute_minimum(all_grids[0], m_alt0, m_az0)
    abs_pos  = [(x0, y0)]
    pred_pos = [(x0, y0)]
    motion_v = [(0, 0)]
    moon_pos = [(m_alt0, m_az0)]
    xp, yp   = x0, y0
    prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)
    HL = 3

    for i in range(1, len(all_grids)):
        if i % 50 == 0:
            print(f"  Tracking frame {i}/{len(all_grids)}")

        m_alt, m_az = get_moon_altaz(all_metas[i]["time"])
        moon_pos.append((m_alt, m_az))

        xa, ya, _ = find_absolute_minimum(all_grids[i], m_alt, m_az)
        abs_pos.append((xa, ya))

        if i >= HL:
            dxs, dys = [], []
            for j in range(1, HL + 1):
                dx, dy, conf = compute_motion_with_correlation(
                    all_grids[i-j], all_grids[i-j+1])
                if conf > 0.5:
                    dxs.append(dx); dys.append(dy)
            dx_avg = float(np.mean(dxs)) if dxs else 0.0
            dy_avg = float(np.mean(dys)) if dys else 0.0
        else:
            dx_avg, dy_avg, _ = compute_motion_with_correlation(
                all_grids[i-1], all_grids[i])

        motion_v.append((dx_avg, dy_avg))

        # (a)+(b) scored prediction with moon avoidance + zenith weighting
        xp, yp = predict_future_position(
            xp, yp, dx_avg, dy_avg, all_grids[i],
            prev_alt=prev_alt_p, prev_az=prev_az_p,
            moon_alt=m_alt, moon_az=m_az)
        pred_pos.append((xp, yp))
        prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)

    return abs_pos, pred_pos, motion_v, moon_pos


# ─────────────────────────────────────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────────────────────────────────────

def load_all_sys_frames(csv_file):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains(".hdf5",     case=False, na=False)].copy()
    df = df[df[URL_COL].str.contains("cloud_sys", case=False, na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    shifted = df[TIME_COL] - pd.Timedelta(hours=12)
    df["night_key"] = shifted.dt.date
    return df


def load_all_zps_frames(csv_file):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains(".hdf5",     case=False, na=False)].copy()
    df = df[df[URL_COL].str.contains("cloud_zps", case=False, na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    shifted = df[TIME_COL] - pd.Timedelta(hours=12)
    df["night_key"] = shifted.dt.date
    df["dream_band"] = df[URL_COL].apply(band_from_url)
    return df


def get_night_df(all_df, night_key):
    return all_df[all_df["night_key"] == night_key].copy().reset_index(drop=True)


def quick_patchiness_check(df_night, n_probe=PATCHINESS_SAMPLE_N):
    if len(df_night) == 0:
        return False, 0.0, 0.0, "No frames", []
    idx    = np.linspace(0, len(df_night) - 1, min(n_probe, len(df_night)), dtype=int)
    grids  = []
    metas  = []
    for i in idx:
        row = df_night.iloc[i]
        url = transform_url(row[URL_COL])
        try:
            clouds, sigma = fetch_sys_map(url)
            meta = {"url": url, "time": row[TIME_COL].to_pydatetime()}
            ext_g, sig_g, _, _, _, _ = process_to_grid(clouds, sigma, meta["time"])
            grids.append(ext_g)
            metas.append(meta)
        except Exception as e:
            print(f"    probe load failed: {e}")
    if len(grids) < 2:
        return False, 0.0, 0.0, "Too few probe frames loaded", []
    patchy, mean_frac, mean_var, reason = compute_patchiness(grids)
    return patchy, mean_frac, mean_var, reason, grids


def load_scheduler_observations(db_file):
    print("\n" + "=" * 70)
    print("LOADING SCHEDULER OBSERVATIONS")
    print("=" * 70)
    conn   = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]
    table_name = next(
        (n for n in ["observations", "SummaryAllProps", "Summary", "obs"]
         if n in tables),
        tables[0] if tables else None)
    print(f"Using table: {table_name}")
    obs_df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()

    night_col = next((c for c in obs_df.columns if "night" in c.lower()), None)
    if night_col is None:
        raise RuntimeError("Could not find night column in scheduler DB")
    obs_df["night"] = obs_df[night_col].astype(int)
    for req in ["observationStartMJD", "fieldRA", "fieldDec"]:
        for col in obs_df.columns:
            if req.lower() in col.lower() and col != req:
                obs_df[req] = obs_df[col]

    night_scores = {}
    for night in obs_df["night"].unique()[:500]:
        no  = obs_df[obs_df["night"] == night]
        if len(no) < 200:
            continue
        dur = (no["observationStartMJD"].max() - no["observationStartMJD"].min()) * 24
        if dur < 5:
            continue
        night_scores[night] = (len(no) * 10 + dur, len(no), dur)
    if not night_scores:
        selected = obs_df.groupby("night").size().idxmax()
    else:
        selected = max(night_scores, key=lambda k: night_scores[k][0])
        _, n, dur = night_scores[selected]
        print(f"  Night {selected}: {n} obs over {dur:.1f} hours")

    night_obs = obs_df[obs_df["night"] == selected].copy()
    night_obs = night_obs.sort_values("observationStartMJD").reset_index(drop=True)
    print(f"Selected night {selected}: {len(night_obs)} observations")
    return night_obs, selected


def match_scheduler_to_frames(scheduler_obs, frame_times):
    n         = len(frame_times)
    fracs     = np.linspace(0, 1, n)
    mjd       = scheduler_obs["observationStartMJD"].values
    mjd_range = mjd.max() - mjd.min()
    s_fracs   = (mjd - mjd.min()) / mjd_range if mjd_range > 0 else np.zeros(len(mjd))
    matched   = [scheduler_obs.iloc[np.argmin(np.abs(s_fracs - f))] for f in fracs]
    print(f"  Matched {len(matched)} frames to scheduler observations")
    return matched


# ─────────────────────────────────────────────────────────────────────────────
# Metrics computation
# ─────────────────────────────────────────────────────────────────────────────

def _record_obs(metrics, key, photons_full, sigma_val, za, ext,
                slew_t, frame_i, alt, az, prev_alt, prev_az,
                x_km, y_km):
    expose_t  = max(0.0, SLOT_TIME - slew_t)
    collected = photons_full * (expose_t / EXPOSURE_TIME)
    metrics[key]["photons"].append(collected)
    metrics[key]["sigma"].append(sigma_val)
    metrics[key]["zenith_angles"].append(za)
    metrics[key]["extinctions"].append(ext)
    metrics[key]["slew_times"].append(slew_t)
    metrics[key]["expose_times"].append(expose_t)
    metrics[key]["frame_indices"].append(frame_i)
    metrics[key]["alt_az"].append((alt, az))
    metrics[key]["xy_km"].append((x_km, y_km))   # (d) store pointing xy
    prev_alt[key] = alt
    prev_az[key]  = az


def compute_metrics(all_grids, all_sigma_grids, all_metas,
                    abs_positions, pred_positions, moon_positions,
                    matched_scheduler):
    print("\n" + "=" * 70)
    print("COMPUTING METRICS (photon collection + slew gating)")
    print("=" * 70)
    print(f"  Slot = {EXPOSURE_TIME}s exp + {READOUT_TIME}s readout = {SLOT_TIME}s")
    print(f"  Horizon guard: alt ≥ {MIN_ALT_DEG}°")
    print(f"  Moon exclusion: {MOON_EXCLUSION_DEG}°  "
          f"| Zenith weight: {W_ZENITH}  Slew weight: {W_SLEW}")

    empty = {"photons": [], "sigma": [], "zenith_angles": [], "extinctions": [],
             "slew_times": [], "expose_times": [], "frame_indices": [],
             "alt_az": [], "xy_km": []}
    metrics  = {s: {k: list(v) for k, v in empty.items()}
                for s in ("absolute", "motion", "scheduler")}
    prev_alt = {s: None for s in metrics}
    prev_az  = {s: None for s in metrics}
    sched_rejected = 0

    for i in range(len(all_grids)):
        grid  = all_grids[i]
        sgrid = all_sigma_grids[i]
        m_alt, m_az = moon_positions[i]

        # ── Absolute minimum  (a: moon-masked) ───────────────────────────────
        xa, ya = abs_positions[i]
        alt_a, az_a = xy_to_altaz(xa, ya)
        if alt_a >= MIN_ALT_DEG and is_moon_safe(alt_a, az_a, m_alt, m_az):
            ext = get_value_at_position(grid,  xa, ya)
            sig = get_value_at_position(sgrid, xa, ya)
            za  = xy_to_zenith_angle(xa, ya)
            if not np.isnan(ext):
                slew = (calculate_slew_time(prev_alt["absolute"], prev_az["absolute"],
                                            alt_a, az_a)
                        if prev_alt["absolute"] is not None else 0.0)
                _record_obs(metrics, "absolute",
                            compute_photon_collection(ext),
                            sig, za, ext, slew, i, alt_a, az_a, prev_alt, prev_az,
                            xa, ya)

        # ── Motion tracking  (a+b: moon-masked + zenith-weighted) ────────────
        xp, yp = pred_positions[i]
        alt_p, az_p = xy_to_altaz(xp, yp)
        if alt_p >= MIN_ALT_DEG and is_moon_safe(alt_p, az_p, m_alt, m_az):
            ext = get_value_at_position(grid,  xp, yp)
            sig = get_value_at_position(sgrid, xp, yp)
            za  = xy_to_zenith_angle(xp, yp)
            if not np.isnan(ext):
                slew = (calculate_slew_time(prev_alt["motion"], prev_az["motion"],
                                            alt_p, az_p)
                        if prev_alt["motion"] is not None else 0.0)
                _record_obs(metrics, "motion",
                            compute_photon_collection(ext),
                            sig, za, ext, slew, i, alt_p, az_p, prev_alt, prev_az,
                            xp, yp)

        # ── Scheduler (unchanged — scheduler owns its own targeting logic) ───
        obs = matched_scheduler[i]
        xs, ys, alt_s, az_s = radec_to_xy(
            float(obs["fieldRA"]), float(obs["fieldDec"]),
            all_metas[i]["time"])
        if xs is None:
            sched_rejected += 1
            continue
        ext = get_value_at_position(grid,  xs, ys)
        sig = get_value_at_position(sgrid, xs, ys)
        za  = xy_to_zenith_angle(xs, ys)
        if not np.isnan(ext):
            slew = (calculate_slew_time(prev_alt["scheduler"], prev_az["scheduler"],
                                        alt_s, az_s)
                    if prev_alt["scheduler"] is not None else 0.0)
            _record_obs(metrics, "scheduler",
                        compute_photon_collection(ext),
                        sig, za, ext, slew, i, alt_s, az_s, prev_alt, prev_az,
                        xs, ys)

    if sched_rejected > 0:
        print(f"  ⚠ Scheduler: {sched_rejected} frames rejected (field below "
              f"alt={MIN_ALT_DEG}°)")
    return metrics


def print_statistics(metrics):
    print("\n" + "=" * 70)
    print("NIGHT PERFORMANCE SUMMARY")
    print("=" * 70)
    for strategy in ("absolute", "motion", "scheduler"):
        photons      = np.array(metrics[strategy]["photons"])
        expose_times = np.array(metrics[strategy]["expose_times"])
        slew_times   = np.array(metrics[strategy]["slew_times"])
        sigmas       = np.array(metrics[strategy]["sigma"])
        if len(photons) == 0:
            print(f"\n{strategy.upper()}: no valid data")
            continue

        total_ph   = np.sum(photons)
        total_exp  = np.sum(expose_times)
        total_slew = np.sum(slew_times)
        total_slot = len(photons) * SLOT_TIME
        eff        = total_exp / total_slot * 100 if total_slot > 0 else 0
        pct_lost   = np.mean(expose_times == 0) * 100
        mags       = [photons_to_magnitude(p) for p in photons]
        mags       = [m for m in mags if m is not None and not np.isnan(m)]
        med_depth  = np.median(mags) if mags else np.nan

        print(f"\n{strategy.upper()} STRATEGY:")
        print(f"  Total photons collected:     {total_ph:.3e}")
        print(f"  Number of slots:             {len(photons)}")
        print(f"  On-sky exposure time:        {total_exp:.0f} s  ({total_exp/3600:.2f} hr)")
        print(f"  Total slew time:             {total_slew:.0f} s  ({total_slew/3600:.2f} hr)")
        print(f"  Shutter-open efficiency:     {eff:.1f}%")
        print(f"  Slots lost to slew:          {pct_lost:.1f}%")
        print(f"  Median 5σ depth:             {med_depth:.2f} mag")
        print(f"  Mean pixel σ:                {np.nanmean(sigmas):.4f} mag")
        print(f"  Mean zenith angle:           {np.mean(metrics[strategy]['zenith_angles']):.1f}°")
        print(f"  Mean extinction:             {np.mean(metrics[strategy]['extinctions']):.3f} mag")


# ─────────────────────────────────────────────────────────────────────────────
# (c) Atmospheric + seeing quality correction using ConsDB quicklook columns
# ─────────────────────────────────────────────────────────────────────────────

def _safe_median(arr):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    return float(np.median(a)) if len(a) else np.nan


def _find_nearest_visits(consdb_df, x_km, y_km, obstime, window_s=300.0,
                         max_sep_deg=2.0):
    """
    Return rows from consdb_df whose pointing is within max_sep_deg of
    (x_km, y_km) and whose obs_midpoint is within ±window_s of obstime.
    Returns empty DataFrame if no match.
    """
    if consdb_df.empty:
        return pd.DataFrame()
    if isinstance(obstime, pd.Timestamp):
        t_center = obstime if obstime.tzinfo is not None else obstime.tz_localize("UTC")
    else:
        ts = pd.Timestamp(obstime)
        t_center = ts if ts.tzinfo is not None else ts.tz_localize("UTC")
    t_lo = t_center - pd.Timedelta(seconds=window_s)
    t_hi = t_center + pd.Timedelta(seconds=window_s)

    nearby = consdb_df[
        (consdb_df["obs_midpoint"] >= t_lo) &
        (consdb_df["obs_midpoint"] <= t_hi)
    ].copy()
    if nearby.empty:
        return nearby

    # Angular separation filter
    alt_p, az_p = xy_to_altaz(x_km, y_km)
    seps = []
    for _, row in nearby.iterrows():
        ra_v  = float(row.get("ra",  row.get("s_ra",  np.nan)))
        dec_v = float(row.get("dec", row.get("s_dec", np.nan)))
        if np.isnan(ra_v) or np.isnan(dec_v):
            seps.append(np.inf)
            continue
        alt_v, az_v = radec_to_altaz(ra_v, dec_v, t_center)
        seps.append(angular_separation_altaz(alt_p, az_p, alt_v, az_v))
    nearby = nearby[np.array(seps) <= max_sep_deg].copy()
    return nearby


def apply_atm_seeing_correction(metrics, all_metas, consdb_df):
    """
    (c) Post-process metrics to add seeing- and atmosphere-corrected photon
    estimates for each recorded slot.

    For every slot in every strategy, we look up the nearest ConsDB visit(s)
    and extract:
      - airmass                 → atmospheric transmission correction
      - psf_sigma_median        → seeing FWHM proxy
      - eff_time_median / eff_time_zero_point_scale_median → effective-time scale
      - zero_point_median       → zeropoint offset
      - seeing_zenith_500nm_median → zenith-normalised seeing

    The corrected metric is stored alongside the raw one.
    New keys added to each strategy sub-dict:
      photons_atm_corrected : list[float]   raw photons × eff_time_scale
      eff_time_scale        : list[float]   eff_time_zero_point_scale_median
      seeing_fwhm           : list[float]   arcsec FWHM at pointing
      airmass_consdb        : list[float]
      zero_point_consdb     : list[float]
    """
    print("\n" + "=" * 70)
    print("APPLYING ATMOSPHERIC + SEEING CORRECTIONS  (ConsDB quicklook)")
    print("=" * 70)

    if consdb_df is None or consdb_df.empty:
        print("  No ConsDB data — skipping corrections.")
        for s in metrics:
            n = len(metrics[s]["photons"])
            metrics[s]["photons_atm_corrected"] = [np.nan] * n
            metrics[s]["eff_time_scale"]        = [np.nan] * n
            metrics[s]["seeing_fwhm"]           = [np.nan] * n
            metrics[s]["airmass_consdb"]        = [np.nan] * n
            metrics[s]["zero_point_consdb"]     = [np.nan] * n
        return metrics

    for strategy in ("absolute", "motion", "scheduler"):
        n = len(metrics[strategy]["photons"])
        ph_corr   = []
        eft_scale = []
        see_fwhm  = []
        am_list   = []
        zp_list   = []

        for slot_i in range(n):
            frame_i = metrics[strategy]["frame_indices"][slot_i]
            x_km, y_km = metrics[strategy]["xy_km"][slot_i]
            obstime = all_metas[frame_i]["time"]
            raw_ph  = metrics[strategy]["photons"][slot_i]

            nearby = _find_nearest_visits(consdb_df, x_km, y_km, obstime)

            if nearby.empty:
                ph_corr.append(raw_ph)
                eft_scale.append(np.nan)
                see_fwhm.append(np.nan)
                am_list.append(np.nan)
                zp_list.append(np.nan)
                continue

            # Aggregate multiple close visits
            am  = _safe_median(nearby.get("airmass",                         [np.nan]))
            eft = _safe_median(nearby.get("eff_time_zero_point_scale_median", [np.nan]))
            if np.isnan(eft):
                eft = _safe_median(nearby.get("eff_time_median", [np.nan]))
            zp  = _safe_median(nearby.get("zero_point_median",               [np.nan]))

            # Seeing: prefer seeing_zenith_500nm_median, fall back to
            # psf_sigma_median → FWHM via FWHM_CONSTANT
            sz  = _safe_median(nearby.get("seeing_zenith_500nm_median", [np.nan]))
            if np.isnan(sz):
                ps = _safe_median(nearby.get("psf_sigma_median", [np.nan]))
                sz = float(FWHM_CONSTANT * ps) if np.isfinite(ps) else np.nan

            # Scale photons by effective-time scale factor (captures
            # atmosphere + sky background + PSF quality together)
            scale = eft if np.isfinite(eft) else 1.0
            ph_corr.append(raw_ph * scale)
            eft_scale.append(eft)
            see_fwhm.append(sz)
            am_list.append(am)
            zp_list.append(zp)

        metrics[strategy]["photons_atm_corrected"] = ph_corr
        metrics[strategy]["eff_time_scale"]        = eft_scale
        metrics[strategy]["seeing_fwhm"]           = see_fwhm
        metrics[strategy]["airmass_consdb"]        = am_list
        metrics[strategy]["zero_point_consdb"]     = zp_list

        n_matched = sum(1 for v in eft_scale if np.isfinite(v))
        print(f"  {strategy:12s}: {n_matched}/{n} slots matched to ConsDB visits")
        if n_matched > 0:
            print(f"    mean eff_time_scale = {np.nanmean(eft_scale):.4f}")
            print(f"    mean seeing FWHM    = {np.nanmean(see_fwhm):.3f} arcsec")

    return metrics


def print_corrected_statistics(metrics):
    """Print the atm/seeing-corrected summary."""
    print("\n" + "=" * 70)
    print("CORRECTED PERFORMANCE SUMMARY  (eff_time_scale applied)")
    print("=" * 70)
    for strategy in ("absolute", "motion", "scheduler"):
        ph_c = np.array(metrics[strategy].get("photons_atm_corrected", []))
        if len(ph_c) == 0 or not np.any(np.isfinite(ph_c)):
            print(f"\n{strategy.upper()}: no corrected data")
            continue
        print(f"\n{strategy.upper()} (corrected):")
        print(f"  Total corrected photons:  {np.nansum(ph_c):.3e}")
        print(f"  Mean eff_time_scale:      "
              f"{np.nanmean(metrics[strategy]['eff_time_scale']):.4f}")
        print(f"  Mean seeing FWHM:         "
              f"{np.nanmean(metrics[strategy]['seeing_fwhm']):.3f} arcsec")
        print(f"  Mean airmass (ConsDB):    "
              f"{np.nanmean(metrics[strategy]['airmass_consdb']):.3f}")


# ─────────────────────────────────────────────────────────────────────────────
# (d) Spatial pointing tensor I/O
# ─────────────────────────────────────────────────────────────────────────────
# Schema per row: [frame_idx, unix_time, alt_deg, az_deg, x_km, y_km,
#                  extinction, strategy_id(0/1/2)]
# Saved as float32 array shape (N_slots, 8).

STRATEGY_IDS = {"absolute": 0, "motion": 1, "scheduler": 2}


def build_pointing_tensor(metrics, all_metas):
    """
    Assemble an (N, 8) float32 array of all pointing records across all
    strategies for a single night.
    Columns: frame_idx, unix_time, alt, az, x_km, y_km, extinction, strategy_id
    """
    rows = []
    for strategy, sid in STRATEGY_IDS.items():
        fi_list   = metrics[strategy]["frame_indices"]
        xy_list   = metrics[strategy]["xy_km"]
        ext_list  = metrics[strategy]["extinctions"]
        for k in range(len(fi_list)):
            fi       = fi_list[k]
            x_km, y_km = xy_list[k]
            alt, az  = xy_to_altaz(x_km, y_km)
            t_unix   = float(all_metas[fi]["time"].timestamp()) \
                if hasattr(all_metas[fi]["time"], "timestamp") \
                else float(pd.Timestamp(all_metas[fi]["time"]).timestamp())
            ext      = ext_list[k]
            rows.append([float(fi), t_unix, alt, az, x_km, y_km,
                         float(ext), float(sid)])
    if not rows:
        return np.zeros((0, 8), dtype=TENSOR_DTYPE)
    return np.array(rows, dtype=TENSOR_DTYPE)


def save_pointing_tensor(tensor, output_path, night_key,
                         all_tensors_accumulator=None):
    """
    Save per-night pointing tensor as .npz.
    Also appends to all_tensors_accumulator list for cross-night aggregation.
    """
    np.savez_compressed(
        output_path,
        pointing_tensor=tensor,
        night_key=np.array([str(night_key)]),
        columns=np.array(["frame_idx", "unix_time", "alt_deg", "az_deg",
                          "x_km", "y_km", "extinction", "strategy_id"]),
        strategy_ids=np.array([0, 1, 2]),
        strategy_names=np.array(["absolute", "motion", "scheduler"]),
    )
    print(f"  Pointing tensor saved: {output_path}  "
          f"({tensor.shape[0]} rows × {tensor.shape[1]} cols)")
    if all_tensors_accumulator is not None:
        all_tensors_accumulator.append(tensor)


def save_all_nights_tensor(all_tensors, output_path):
    """Concatenate per-night tensors and save as one .npz."""
    if not all_tensors:
        print("  No tensors to aggregate.")
        return
    combined = np.concatenate(all_tensors, axis=0)
    np.savez_compressed(
        output_path,
        pointing_tensor=combined,
        columns=np.array(["frame_idx", "unix_time", "alt_deg", "az_deg",
                          "x_km", "y_km", "extinction", "strategy_id"]),
        strategy_ids=np.array([0, 1, 2]),
        strategy_names=np.array(["absolute", "motion", "scheduler"]),
    )
    print(f"\nAll-nights pointing tensor saved: {output_path}  "
          f"({combined.shape[0]} total rows)")


# ─────────────────────────────────────────────────────────────────────────────
# Metrics plot  (LEFT side only — no live-video subplot)
# ─────────────────────────────────────────────────────────────────────────────

def create_metrics_plot(metrics, output_file="metrics_jan25_2026.png"):
    print("\n" + "=" * 70)
    print("CREATING METRICS PLOT")
    print("=" * 70)

    strategies = ("absolute", "motion", "scheduler")
    colors     = ("green", "blue", "red")
    labels     = ("Absolute Min", "Motion Tracking", "Scheduler")

    fig = plt.figure(figsize=(22, 22))
    gs  = fig.add_gridspec(5, 3, hspace=0.52, wspace=0.34)

    # ── Row 0: Zenith | Slew | σ distributions ───────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    for s, c, l in zip(strategies, colors, labels):
        if metrics[s]["zenith_angles"]:
            ax.hist(metrics[s]["zenith_angles"], bins=20, alpha=0.6,
                    color=c, label=l, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Zenith Angle (°)"); ax.set_ylabel("Count")
    ax.set_title("Zenith Angle Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 1])
    for s, c, l in zip(strategies, colors, labels):
        slew = metrics[s]["slew_times"][1:]
        if slew:
            ax.hist(slew, bins=30, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Slew Time (s)"); ax.set_ylabel("Count")
    ax.set_title("Slew Time Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 2])
    for s, c, l in zip(strategies, colors, labels):
        sigs = [x for x in metrics[s]["sigma"] if not np.isnan(x)]
        if sigs:
            ax.hist(sigs, bins=30, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.axvline(MAX_SIGMA_MAG, color="k", linestyle="--", lw=1.5,
               label=f"Mask threshold ({MAX_SIGMA_MAG})")
    ax.set_xlabel("Pixel σ uncertainty (mag)"); ax.set_ylabel("Count")
    ax.set_title("Extinction Uncertainty Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # ── Row 1: Photon timeline (full width) ──────────────────────────────────
    ax = fig.add_subplot(gs[1, :])
    for s, c, l in zip(strategies, colors, labels):
        fi = metrics[s]["frame_indices"]
        ph = metrics[s]["photons"]
        ph_c = metrics[s].get("photons_atm_corrected", [])
        if fi:
            ax.plot(fi, ph, color=c, alpha=0.45, linewidth=1.2,
                    label=f"{l} (raw)")
            if len(ph_c) == len(fi) and np.any(np.isfinite(ph_c)):
                ax.plot(fi, ph_c, color=c, alpha=0.9, linewidth=1.8,
                        linestyle="--", label=f"{l} (atm-corrected)")
    ax.set_xlabel("Frame Number (Time →)")
    ax.set_ylabel("Slew-gated photons per slot")
    ax.set_title("Photon Collection Over the Night  "
                 "(solid=raw, dashed=atm+seeing corrected)", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

    # ── Row 2: Total photons (raw + corrected) | Median depth | Efficiency ───
    ax = fig.add_subplot(gs[2, 0])
    x_pos  = np.arange(len(strategies))
    width  = 0.35
    totals_raw = [np.sum(metrics[s]["photons"]) if metrics[s]["photons"] else 0
                  for s in strategies]
    totals_cor = [np.nansum(metrics[s].get("photons_atm_corrected", [np.nan]))
                  for s in strategies]
    bars1 = ax.bar(x_pos - width/2, totals_raw, width, color=colors,
                   alpha=0.55, edgecolor="black", linewidth=1.2, label="raw")
    bars2 = ax.bar(x_pos + width/2, totals_cor, width, color=colors,
                   alpha=0.90, edgecolor="black", linewidth=1.2,
                   hatch="//", label="atm-corrected")
    baseline = totals_raw[0] if totals_raw[0] > 0 else 1
    for i, t in enumerate(totals_raw):
        ax.text(i - width/2, t, f"{((t-baseline)/baseline)*100:+.1f}%",
                ha="center", va="bottom", fontsize=9, weight="bold")
    ax.set_xticks(x_pos); ax.set_xticklabels(labels)
    ax.set_ylabel("Total Photons")
    ax.set_title("Total Night Photon Collection", weight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
    ax.ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

    ax = fig.add_subplot(gs[2, 1])
    depths = []
    for s in strategies:
        mags = [photons_to_magnitude(p) for p in metrics[s]["photons"]]
        mags = [m for m in mags if m is not None and not np.isnan(m)]
        depths.append(np.median(mags) if mags else 0)
    bars = ax.bar(labels, depths, color=colors, alpha=0.7,
                  edgecolor="black", linewidth=1.5)
    for bar, val in zip(bars, depths):
        ax.text(bar.get_x() + bar.get_width()/2, val, f"{val:.2f}",
                ha="center", va="bottom", fontsize=10, weight="bold")
    ax.set_ylabel("Median 5σ Depth (mag)")
    ax.set_title("Survey Depth (slew-scaled)", weight="bold")
    ax.grid(alpha=0.3, axis="y"); ax.invert_yaxis()

    ax = fig.add_subplot(gs[2, 2])
    effs = []
    for s in strategies:
        et    = np.array(metrics[s]["expose_times"])
        slots = len(et) * SLOT_TIME
        effs.append(np.sum(et) / slots * 100 if slots > 0 else 0)
    bars = ax.bar(labels, effs, color=colors, alpha=0.7,
                  edgecolor="black", linewidth=1.5)
    for bar, val in zip(bars, effs):
        ax.text(bar.get_x() + bar.get_width()/2, val, f"{val:.1f}%",
                ha="center", va="bottom", fontsize=10, weight="bold")
    ax.set_ylabel("Shutter-open efficiency (%)")
    ax.set_title("Observation Efficiency", weight="bold")
    ax.grid(alpha=0.3, axis="y"); ax.set_ylim(0, 105)

    # ── Row 3: Correction inputs — seeing / eff_time_scale / airmass ─────────

    # [3,0] Seeing FWHM distribution
    ax = fig.add_subplot(gs[3, 0])
    for s, c, l in zip(strategies, colors, labels):
        sf = [v for v in metrics[s].get("seeing_fwhm", []) if np.isfinite(v)]
        if sf:
            ax.hist(sf, bins=25, alpha=0.60, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Seeing FWHM (arcsec)"); ax.set_ylabel("Count")
    ax.set_title("Seeing FWHM Distribution\n(ConsDB zenith 500 nm)", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # [3,1] eff_time_scale over the night — the actual multiplicative factor
    ax = fig.add_subplot(gs[3, 1])
    for s, c, l in zip(strategies, colors, labels):
        fi  = metrics[s]["frame_indices"]
        ets = metrics[s].get("eff_time_scale", [])
        if fi and len(ets) == len(fi):
            ets_clean = [v if np.isfinite(v) else np.nan for v in ets]
            ax.plot(fi, ets_clean, color=c, alpha=0.85, linewidth=1.4, label=l)
    ax.axhline(1.0, color="k", ls="--", lw=1.5, label="ideal (1.0)")
    ax.axhline(0.5, color="grey", ls=":", lw=1.0, alpha=0.6, label="50% degraded")
    ax.fill_between(range(max((len(metrics["absolute"]["frame_indices"]),
                               len(metrics["motion"]["frame_indices"]),
                               len(metrics["scheduler"]["frame_indices"])), default=1)),
                    0, 0.5, color="red", alpha=0.04)
    ax.set_xlabel("Frame Number (Time →)")
    ax.set_ylabel("eff_time_zero_point_scale")
    ax.set_title("Eff-Time Scale Factor Over Night\n"
                 "(< 1 = atm+seeing degraded)", weight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylim(bottom=0)

    # [3,2] Airmass over the night
    ax = fig.add_subplot(gs[3, 2])
    for s, c, l in zip(strategies, colors, labels):
        fi  = metrics[s]["frame_indices"]
        am  = metrics[s].get("airmass_consdb", [])
        if fi and len(am) == len(fi):
            am_f = [v if np.isfinite(v) else np.nan for v in am]
            ax.plot(fi, am_f, color=c, alpha=0.85, linewidth=1.4, label=l)
    ax.set_xlabel("Frame Number (Time →)"); ax.set_ylabel("Airmass")
    ax.set_title("Airmass Over Night\n(ConsDB — atmospheric path length)",
                 weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # ── Row 4: Before vs After correction ────────────────────────────────────

    # [4,0] Photon time-series: raw vs corrected overlaid for all strategies
    ax = fig.add_subplot(gs[4, 0])
    for s, c, l in zip(strategies, colors, labels):
        fi   = metrics[s]["frame_indices"]
        ph   = metrics[s]["photons"]
        ph_c = metrics[s].get("photons_atm_corrected", [])
        if not fi:
            continue
        ax.plot(fi, ph, color=c, lw=1.2, alpha=0.30,
                linestyle="-", label=f"{l} — raw")
        if len(ph_c) == len(fi) and np.any(np.isfinite(ph_c)):
            ax.plot(fi, ph_c, color=c, lw=1.9, alpha=0.90,
                    linestyle="--", label=f"{l} — corrected")
    ax.set_xlabel("Frame Number (Time →)")
    ax.set_ylabel("Photons / slot")
    ax.set_title("Raw vs Corrected Photon Collection\n"
                 "(solid=raw  dashed=× eff_time_scale)", weight="bold")
    ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

    # [4,1] Grouped bar: raw vs corrected total photons, % change annotated
    ax = fig.add_subplot(gs[4, 1])
    x_pos = np.arange(len(strategies))
    w     = 0.32
    t_raw = [float(np.nansum(metrics[s]["photons"]))           for s in strategies]
    t_cor = [float(np.nansum(metrics[s].get("photons_atm_corrected",
                                             [np.nan])))       for s in strategies]
    ax.bar(x_pos - w/2, t_raw, w, color=colors, alpha=0.42,
           edgecolor="black", lw=1.2, label="Raw")
    ax.bar(x_pos + w/2, t_cor, w, color=colors, alpha=0.92,
           edgecolor="black", lw=1.2, hatch="///", label="Atm-corrected")
    for i, (r, cv) in enumerate(zip(t_raw, t_cor)):
        if r > 0 and np.isfinite(cv):
            pct = (cv - r) / r * 100
            col = "#a00000" if pct < 0 else "#004d00"
            ax.text(i + w/2, cv, f"{pct:+.1f}%",
                    ha="center", va="bottom", fontsize=9,
                    weight="bold", color=col)
    ax.set_xticks(x_pos); ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Total Photons Night")
    ax.set_title("Total Photons: Before vs After Correction\n"
                 "(% = how much atm+seeing changed the count)", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

    # [4,2] Scatter: seeing FWHM vs correction ratio, coloured by eff_time_scale
    #   Diagnosis plot — reveals whether corrections cluster by seeing/airmass
    ax = fig.add_subplot(gs[4, 2])
    plotted = False
    for s, c, l in zip(strategies, colors, labels):
        ph   = np.array(metrics[s]["photons"],    dtype=float)
        ph_c = np.array(metrics[s].get("photons_atm_corrected",
                                        [np.nan]*len(ph)), dtype=float)
        sfwhm = np.array(metrics[s].get("seeing_fwhm",
                                         [np.nan]*len(ph)), dtype=float)
        ets   = np.array(metrics[s].get("eff_time_scale",
                                         [np.nan]*len(ph)), dtype=float)
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(ph > 0, ph_c / ph, np.nan)
        mask = np.isfinite(sfwhm) & np.isfinite(ratio) & np.isfinite(ets)
        if not np.any(mask):
            continue
        sc = ax.scatter(sfwhm[mask], ratio[mask],
                        c=ets[mask], cmap="RdYlGn",
                        vmin=0.3, vmax=1.5, s=18, alpha=0.65,
                        edgecolors=c, linewidths=0.5)
        plotted = True
    if plotted:
        plt.colorbar(ax.collections[0], ax=ax,
                     label="eff_time_scale", shrink=0.85)
    ax.axhline(1.0, color="k", ls="--", lw=1.4, label="ratio = 1 (no change)")
    ax.set_xlabel("Seeing FWHM (arcsec)")
    ax.set_ylabel("Corrected / Raw photon ratio")
    ax.set_title("Correction Ratio vs Seeing FWHM\n"
                 "(colour = eff_time_scale · red<1 = loss  green>1 = gain)",
                 weight="bold")
    ax.legend(fontsize=8, loc="upper right"); ax.grid(alpha=0.3)

    plt.suptitle(
        "Rubin Observatory Pointing Strategy Comparison\n"
        "(sys map  |  σ-masked  |  slew-gated  |  moon-avoided  |"
        "  zenith-weighted  |  atm+seeing corrected)",
        fontsize=12, weight="bold", y=0.999)
    plt.savefig(output_file, dpi=150, bbox_inches="tight")
    print(f"  Metrics plot saved: {output_file}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# Video  (map panel only)
# ─────────────────────────────────────────────────────────────────────────────

def create_video(all_grids, all_sigma_grids, all_metas,
                 abs_positions, pred_positions, motion_vectors, moon_positions,
                 matched_scheduler, x_edges, y_edges,
                 output_file="cloud_tracking.mp4", fps=10):
    print("\n" + "=" * 70)
    print("CREATING VIDEO (map panel only)")
    print("=" * 70)

    sched_xy = []
    for i, obs in enumerate(matched_scheduler):
        x, y, alt, az = radec_to_xy(
            float(obs["fieldRA"]), float(obs["fieldDec"]),
            all_metas[i]["time"])
        sched_xy.append((x, y, alt, az))

    fig, ax_map = plt.subplots(1, 1, figsize=(12, 10))

    def update(fi):
        ax_map.clear()
        grid  = all_grids[fi]
        sgrid = all_sigma_grids[fi]
        ts    = max(0, fi - 12)
        m_alt, m_az = moon_positions[fi]

        ax_map.pcolormesh(x_edges, y_edges, grid,
                          cmap="viridis", vmin=CBAR_VMIN, vmax=CBAR_VMAX,
                          shading="flat", alpha=0.85)
        th = np.linspace(0, 2*np.pi, 300)
        ax_map.plot(15000*np.cos(th), 15000*np.sin(th),
                    "k-", lw=2, alpha=0.5)
        ax_map.plot(0, 0, "r+", ms=15, mew=3, label="Zenith", zorder=5)

        # Moon exclusion circle (if moon is above horizon)
        if m_alt > 0:
            moon_x, moon_y = altaz_to_xy(m_alt, m_az)
            if moon_x is not None:
                # Draw exclusion circle in km (approximate)
                excl_r = MOON_EXCLUSION_DEG / 90.0 * R_PROJECTION
                ax_map.plot(moon_x + excl_r*np.cos(th),
                            moon_y + excl_r*np.sin(th),
                            "y--", lw=1.5, alpha=0.7, label=f"Moon excl. {MOON_EXCLUSION_DEG}°")
                ax_map.plot(moon_x, moon_y, "y*", ms=14, mew=1.5, mec="orange",
                            zorder=6, label=f"Moon alt={m_alt:.1f}°")

        # Green — absolute minimum
        xa, ya = abs_positions[fi]
        va = get_value_at_position(grid,  xa, ya)
        sa = get_value_at_position(sgrid, xa, ya)
        gx = [abs_positions[j][0] for j in range(ts, fi)]
        gy = [abs_positions[j][1] for j in range(ts, fi)]
        if len(gx) > 1:
            ax_map.plot(gx, gy, "g--", alpha=0.35, lw=1.5)
        ax_map.plot(xa, ya, "go", ms=20, mew=2.5, mec="white",
                    label=f"Abs Min  ext={va:.3f} σ={sa:.3f}", zorder=10)

        # Blue — motion tracking
        xp, yp = pred_positions[fi]
        vp = get_value_at_position(grid,  xp, yp)
        sp = get_value_at_position(sgrid, xp, yp)
        bx = [pred_positions[j][0] for j in range(ts, fi)]
        by = [pred_positions[j][1] for j in range(ts, fi)]
        if len(bx) > 1:
            ax_map.plot(bx, by, "b--", alpha=0.35, lw=1.5)
        ax_map.plot(xp, yp, "bo", ms=20, mew=2.5, mec="white",
                    label=f"Motion   ext={vp:.3f} σ={sp:.3f}", zorder=10)

        # Motion arrow
        if fi > 0:
            dx_pix, dy_pix = motion_vectors[fi]
            dxk = dx_pix * BIN_SIZE_KM
            dyk = dy_pix * BIN_SIZE_KM
            mag = np.sqrt(dxk**2 + dyk**2)
            if mag > 300:
                ax_map.add_patch(FancyArrowPatch(
                    (xp, yp), (xp+dxk, yp+dyk),
                    arrowstyle="->", mutation_scale=28, lw=2.5,
                    color="cyan", alpha=0.9, zorder=15))
                ax_map.text(xp+dxk+400, yp+dyk+400, f"{mag:.0f} km",
                            fontsize=9, color="cyan", weight="bold",
                            bbox=dict(boxstyle="round", fc="black", alpha=0.6))

        # Red — scheduler
        xs, ys, alt_s, _ = sched_xy[fi]
        if xs is not None:
            vs = get_value_at_position(grid,  xs, ys)
            ss = get_value_at_position(sgrid, xs, ys)
            ax_map.plot(xs, ys, "ro", ms=20, mew=2.5, mec="white",
                        label=f"Scheduler ext={vs:.3f} σ={ss:.3f}", zorder=10)
        else:
            ax_map.plot([], [], "ro", ms=20, mew=2.5, mec="white", alpha=0.3,
                        label=f"Scheduler (below horizon, alt={alt_s:.1f}°)")

        def _fmt(v, ref):
            return f"{v:.3f} ({v-ref:+.3f})" if not np.isnan(v) else "N/A"

        txt = (f"EXTINCTION (sys, σ-masked)\n"
               f" Abs Min:  {va:.3f} mag  σ={sa:.3f}  [baseline]\n"
               f" Motion:   {_fmt(vp, va)}  σ={sp:.3f}\n"
               f" Sched:    {_fmt(vs if xs is not None else np.nan, va)}"
               f"  σ={ss:.3f}" if xs is not None else
               f"  Sched: below horizon")
        ax_map.text(0.02, 0.98, txt, transform=ax_map.transAxes,
                    fontsize=9.5, va="top", family="monospace", zorder=20,
                    bbox=dict(boxstyle="round", fc="white",
                              alpha=0.92, ec="black", lw=1.5))

        ax_map.set_xlabel("X (km)"); ax_map.set_ylabel("Y (km)")
        ax_map.set_aspect("equal"); ax_map.grid(alpha=0.2)
        ax_map.set_xlim(-16000, 16000); ax_map.set_ylim(-16000, 16000)
        ts_str = all_metas[fi]["time"].strftime("%Y-%m-%d %H:%M:%S UTC")
        ax_map.set_title(f"Cloud Coverage — {ts_str}\nFrame {fi+1}/{len(all_grids)}",
                         fontsize=12, weight="bold")
        ax_map.legend(loc="upper right", fontsize=8.5, framealpha=0.9)
        return []

    anim = animation.FuncAnimation(
        fig, update, frames=len(all_grids),
        interval=1000/fps, blit=False, repeat=False)
    print(f"Saving {output_file}  ({len(all_grids)} frames @ {fps} fps) ...")
    Writer = animation.writers["ffmpeg"]
    writer = Writer(fps=fps, bitrate=5000, codec="libx264",
                    extra_args=["-pix_fmt", "yuv420p"])
    anim.save(output_file, writer=writer)
    print(f"  Video saved: {output_file}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# ZP cross-validation
# ─────────────────────────────────────────────────────────────────────────────

def _safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def zps_value_at_radec(clouds, sigma_map, ra_deg, dec_deg, obstime):
    t   = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=loc))
    if aa.alt.deg < MIN_ALT_DEG:
        return np.nan, np.nan

    npix  = hp.nside2npix(NSIDE_EXPECTED)
    pix   = np.arange(npix)
    th, ph = hp.pix2ang(NSIDE_EXPECTED, pix, nest=NEST)
    all_sky = SkyCoord(ra=np.degrees(ph)*u.deg,
                       dec=(90-np.degrees(th))*u.deg, frame="icrs")
    all_aa  = all_sky.transform_to(AltAz(obstime=t, location=loc))

    v_alt = np.radians(aa.alt.deg)
    v_az  = np.radians(aa.az.deg % 360)
    ar    = np.radians(all_aa.alt.deg)
    azr   = np.radians(all_aa.az.deg % 360)
    cos_sep = (np.sin(v_alt)*np.sin(ar)
               + np.cos(v_alt)*np.cos(ar)*np.cos(v_az - azr))

    valid = ~np.isnan(clouds) & (all_aa.alt.deg >= MIN_ALT_DEG)
    if not np.any(valid):
        return np.nan, np.nan
    best = np.nanargmax(np.where(valid, cos_sep, -np.inf))
    return float(clouds[best]), float(sigma_map[best])


def build_consdb_df(start_night, instrument=INSTRUMENT, consdb_url=CONSDB_URL,
                    h5_file=None):
    """
    Build ConsDB DataFrame from live API or from a pre-saved HDF5 file.
    Pass h5_file="visits_20261024_20260219.h5" to load from disk instead.
    """
    print("\n" + "=" * 70)
    print("BUILDING ConsDB DATAFRAME")
    print("=" * 70)

    if h5_file and os.path.exists(h5_file):
        print(f"  Loading ConsDB visits from {h5_file} …")
        df = pd.read_hdf(h5_file, key="visits")
        # Normalise column names (may vary between dumps)
        df.columns = df.columns.str.strip()
        print(f"  Loaded {len(df)} rows, {len(df.columns)} columns")
        # Ensure required derived columns exist
        if "obs_midpoint" not in df.columns:
            for sc, ec in [("obs_start", "obs_end"),
                           ("obs_start_mjd", "obs_end_mjd")]:
                if sc in df.columns and ec in df.columns:
                    t0 = pd.to_datetime(df[sc], utc=True, errors="coerce")
                    t1 = pd.to_datetime(df[ec], utc=True, errors="coerce")
                    df["obs_midpoint"] = t0 + (t1 - t0) / 2
                    break
        # Map column aliases
        for want, cands in {
            "ra":  ["s_ra",  "ra_v1",  "ra_ql",  "ra"],
            "dec": ["s_dec", "dec_v1", "dec_ql", "dec"],
        }.items():
            if want not in df.columns:
                for c in cands:
                    if c in df.columns:
                        df[want] = df[c]; break
        if "lambda_eff" not in df.columns and "band" in df.columns:
            df["lambda_eff"] = df["band"].map(EFFECTIVE_WAVELENGTHS)
        if "fwhm_observed" not in df.columns and "psf_sigma_median" in df.columns:
            df["fwhm_observed"] = FWHM_CONSTANT * pd.to_numeric(
                df["psf_sigma_median"], errors="coerce")
        if "fwhm_zenith" not in df.columns and "fwhm_observed" in df.columns \
                and "airmass" in df.columns:
            df["fwhm_zenith"] = (pd.to_numeric(df["fwhm_observed"], errors="coerce")
                                 / pd.to_numeric(df["airmass"], errors="coerce") ** ALPHA_SEEING)
        if "seeing" not in df.columns and "fwhm_zenith" in df.columns \
                and "lambda_eff" in df.columns:
            df["seeing"] = (pd.to_numeric(df["fwhm_zenith"], errors="coerce")
                            * (pd.to_numeric(df["lambda_eff"], errors="coerce") / 500.0)
                            ** BETA_SEEING)
        df["zp_consdb"] = pd.to_numeric(
            df.get("zero_point_median", pd.Series(np.nan, index=df.index)),
            errors="coerce")
        df = df[df["ra"].notna() & df["dec"].notna()].reset_index(drop=True)
        print(f"  Final: {len(df)} rows, zp_consdb non-null: {df['zp_consdb'].notna().sum()}")
        return df

    # ── Live API path ─────────────────────────────────────────────────────────
    client = ConsDbClient(consdb_url)
    t0     = pd.to_datetime(start_night)
    nights = np.arange(int(t0.strftime("%Y%m%d")),
                       int(pd.Timestamp.now().strftime("%Y%m%d")))
    dfs    = []
    zp_col = None

    for night in nights:
        try:
            v1 = client.query(
                f"SELECT * FROM cdb_{instrument}.visit1 "
                f"WHERE day_obs = {night}"
            ).to_pandas()
            ql = client.query(
                f"SELECT * FROM cdb_{instrument}.visit1_quicklook "
                f"WHERE day_obs = {night}"
            ).to_pandas()
            if ql.empty or v1.empty:
                continue
            merged = pd.merge(ql, v1, on="visit_id", how="inner",
                              suffixes=("_ql", "_v1"))
            merged = merged[merged["airmass"].notnull()
                            & (merged["airmass"] != 0.0)].copy()
            if merged.empty:
                continue
            if zp_col is None:
                candidates = [c for c in merged.columns
                              if any(k in c.lower() for k in
                                     ["zeropoint","zero_point","zp","fluxmag0",
                                      "flux_mag0","photometric","calib"])]
                for name in ["zeroPoint","zero_point","zp_mean","zeropoint",
                             "flux_mag0","fluxMag0"]:
                    if name in merged.columns:
                        zp_col = name; break
                if zp_col is None and candidates:
                    zp_col = candidates[0]
                print(f"  ZP column: '{zp_col}'")

            actual_zp = None
            if zp_col:
                for sfx in ("", "_ql", "_v1"):
                    if zp_col + sfx in merged.columns:
                        actual_zp = zp_col + sfx; break
            merged["zp_consdb"] = (pd.to_numeric(merged[actual_zp], errors="coerce")
                                   if actual_zp else np.nan)
            merged["lambda_eff"]    = merged["band"].map(EFFECTIVE_WAVELENGTHS)
            merged["fwhm_observed"] = FWHM_CONSTANT * merged["psf_sigma_median"]
            merged["fwhm_zenith"]   = merged["fwhm_observed"] / (merged["airmass"] ** ALPHA_SEEING)
            merged["seeing"]        = (merged["fwhm_zenith"]
                                       * (merged["lambda_eff"] / 500.0) ** BETA_SEEING)
            merged["obs_start"]    = pd.to_datetime(merged["obs_start"],
                                                     format="ISO8601", utc=True,
                                                     errors="coerce")
            merged["obs_end"]      = pd.to_datetime(merged["obs_end"],
                                                     format="ISO8601", utc=True,
                                                     errors="coerce")
            merged["obs_midpoint"] = (merged["obs_start"]
                                      + (merged["obs_end"] - merged["obs_start"]) / 2)
            for want, cands in {
                "ra":  ["s_ra",  "ra_v1",  "ra_ql",  "ra"],
                "dec": ["s_dec", "dec_v1", "dec_ql", "dec"],
            }.items():
                if want not in merged.columns:
                    for c in cands:
                        if c in merged.columns:
                            merged[want] = merged[c]; break
            dfs.append(merged)
            print(f"  night {night}: {len(merged)} visits")
        except Exception as e:
            print(f"  night {night}: skipped ({e})")

    if not dfs:
        print("No ConsDB data found.")
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df = df[df["ra"].notna() & df["dec"].notna()].reset_index(drop=True)
    df["zp_consdb"] = pd.to_numeric(df["zp_consdb"], errors="coerce")
    print(f"ConsDB total: {len(df)} visits, "
          f"zp_consdb non-null: {df['zp_consdb'].notna().sum()}")
    return df


def cross_validate_zp(dream_zps_df, consdb_df,
                       n_samples=N_ZP_SAMPLES, seed=ZP_SEED):
    print("\n" + "=" * 70)
    print("ZP CROSS-VALIDATION (DREAM .zps vs ConsDB)")
    print("=" * 70)
    if consdb_df.empty:
        print("No ConsDB data — skipping cross-validation.")
        return pd.DataFrame()

    d_min, d_max = dream_zps_df[TIME_COL].min(), dream_zps_df[TIME_COL].max()
    c_min, c_max = consdb_df["obs_midpoint"].min(), consdb_df["obs_midpoint"].max()
    ov_start = max(d_min, c_min)
    ov_end   = min(d_max, c_max)
    if ov_start >= ov_end:
        print("No temporal overlap between .zps frames and ConsDB.")
        return pd.DataFrame()

    dream_ov  = dream_zps_df[(dream_zps_df[TIME_COL] >= ov_start)
                              & (dream_zps_df[TIME_COL] <= ov_end)].reset_index(drop=True)
    consdb_ov = consdb_df[(consdb_df["obs_midpoint"] >= ov_start)
                           & (consdb_df["obs_midpoint"] <= ov_end)].reset_index(drop=True)
    print(f"DREAM .zps in overlap: {len(dream_ov)},  ConsDB: {len(consdb_ov)}")
    if dream_ov.empty or consdb_ov.empty:
        return pd.DataFrame()

    rng    = np.random.default_rng(seed)
    idx    = rng.choice(len(dream_ov), size=min(n_samples, len(dream_ov)), replace=False)
    sample = dream_ov.iloc[sorted(idx)].reset_index(drop=True)

    cadence = max(
        (dream_ov[TIME_COL].max() - dream_ov[TIME_COL].min()).total_seconds()
        / max(len(dream_ov), 1), 300.0)
    window = pd.Timedelta(seconds=cadence / 2)
    print(f"Match window: ±{int(window.total_seconds())}s\n")

    rows = []
    for _, frame in sample.iterrows():
        t_frame   = frame[TIME_COL]
        url       = transform_url(frame[URL_COL])
        dbnd      = frame.get("dream_band")
        try:
            clouds, sigma_map = fetch_zps_map(url)
        except Exception as e:
            print(f"   load failed: {e}")
            continue

        nearby = consdb_ov[
            (consdb_ov["obs_midpoint"] >= t_frame - window)
            & (consdb_ov["obs_midpoint"] <= t_frame + window)
        ].copy()
        lsst_band = FILENAME_BAND_MAP.get(dbnd)
        if lsst_band:
            nearby = nearby[nearby["band"] == lsst_band]
        print(f"  {t_frame.strftime('%H:%M:%S')} dream_band={dbnd}  "
              f"consdb matches={len(nearby)}")

        for _, visit in nearby.iterrows():
            ra  = _safe_float(visit.get("ra"))
            dec = _safe_float(visit.get("dec"))
            if np.isnan(ra) or np.isnan(dec):
                continue
            zp_d, sig_d = zps_value_at_radec(clouds, sigma_map, ra, dec, t_frame)
            zp_c        = _safe_float(visit.get("zp_consdb", np.nan))
            airmass     = _safe_float(visit.get("airmass",   np.nan))
            seeing      = _safe_float(visit.get("seeing",    np.nan))
            band        = str(visit.get("band", "?"))
            # (c) attach quicklook quality columns to the ZP row
            eff_time_med = _safe_float(visit.get("eff_time_median",                    np.nan))
            eft_zp_scale = _safe_float(visit.get("eff_time_zero_point_scale_median",   np.nan))
            psf_sig_med  = _safe_float(visit.get("psf_sigma_median",                   np.nan))
            sz500        = _safe_float(visit.get("seeing_zenith_500nm_median",          np.nan))
            zp_med       = _safe_float(visit.get("zero_point_median",                  np.nan))
            sky_bg       = _safe_float(visit.get("sky_bg_median",                      np.nan))
            delta = (zp_d - zp_c
                     if not (np.isnan(zp_d) or np.isnan(zp_c)) else np.nan)
            rows.append(dict(
                frame_time=t_frame, visit_id=visit.get("visit_id"),
                band=band, dream_band=dbnd, ra=ra, dec=dec,
                airmass=airmass, seeing=seeing,
                zp_offset_dream=zp_d, sig_dream=sig_d,
                zp_consdb=zp_c, delta=delta,
                eff_time_median=eff_time_med,
                eff_time_zero_point_scale_median=eft_zp_scale,
                psf_sigma_median=psf_sig_med,
                seeing_zenith_500nm_median=sz500,
                zero_point_median=zp_med,
                sky_bg_median=sky_bg,
            ))
    return pd.DataFrame(rows)


def plot_zp_comparison(df, output="dream_zps_vs_consdb.png"):
    print("\n" + "=" * 70)
    print("CREATING ZP COMPARISON PLOT")
    print("=" * 70)
    if df.empty:
        print("No data to plot.")
        return

    for col in ["zp_offset_dream", "sig_dream", "zp_consdb", "delta",
                "airmass", "seeing", "eff_time_zero_point_scale_median",
                "seeing_zenith_500nm_median"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    valid = df.dropna(subset=["zp_offset_dream"])
    if valid.empty:
        print("No valid DREAM ZP offsets.")
        return

    bands  = sorted(valid["band"].dropna().unique())
    cmap   = plt.cm.get_cmap("tab10", max(len(bands), 1))
    bc     = {b: cmap(i) for i, b in enumerate(bands)}
    has_cdb = valid["zp_consdb"].notna().any()
    ncols   = 4 if has_cdb else 3

    fig, axes = plt.subplots(1, ncols, figsize=(7*ncols, 5))
    fig.suptitle("DREAM .zps ZP Offset vs ConsDB", fontsize=13, weight="bold")

    # Panel 1: ZP offset vs airmass
    ax = axes[0]
    for b in bands:
        s = valid[valid["band"] == b].dropna(subset=["airmass"])
        ax.errorbar(s["airmass"], s["zp_offset_dream"],
                    yerr=s["sig_dream"], fmt="o", ms=6,
                    color=bc[b], label=b, alpha=0.8, capsize=3)
    ax.axhline(0, color="k", ls="--", lw=1.5, label="zero offset")
    ax.set_xlabel("Airmass"); ax.set_ylabel("DREAM ZP offset (mag)")
    ax.set_title("ZP offset vs Airmass\n(0 = photometric)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 2: ZP offset timeline
    ax = axes[1]
    for b in bands:
        s = valid[valid["band"] == b]
        ax.scatter(s["frame_time"], s["zp_offset_dream"],
                   color=bc[b], s=40, alpha=0.9, label=b)
    ax.axhline(0, color="k", ls="--", lw=1.5)
    ax.set_ylabel("DREAM ZP offset (mag)")
    ax.set_title("ZP offset over night")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 3: ZP offset vs seeing_zenith_500nm
    ax = axes[2]
    see_col = "seeing_zenith_500nm_median" if "seeing_zenith_500nm_median" in valid.columns \
        else "seeing"
    for b in bands:
        s = valid[valid["band"] == b].dropna(subset=[see_col])
        if s.empty:
            continue
        ax.scatter(s[see_col], s["zp_offset_dream"],
                   color=bc[b], s=40, alpha=0.9, label=b)
    ax.axhline(0, color="k", ls="--", lw=1.5)
    ax.set_xlabel("Seeing zenith 500nm (arcsec)"); ax.set_ylabel("DREAM ZP offset (mag)")
    ax.set_title("ZP offset vs Seeing")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 4: DREAM vs ConsDB scatter (if data available)
    if has_cdb:
        valid2 = valid.dropna(subset=["zp_consdb"])
        ax = axes[3]
        for b in bands:
            s = valid2[valid2["band"] == b]
            if s.empty:
                continue
            ax.errorbar(s["zp_consdb"], s["zp_offset_dream"],
                        yerr=s["sig_dream"], fmt="o", ms=6,
                        color=bc[b], label=b, alpha=0.8, capsize=3)
        if len(valid2) >= 2:
            sl, ic, r, *_ = stats.linregress(
                valid2["zp_consdb"].astype(float),
                valid2["zp_offset_dream"].astype(float))
            xs = np.linspace(valid2["zp_consdb"].min(),
                             valid2["zp_consdb"].max(), 100)
            ax.plot(xs, sl*xs+ic, "r-", lw=1.5,
                    label=f"slope={sl:.2f}  r={r:.2f}")
        ax.set_xlabel("ConsDB ZP"); ax.set_ylabel("DREAM ZP offset (mag)")
        ax.set_title("DREAM offset vs ConsDB ZP")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(output, dpi=150, bbox_inches="tight")
    print(f"  ZP comparison plot saved: {output}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# MAIN per-night runner
# ─────────────────────────────────────────────────────────────────────────────

def run_night(night_key, df_sys_night, df_zps_night,
              scheduler_obs, consdb_df,
              output_dir=".", fps=10, max_frames=None,
              all_tensors_accumulator=None):
    night_str  = str(night_key)
    tag        = night_str.replace("-", "")
    plot_file   = os.path.join(output_dir, f"metrics_{tag}.png")
    video_file  = os.path.join(output_dir, f"cloud_tracking_{tag}.mp4")
    zp_file     = os.path.join(output_dir, f"dream_zps_vs_consdb_{tag}.png")
    tensor_file = os.path.join(output_dir, f"pointing_tensor_{tag}.npz")  # (d)

    print("\n" + "█" * 70)
    print(f"  NIGHT: {night_str}  ({len(df_sys_night)} sys frames)")
    print("█" * 70)

    if len(df_sys_night) < 5:
        print("  Too few frames — skipping.")
        return None

    print(f"\n  [Patchiness probe — {PATCHINESS_SAMPLE_N} frames]")
    patchy, mean_frac, mean_var, reason, _ = quick_patchiness_check(df_sys_night)
    print(f"    valid_frac={mean_frac:.3f}  spatial_var={mean_var:.4f} mag²")
    print(f"    → {reason}")
    if not patchy:
        print("  Skipping night (not patchy).")
        return None

    print("  Patchy clouds confirmed — loading all frames …")

    all_grids  = []
    all_sigma  = []
    all_metas  = []
    x_edges = y_edges = None

    rows = (df_sys_night.iloc[:max_frames] if max_frames else df_sys_night).iterrows()
    for _, row in rows:
        url = transform_url(row[URL_COL])
        try:
            clouds, sigma = fetch_sys_map(url)
            meta = {"url": url, "time": row[TIME_COL].to_pydatetime()}
            ext_g, sig_g, _, _, xe, ye = process_to_grid(clouds, sigma, meta["time"])
            all_grids.append(ext_g)
            all_sigma.append(sig_g)
            all_metas.append(meta)
            if x_edges is None:
                x_edges, y_edges = xe, ye
            if len(all_grids) % 100 == 0:
                print(f"    Loaded {len(all_grids)} frames …")
        except Exception as e:
            print(f"    Frame load failed: {e}")

    print(f"  Loaded {len(all_grids)} frames total")
    if len(all_grids) < 2:
        print("  Too few frames loaded — skipping.")
        return None

    patchy2, mf2, mv2, reason2 = compute_patchiness(all_grids)
    print(f"  Full-set patchiness: valid_frac={mf2:.3f}  var={mv2:.4f}  → {reason2}")
    if not patchy2:
        print("  Full-set check failed — skipping.")
        return None

    matched_sched = match_scheduler_to_frames(
        scheduler_obs, [m["time"] for m in all_metas])

    print(f"\n  [Cloud motion tracking — {len(all_grids)} frames]")
    # (a)+(b) moon positions computed inside compute_all_positions
    abs_pos, pred_pos, motion_v, moon_pos = compute_all_positions(
        all_grids, all_metas)

    metrics = compute_metrics(
        all_grids, all_sigma, all_metas,
        abs_pos, pred_pos, moon_pos, matched_sched)
    print_statistics(metrics)

    # (c) Atmospheric + seeing correction
    metrics = apply_atm_seeing_correction(metrics, all_metas, consdb_df)
    print_corrected_statistics(metrics)

    create_metrics_plot(metrics, output_file=plot_file)

    create_video(all_grids, all_sigma, all_metas,
                 abs_pos, pred_pos, motion_v, moon_pos,
                 matched_sched, x_edges, y_edges,
                 output_file=video_file, fps=fps)

    # ── (d) Save pointing tensor ─────────────────────────────────────────────
    tensor = build_pointing_tensor(metrics, all_metas)
    save_pointing_tensor(tensor, tensor_file, night_key,
                         all_tensors_accumulator=all_tensors_accumulator)

    # ── ZP cross-validation ──────────────────────────────────────────────────
    if not df_zps_night.empty and not consdb_df.empty:
        print(f"\n  [ZP cross-validation — {len(df_zps_night)} zps frames]")
        comparison = cross_validate_zp(df_zps_night, consdb_df)
        if not comparison.empty:
            for col in ["zp_offset_dream", "sig_dream", "zp_consdb", "delta", "airmass"]:
                if col in comparison.columns:
                    comparison[col] = pd.to_numeric(comparison[col], errors="coerce")
            valid_zp = comparison.dropna(subset=["zp_offset_dream"])
            print(f"  Mean DREAM ZP offset : {valid_zp['zp_offset_dream'].mean():.4f} mag")
            print(f"  Std  DREAM ZP offset : {valid_zp['zp_offset_dream'].std():.4f} mag")
            valid2 = comparison.dropna(subset=["zp_offset_dream", "zp_consdb"])
            if len(valid2) >= 2:
                r, p = stats.pearsonr(
                    valid2["zp_offset_dream"].astype(float),
                    valid2["zp_consdb"].astype(float))
                print(f"  Pearson r : {r:.3f}  (p={p:.4f})")
            plot_zp_comparison(comparison, output=zp_file)
        else:
            print("  No matched ZP pairs found.")
    else:
        print("  Skipping ZP validation (no zps frames or no ConsDB data).")

    print(f"\n  ✓ Night {night_str} outputs:")
    print(f"      {plot_file}")
    print(f"      {video_file}")
    print(f"      {zp_file}")
    print(f"      {tensor_file}")
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main(
    csv_file     = "feb5_data.csv",
    db_file      = "baseline_v5.1.0_10yrs.db",
    consdb_start = "2025-05-01",
    consdb_h5    = None,          # e.g. "visits_20261024_20260219.h5"
    output_dir   = ".",
    fps          = 10,
    max_frames   = None,
):
    """
    Iterate over every distinct night present in the DREAM CSV.
    For each night:
      - Quick patchiness probe (only ~10 frames fetched)
      - If not patchy → print reason and move on
      - If patchy     → full analysis (tracking, metrics, video, ZP, tensor)
    Scheduler observations and ConsDB data are loaded once and reused.

    New arguments:
      consdb_h5 : path to pre-saved HDF5 visits file; if provided, loaded
                  instead of querying the live ConsDB API.
    """
    print("\n" + "=" * 70)
    print("DREAM CLOUD ANALYSIS PIPELINE  —  ALL NIGHTS")
    print("=" * 70)
    print(f"  CSV          : {csv_file}")
    print(f"  Scheduler DB : {db_file}")
    print(f"  Output dir   : {output_dir}")
    print(f"  Moon excl.   : {MOON_EXCLUSION_DEG}°")
    print(f"  Zenith wt.   : {W_ZENITH}  Slew wt.: {W_SLEW}  Cloud wt.: {W_CLOUD}")
    os.makedirs(output_dir, exist_ok=True)

    print("\nIndexing CSV …")
    all_sys = load_all_sys_frames(csv_file)
    all_zps = load_all_zps_frames(csv_file)

    all_nights = sorted(all_sys["night_key"].unique())
    print(f"Found {len(all_nights)} distinct nights in CSV:")
    for nk in all_nights:
        n = (all_sys["night_key"] == nk).sum()
        print(f"  {nk}  ({n} sys frames)")

    scheduler_obs, _ = load_scheduler_observations(db_file)

    print("\nLoading ConsDB …")
    consdb_df = build_consdb_df(consdb_start, h5_file=consdb_h5)

    summary              = []
    all_tensors          = []   # (d) accumulator for cross-night tensor
    nights_checked       = 0    # total nights with enough frames to probe
    nights_patchy        = 0    # nights that passed patchiness and ran fully

    for night_key in all_nights:
        df_sys_night = get_night_df(all_sys, night_key)
        df_zps_night = get_night_df(all_zps, night_key)

        if len(df_sys_night) >= 5:
            nights_checked += 1

        metrics = run_night(
            night_key, df_sys_night, df_zps_night,
            scheduler_obs, consdb_df,
            output_dir=output_dir, fps=fps, max_frames=max_frames,
            all_tensors_accumulator=all_tensors)

        if metrics is not None:
            nights_patchy += 1
            row = {"night": str(night_key)}
            for s in ("absolute", "motion", "scheduler"):
                ph   = np.array(metrics[s]["photons"])
                ph_c = np.array(metrics[s].get("photons_atm_corrected", [np.nan]))
                row[f"total_photons_{s}"]          = float(np.sum(ph))   if len(ph)   else 0.0
                row[f"total_photons_corr_{s}"]     = float(np.nansum(ph_c)) if len(ph_c) else 0.0
                row[f"n_slots_{s}"]                = len(ph)
                row[f"mean_seeing_{s}"]            = float(np.nanmean(
                    metrics[s].get("seeing_fwhm", [np.nan])))
                row[f"mean_eff_time_scale_{s}"]    = float(np.nanmean(
                    metrics[s].get("eff_time_scale", [np.nan])))
            summary.append(row)

    # (d) All-nights combined tensor
    all_tensor_file = os.path.join(output_dir, "pointing_tensor_all_nights.npz")
    save_all_nights_tensor(all_tensors, all_tensor_file)

    print("\n" + "=" * 70)
    print("ALL-NIGHTS SUMMARY")
    print("=" * 70)

    # Patchiness overview — always printed regardless of whether any ran
    n_total_csv = len(all_nights)
    n_skipped_frames = n_total_csv - nights_checked
    n_skipped_uniform = nights_checked - nights_patchy
    print(f"\n{'PATCHINESS OVERVIEW':}")
    print(f"  Total nights in CSV          : {n_total_csv}")
    print(f"  Skipped (< 5 frames)         : {n_skipped_frames}")
    print(f"  Probed for patchiness        : {nights_checked}")
    print(f"  Skipped (not patchy / clear) : {n_skipped_uniform}")
    print(f"  Patchy — full analysis run   : {nights_patchy}  "
          f"({100*nights_patchy/max(nights_checked,1):.1f}% of probed nights)")

    if summary:
        df_sum = pd.DataFrame(summary)
        print(df_sum.to_string(index=False))
        sum_file = os.path.join(output_dir, "all_nights_summary.csv")
        df_sum.to_csv(sum_file, index=False)
        print(f"\nSummary saved → {sum_file}")

        # Patchiness log — one row per night in CSV, patchy flag included
        patch_rows = []
        for nk in all_nights:
            n_frames = int((all_sys["night_key"] == nk).sum())
            was_patchy = str(nk) in df_sum["night"].values
            patch_rows.append({"night": str(nk),
                                "n_sys_frames": n_frames,
                                "enough_frames": n_frames >= 5,
                                "patchy": was_patchy})
        patch_df = pd.DataFrame(patch_rows)
        patch_file = os.path.join(output_dir, "patchiness_log.csv")
        patch_df.to_csv(patch_file, index=False)
        print(f"Patchiness log saved → {patch_file}")
        print(f"\n{patch_df[['night','n_sys_frames','patchy']].to_string(index=False)}")
    else:
        print("No patchy nights found in this CSV.")

    print("\nDone.")
    return summary


if __name__ == "__main__":
    # ── Jupyter-safe entry point ──────────────────────────────────────────────
    # Running this cell in a notebook calls main() with defaults below.
    # Edit the keyword arguments here to configure your run, or import and
    # call main() directly from another cell:
    #
    #   from dream_pipeline import main
    #   main(csv_file="feb5_data.csv", consdb_h5="visits_....h5", fps=5)
    #
    import sys as _sys
    _is_jupyter = hasattr(_sys, "ps1") or "ipykernel" in _sys.modules or \
                  any("jupyter" in a or "ipykernel" in a for a in _sys.argv)
    if _is_jupyter:
        main(
            csv_file    = "feb5_data.csv",
            db_file     = "baseline_v5.1.0_10yrs.db",
            consdb_start= "2025-05-01",
            consdb_h5   = None,       # e.g. "visits_20261024_20260219.h5"
            output_dir  = ".",
            fps         = 10,
            max_frames  = None,
        )
    else:
        # CLI — wire up argparse only when genuinely running as a script
        import argparse as _ap
        p = _ap.ArgumentParser(description="DREAM cloud analysis pipeline")
        p.add_argument("--csv_file",     default="feb5_data.csv")
        p.add_argument("--db_file",      default="baseline_v5.1.0_10yrs.db")
        p.add_argument("--consdb_start", default="2025-05-01")
        p.add_argument("--consdb_h5",    default=None)
        p.add_argument("--output_dir",   default=".")
        p.add_argument("--fps",          type=int,   default=10)
        p.add_argument("--max_frames",   type=int,   default=None)
        # parse_known_args is safe even inside Jupyter if someone does
        # `!python dream_pipeline.py --csv_file ...`
        args, _ = p.parse_known_args()
        main(
            csv_file    = args.csv_file,
            db_file     = args.db_file,
            consdb_start= args.consdb_start,
            consdb_h5   = args.consdb_h5,
            output_dir  = args.output_dir,
            fps         = args.fps,
            max_frames  = args.max_frames,
        )


DREAM CLOUD ANALYSIS PIPELINE  —  ALL NIGHTS
  CSV          : feb5_data.csv
  Scheduler DB : baseline_v5.1.0_10yrs.db
  Output dir   : .
  Moon excl.   : 30.0°
  Zenith wt.   : 0.25  Slew wt.: 0.25  Cloud wt.: 0.5

Indexing CSV …
Found 189 distinct nights in CSV:
  2025-06-25  (432 sys frames)
  2025-06-26  (1536 sys frames)
  2025-06-27  (1248 sys frames)
  2025-06-28  (1559 sys frames)
  2025-06-29  (1583 sys frames)
  2025-06-30  (1396 sys frames)
  2025-07-01  (1377 sys frames)
  2025-07-02  (1376 sys frames)
  2025-07-03  (1375 sys frames)
  2025-07-04  (1375 sys frames)
  2025-07-05  (495 sys frames)
  2025-07-06  (1371 sys frames)
  2025-07-07  (1371 sys frames)
  2025-07-08  (942 sys frames)
  2025-07-09  (1152 sys frames)
  2025-07-10  (1369 sys frames)
  2025-07-11  (1367 sys frames)
  2025-07-12  (1375 sys frames)
  2025-07-13  (1060 sys frames)
  2025-07-14  (1304 sys frames)
  2025-07-15  (1361 sys frames)
  2025-07-16  (1360 sys frames)
  2025-07-17  (1359 sys frames)
  

In [2]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, glob, os

# ── Load all per-night summary rows ──────────────────────────────────────
rows = []
for f in sorted(glob.glob("metrics_*.csv")):
    rows.append(pd.read_csv(f))
df = pd.concat(rows, ignore_index=True)

strategies = ["absolute", "motion", "scheduler"]
colors     = {"absolute": "green", "motion": "blue", "scheduler": "red"}
labels     = {"absolute": "Abs Min", "motion": "Motion", "scheduler": "Scheduler"}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Total photons per night
ax = axes[0]
x  = np.arange(len(df))
w  = 0.25
for k, s in enumerate(strategies):
    ax.bar(x + k*w, df[f"total_photons_{s}"], w,
           color=colors[s], alpha=0.8, label=labels[s])
ax.set_xticks(x + w); ax.set_xticklabels(df["night"], rotation=30, ha="right")
ax.set_ylabel("Total photons"); ax.set_title("Total Photon Yield by Night")
ax.legend(); ax.ticklabel_format(axis="y", style="sci", scilimits=(0,0))

# Shutter efficiency
ax = axes[1]
for k, s in enumerate(strategies):
    ax.bar(x + k*w, df[f"shutter_eff_{s}"], w,
           color=colors[s], alpha=0.8, label=labels[s])
ax.set_xticks(x + w); ax.set_xticklabels(df["night"], rotation=30, ha="right")
ax.set_ylabel("Shutter efficiency (%)"); ax.set_title("Shutter-open Efficiency")
ax.set_ylim(60, 100); ax.legend()

# Median 5-sigma depth
ax = axes[2]
for k, s in enumerate(strategies):
    ax.bar(x + k*w, df[f"median_depth_{s}"], w,
           color=colors[s], alpha=0.8, label=labels[s])
ax.set_xticks(x + w); ax.set_xticklabels(df["night"], rotation=30, ha="right")
ax.set_ylabel(r"Median $m_{5\sigma}$ (mag)")
ax.set_title(r"Survey Depth ($m_{5\sigma}$)")
ax.invert_yaxis(); ax.legend()

plt.tight_layout()
plt.savefig("outputs/all_nights_comparison.png", dpi=150, bbox_inches="tight")

ValueError: No objects to concatenate